# Torre de Londres

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/torre_de_londres.ipynb)

DCC014 Inteligência Artificial · UFJF · Trabalho 1 · Grupo 5 · Profa. Luciana

Resolvedor da Torre de Londres (3 hastes, 3 discos) com quatro algoritmos de busca sobre o mesmo motor. Cada seção de algoritmo segue a ordem dos slides: o problema, a ideia, o laço e as listas, as regras, a estratégia, a árvore sem poda e com poda, o pseudocódigo com o código do projeto, o caminho solução, os comparativos e a complexidade.

1. Busca irrevogável
2. Backtracking
3. Busca em largura
4. Busca ordenada
5. Comparação dos algoritmos
6. Integrantes

Todo o código marcado com **(ignore)** está no topo, na seção Setup: os módulos de apoio do projeto e as funções que desenham tabelas, figuras e gráficos. Rode **Ambiente de execução → Executar tudo** e comece a apresentação na seção 1. Dali em diante só aparece o código do projeto que vai para a apresentação (problema, regras, estratégias, fronteira, motor e algoritmos) e chamadas de uma linha que desenham as tabelas e os gráficos.

# Setup (ignore)

Rode esta seção inteira antes do resto do notebook. Nada aqui aparece nos slides: são os módulos de apoio do projeto e as funções que desenham tabelas e gráficos.

In [ ]:
#@title Prepara as pastas (ignore)
import sys
from pathlib import Path

for folder in ('config', 'core/rules/domain', 'core/rules', 'core/rules/strategies/domain', 'core/rules/strategies', 'core/domain', 'core/search_tree', 'core/algorithms/domain', 'core/algorithms/irrevocable', 'core/algorithms/backtracking', 'core/algorithms/breadth_first', 'core/algorithms/ordered', 'libs/inputs', 'libs/outputs', 'libs/ranking', 'runner'):
    Path("src", folder).mkdir(parents=True, exist_ok=True)
if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

## config/ (ignore)

In [ ]:
%%writefile src/config/logging.py
# (ignore)
from __future__ import annotations

import logging
import sys
from typing import Final

LOGGER_NAME: Final = "torre_de_londres"
LOG_FORMAT: Final = "%(levelname)s %(name)s: %(message)s"


def configure_logging(*, verbose: bool = False) -> logging.Logger:
    logger = logging.getLogger(LOGGER_NAME)
    logger.setLevel(logging.DEBUG if verbose else logging.WARNING)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stderr)
        handler.setFormatter(logging.Formatter(LOG_FORMAT))
        logger.addHandler(handler)
    return logger


def get_logger(suffix: str) -> logging.Logger:
    return logging.getLogger(f"{LOGGER_NAME}.{suffix}")

In [ ]:
%%writefile src/config/settings.py
# (ignore)
from __future__ import annotations

from collections.abc import Mapping
from typing import Final

MAX_ITERATIONS: Final = 10_000

DEFAULT_OUTPUT_FORMAT: Final = "console"
DEFAULT_RANK_MODE: Final = "lexicographic"

OUTPUT_FORMATS: Final[tuple[str, ...]] = ("console", "json")
RANK_MODES: Final[tuple[str, ...]] = ("lexicographic", "score")

EXIT_OK: Final = 0
EXIT_INVALID_INPUT: Final = 2


PROGRAM_NAME: Final = "main.py"

DESCRIPTION: Final = (
    "Resolvedor da Torre de Londres: executa busca irrevogável, backtracking, "
    "busca em largura e busca ordenada sobre as cartas do problema, variando a "
    "estratégia de controle, e ranqueia os resultados."
)

EPILOG: Final = (
    'Cada eixo omitido significa "todos": sem --problem roda todas as cartas, '
    "sem --algorithm roda todos os algoritmos, sem --strategy roda todas as "
    "estratégias."
)

HELP_HELP: Final = "Mostra esta ajuda e encerra."
HELP_PROBLEM: Final = "Cartas a resolver (repetível ou separado por vírgula)."
HELP_ALGORITHM: Final = "Algoritmos a executar (repetível ou separado por vírgula)."
HELP_STRATEGY: Final = "Estratégias de controle (repetível ou separado por vírgula)."
HELP_ALL: Final = (
    "Produto cartesiano completo: todas as cartas, algoritmos e estratégias."
)
HELP_ALL_GOALS: Final = (
    "Resolve todos os 36 estados finais possíveis a partir da posição inicial "
    "e mostra só o resumo, com média e mediana de cada algoritmo e estratégia."
)
HELP_NO_PRUNE: Final = (
    "Desliga a poda: a árvore aceita estados repetidos. A irrevogável e o "
    "backtracking podem então girar em ciclo até o limite de iterações."
)
HELP_SHOW_TREE: Final = "Renderiza a árvore de busca de cada execução."
HELP_SHOW_TRACE: Final = (
    "Mostra o passo a passo com regra aplicada e estado resultante."
)
HELP_SHOW_STATES: Final = "Desenha as hastes em cada estado do caminho solução."
HELP_FORMAT: Final = "Formato da saída."
HELP_OUTPUT: Final = "Arquivo de destino da saída; sem ele, escreve no terminal."
HELP_SVG_DIR: Final = (
    "Pasta onde gravar a árvore de busca de cada execução em DOT e SVG "
    "(o SVG exige o Graphviz instalado)."
)
HELP_RANK_MODE: Final = "Critério do placar."
HELP_MAX_ITERATIONS: Final = "Guarda contra execução patológica."
HELP_SEED: Final = "Reservado; a execução é determinística por construção."
HELP_LIST: Final = "Lista cartas, algoritmos e estratégias disponíveis e encerra."
HELP_VERBOSE: Final = "Habilita log de diagnóstico na saída de erro."

KIND_LABELS: Final[Mapping[str, str]] = {
    "problem": "Carta não reconhecida",
    "algorithm": "Algoritmo não reconhecido",
    "strategy": "Estratégia não reconhecida",
    "format": "Formato de saída não reconhecido",
    "rank_mode": "Modo de ranqueamento não reconhecido",
}

ERROR_PREFIX: Final = "Erro:"
ERROR_UNKNOWN_SELECTION: Final = "{kind}: {name}. Disponíveis: {available}."
ERROR_INVALID_ARGUMENT: Final = "argumento inválido: {detail}."
ERROR_HINT: Final = "Use --help para ver as opções."

USAGE_PREFIX: Final = "Uso: "
OPTIONS_TITLE: Final = "Opções"

LIST_PROBLEMS_HEADER: Final = "Cartas disponíveis"
LIST_ALGORITHMS_HEADER: Final = "Algoritmos disponíveis"
LIST_STRATEGIES_HEADER: Final = "Estratégias disponíveis"
OUTPUT_WRITTEN: Final = "Saída gravada em {path}"
GRAPHS_WRITTEN: Final = "{count} arquivos de árvore gravados em {path}"
GRAPHVIZ_MISSING: Final = (
    "Aviso: Graphviz (dot) não encontrado; só os arquivos .dot foram gravados."
)

## core/rules/domain/ (ignore)

In [ ]:
%%writefile src/core/rules/domain/exceptions.py
# (ignore)
class RuleError(Exception): ...


class RuleNotApplicableError(RuleError):
    def __init__(self, rule_id: str) -> None:
        super().__init__(f"rule {rule_id} is not applicable to the given state")
        self.rule_id = rule_id


class UnknownRuleError(RuleError):
    def __init__(self, rule_id: str) -> None:
        super().__init__(f"unknown rule identifier: {rule_id!r}")
        self.rule_id = rule_id


class UnknownStrategyError(RuleError):
    def __init__(self, name: str) -> None:
        super().__init__(f"unknown control strategy: {name!r}")
        self.name = name

## core/rules/strategies/domain/ (ignore)

In [ ]:
%%writefile src/core/rules/strategies/domain/base.py
# (ignore)
from __future__ import annotations

from typing import Protocol

from core.rules.domain.base import TransitionRule


class ControlStrategy(Protocol):
    @property
    def name(self) -> str: ...

    def order(
        self, rules: tuple[TransitionRule, ...]
    ) -> tuple[TransitionRule, ...]: ...

In [ ]:
%%writefile src/core/rules/strategies/domain/registry.py
# (ignore)
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.rules.domain.exceptions import UnknownStrategyError
from core.rules.strategies.ascending import ASCENDING
from core.rules.strategies.descending import DESCENDING
from core.rules.strategies.domain.base import ControlStrategy

STRATEGIES: Mapping[str, ControlStrategy] = MappingProxyType(
    {
        ASCENDING.name: ASCENDING,
        DESCENDING.name: DESCENDING,
    }
)

STRATEGY_NAMES: tuple[str, ...] = tuple(STRATEGIES)


def get_strategy(name: str) -> ControlStrategy:
    try:
        return STRATEGIES[name]
    except KeyError:
        raise UnknownStrategyError(name) from None

## core/domain/ (ignore)

In [ ]:
%%writefile src/core/domain/disk.py
# (ignore)
from core.rules.domain.base import TOTAL_DISKS, Disk

__all__ = ["TOTAL_DISKS", "Disk"]

In [ ]:
%%writefile src/core/domain/peg.py
# (ignore)
from core.rules.domain.base import CAPACITIES, Peg

__all__ = ["CAPACITIES", "Peg"]

In [ ]:
%%writefile src/core/domain/state.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterable

from core.rules.domain.base import CAPACITIES, TOTAL_DISKS, Disk, Peg, Stack, State

__all__ = [
    "CAPACITIES",
    "EMPTY_STATE",
    "TOTAL_DISKS",
    "Disk",
    "InvalidStateError",
    "Peg",
    "Stack",
    "State",
    "build_state",
    "disks_of",
    "is_valid",
    "stack_of",
    "top_of",
    "validate",
]

EMPTY_STATE: State = ((), (), ())


class InvalidStateError(Exception):
    def __init__(self, reason: str) -> None:
        super().__init__(f"invalid state: {reason}")
        self.reason = reason


def build_state(h1: Iterable[Disk], h2: Iterable[Disk], h3: Iterable[Disk]) -> State:
    state: State = (tuple(h1), tuple(h2), tuple(h3))
    validate(state)
    return state


def validate(state: State) -> None:
    if len(state) != len(Peg):
        raise InvalidStateError(f"expected {len(Peg)} pegs, got {len(state)}")

    for peg in Peg:
        capacity = CAPACITIES[peg]
        if len(state[peg]) > capacity:
            raise InvalidStateError(
                f"peg {peg.name} holds {len(state[peg])} disks, capacity is {capacity}"
            )

    disks = disks_of(state)
    if len(disks) != TOTAL_DISKS:
        raise InvalidStateError(
            f"expected {TOTAL_DISKS} distinct disks, got {len(disks)}"
        )


def is_valid(state: State) -> bool:
    try:
        validate(state)
    except InvalidStateError:
        return False
    return True


def disks_of(state: State) -> frozenset[Disk]:
    return frozenset(disk for stack in state for disk in stack)


def stack_of(state: State, peg: Peg) -> Stack:
    return state[peg]


def top_of(state: State, peg: Peg) -> Disk | None:
    stack = state[peg]
    return stack[-1] if stack else None

In [ ]:
%%writefile src/core/domain/state_space.py
# (ignore)
"""Enumeração exaustiva do espaço de estados.

Não participa de busca alguma: descreve o espaço por força bruta e serve de
oráculo independente aos testes.

Contagem:
    3! = 6 permutações x 10 cortes em três partes = 60 disposições
    as capacidades (3, 2, 1) deixam 6 distribuições:
        (3,0,0)  (2,1,0)  (2,0,1)  (1,2,0)  (1,1,1)  (0,2,1)
    6 distribuições x 6 arranjos = 36 estados válidos

Propriedades:
    - Grafo não dirigido: toda regra tem inversa exata.
    - Grafo conexo: todo objetivo é alcançável de qualquer inicial. Logo
      backtracking e largura sempre terminam em SUCESSO neste problema; só a
      busca irrevogável pode terminar em IMPASSE.
    - Fator de ramificação: 2 a 4 regras aplicáveis por estado.
    - Profundidade máxima de um caminho sem repetição: 36.

`shortest_distance` é uma busca em largura escrita à parte do motor, sem árvore,
nó ou estratégia. É o oráculo contra o qual a otimalidade de `breadth_first` é
verificada. `cheapest_cost` faz o mesmo papel para a busca ordenada: um
Dijkstra sobre o custo das regras, também fora do motor.
"""

from __future__ import annotations

import heapq
from collections import deque
from collections.abc import Iterator
from itertools import permutations

from core.domain.state import Disk, State, is_valid
from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES


def all_states() -> tuple[State, ...]:
    return tuple(_generate_states())


def _generate_states() -> Iterator[State]:
    disks = tuple(Disk)
    for arrangement in permutations(disks):
        for first in range(len(disks) + 1):
            for second in range(first, len(disks) + 1):
                candidate: State = (
                    arrangement[:first],
                    arrangement[first:second],
                    arrangement[second:],
                )
                if is_valid(candidate):
                    yield candidate


def applicable_rules(state: State) -> tuple[TransitionRule, ...]:
    return tuple(rule for rule in RULES if rule.is_applicable(state))


def successors(state: State) -> tuple[tuple[TransitionRule, State], ...]:
    return tuple((rule, rule.apply(state)) for rule in applicable_rules(state))


def reachable_from(source: State) -> frozenset[State]:
    seen = {source}
    queue = deque([source])
    while queue:
        current = queue.popleft()
        for _, successor in successors(current):
            if successor not in seen:
                seen.add(successor)
                queue.append(successor)
    return frozenset(seen)


def shortest_distance(source: State, target: State) -> int | None:
    if source == target:
        return 0
    depths = {source: 0}
    queue = deque([source])
    while queue:
        current = queue.popleft()
        for _, successor in successors(current):
            if successor in depths:
                continue
            depths[successor] = depths[current] + 1
            if successor == target:
                return depths[successor]
            queue.append(successor)
    return None


def cheapest_cost(source: State, target: State) -> int | None:
    costs = {source: 0}
    heap: list[tuple[int, int, State]] = [(0, 0, source)]
    pushed = 1
    while heap:
        cost, _, current = heapq.heappop(heap)
        if current == target:
            return cost
        if cost > costs[current]:
            continue
        for rule, successor in successors(current):
            candidate = cost + rule.cost(current)
            if candidate < costs.get(successor, candidate + 1):
                costs[successor] = candidate
                heapq.heappush(heap, (candidate, pushed, successor))
                pushed += 1
    return None

## core/search_tree/ (ignore)

In [ ]:
%%writefile src/core/search_tree/metrics.py
# (ignore)
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True, slots=True)
class SearchMetrics:
    iterations: int
    nodes_generated: int
    nodes_visited: int
    rules_tested: int
    backtracks: int
    deadlocks: int
    max_depth: int
    elapsed_ms: float
    max_frontier: int = 0


@dataclass(slots=True)
class MetricsCollector:
    iterations: int = 0
    nodes_generated: int = 0
    nodes_visited: int = 0
    rules_tested: int = 0
    backtracks: int = 0
    deadlocks: int = 0
    max_depth: int = field(default=0)
    max_frontier: int = 0

    def count_iteration(self) -> None:
        self.iterations += 1

    def count_generated(self, depth: int) -> None:
        self.nodes_generated += 1
        self.max_depth = max(self.max_depth, depth)

    def observe_frontier(self, size: int) -> None:
        self.max_frontier = max(self.max_frontier, size)

    def count_visited(self) -> None:
        self.nodes_visited += 1

    def count_rule_test(self) -> None:
        self.rules_tested += 1

    def count_backtrack(self) -> None:
        self.backtracks += 1

    def count_deadlock(self) -> None:
        self.deadlocks += 1

    def seal(self, elapsed_ms: float) -> SearchMetrics:
        return SearchMetrics(
            iterations=self.iterations,
            nodes_generated=self.nodes_generated,
            nodes_visited=self.nodes_visited,
            rules_tested=self.rules_tested,
            backtracks=self.backtracks,
            deadlocks=self.deadlocks,
            max_depth=self.max_depth,
            elapsed_ms=elapsed_ms,
            max_frontier=self.max_frontier,
        )

In [ ]:
%%writefile src/core/search_tree/node.py
# (ignore)
from __future__ import annotations

from dataclasses import dataclass

from core.domain.state import State
from core.rules.domain.base import TransitionRule


@dataclass(frozen=True, slots=True)
class Node:
    state: State
    parent: Node | None
    rule: TransitionRule | None
    depth: int
    order: int
    cost: int = 0

    @property
    def is_root(self) -> bool:
        return self.parent is None

    def __hash__(self) -> int:
        return hash((self.order, self.state))

In [ ]:
%%writefile src/core/search_tree/outcome.py
# (ignore)
from __future__ import annotations

from enum import Enum, unique


@unique
class Outcome(Enum):
    SUCCESS = "SUCCESS"
    DEADLOCK = "DEADLOCK"
    CUTOFF = "CUTOFF"
    FAILURE = "FAILURE"

    @property
    def is_success(self) -> bool:
        return self is Outcome.SUCCESS

In [ ]:
%%writefile src/core/search_tree/path.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterator

from core.domain.state import State
from core.search_tree.node import Node


def ancestors(node: Node) -> Iterator[Node]:
    current = node.parent
    while current is not None:
        yield current
        current = current.parent


def path_to(node: Node) -> tuple[Node, ...]:
    reversed_path = [node, *ancestors(node)]
    reversed_path.reverse()
    return tuple(reversed_path)


def states_on_path(node: Node) -> frozenset[State]:
    return frozenset(step.state for step in path_to(node))


def contains_state(node: Node, state: State) -> bool:
    if node.state == state:
        return True
    return any(ancestor.state == state for ancestor in ancestors(node))


def applied_rules(node: Node) -> tuple[str, ...]:
    return tuple(step.rule.id for step in path_to(node) if step.rule is not None)

In [ ]:
%%writefile src/core/search_tree/result.py
# (ignore)
from __future__ import annotations

from dataclasses import dataclass

from core.search_tree.metrics import SearchMetrics
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.trace import Trace


@dataclass(frozen=True, slots=True)
class SearchResult:
    algorithm: str
    strategy: str
    problem: str
    outcome: Outcome
    solution_path: tuple[Node, ...]
    applied_rules: tuple[str, ...]
    metrics: SearchMetrics
    trace: Trace
    pruned: bool = True

    @property
    def solution_length(self) -> int | None:
        return len(self.applied_rules) if self.outcome.is_success else None

    @property
    def solution_cost(self) -> int | None:
        return self.solution_path[-1].cost if self.outcome.is_success else None

    @property
    def label(self) -> str:
        return f"{self.algorithm} · {self.strategy}"

In [ ]:
%%writefile src/core/search_tree/trace.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass, field
from enum import Enum, unique

from core.domain.state import State
from core.search_tree.node import Node


@unique
class TraceEvent(Enum):
    ROOT = "ROOT"
    VISIT = "VISIT"
    GENERATE = "GENERATE"
    PRUNE = "PRUNE"
    GOAL = "GOAL"
    DEADLOCK = "DEADLOCK"
    BACKTRACK = "BACKTRACK"
    CUTOFF = "CUTOFF"
    EXHAUSTED = "EXHAUSTED"


@dataclass(frozen=True, slots=True)
class TraceStep:
    iteration: int
    event: TraceEvent
    node_order: int
    parent_order: int | None
    depth: int
    rule_id: str | None
    state: State


@dataclass(frozen=True, slots=True)
class Trace:
    steps: tuple[TraceStep, ...] = ()

    def __iter__(self) -> Iterator[TraceStep]:
        return iter(self.steps)

    def __len__(self) -> int:
        return len(self.steps)

    def of_event(self, event: TraceEvent) -> tuple[TraceStep, ...]:
        return tuple(step for step in self.steps if step.event is event)


@dataclass(slots=True)
class TraceRecorder:
    _steps: list[TraceStep] = field(default_factory=list)

    def record(
        self,
        *,
        iteration: int,
        event: TraceEvent,
        node: Node,
        rule_id: str | None = None,
    ) -> None:
        parent = node.parent
        self._steps.append(
            TraceStep(
                iteration=iteration,
                event=event,
                node_order=node.order,
                parent_order=None if parent is None else parent.order,
                depth=node.depth,
                rule_id=rule_id if rule_id is not None else _rule_id_of(node),
                state=node.state,
            )
        )

    def record_prune(
        self, *, iteration: int, node: Node, rule_id: str, state: State
    ) -> None:
        self._steps.append(
            TraceStep(
                iteration=iteration,
                event=TraceEvent.PRUNE,
                node_order=node.order,
                parent_order=None if node.parent is None else node.parent.order,
                depth=node.depth,
                rule_id=rule_id,
                state=state,
            )
        )

    def seal(self) -> Trace:
        return Trace(steps=tuple(self._steps))


def _rule_id_of(node: Node) -> str | None:
    return None if node.rule is None else node.rule.id

In [ ]:
%%writefile src/core/search_tree/tree.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterator

from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES
from core.search_tree.node import Node
from core.search_tree.path import ancestors


class SearchTree:
    __slots__ = ("_nodes", "_rules")

    def __init__(self, rules: tuple[TransitionRule, ...] = RULES) -> None:
        self._rules = rules
        self._nodes: list[Node] = []

    @property
    def rules(self) -> tuple[TransitionRule, ...]:
        return self._rules

    @property
    def nodes(self) -> tuple[Node, ...]:
        return tuple(self._nodes)

    def root(self, state: State) -> Node:
        self._nodes = []
        node = Node(state=state, parent=None, rule=None, depth=0, order=0)
        self._nodes.append(node)
        return node

    def expand(self, node: Node, rule: TransitionRule) -> Node:
        child = Node(
            state=rule.apply(node.state),
            parent=node,
            rule=rule,
            depth=node.depth + 1,
            order=len(self._nodes),
            cost=node.cost + rule.cost(node.state),
        )
        self._nodes.append(child)
        return child

    def ancestors(self, node: Node) -> Iterator[Node]:
        return ancestors(node)

## core/algorithms/domain/ (ignore)

In [ ]:
%%writefile src/core/algorithms/domain/registry.py
# (ignore)
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.algorithms.backtracking.algorithm import BacktrackingSearch
from core.algorithms.breadth_first.algorithm import BreadthFirstSearch
from core.algorithms.domain.base import SearchAlgorithm
from core.algorithms.irrevocable.algorithm import IrrevocableSearch
from core.algorithms.ordered.algorithm import OrderedSearch


class UnknownAlgorithmError(Exception):
    def __init__(self, name: str) -> None:
        super().__init__(f"unknown search algorithm: {name!r}")
        self.name = name


ALGORITHMS: Mapping[str, type[SearchAlgorithm]] = MappingProxyType(
    {
        IrrevocableSearch.name: IrrevocableSearch,
        BacktrackingSearch.name: BacktrackingSearch,
        BreadthFirstSearch.name: BreadthFirstSearch,
        OrderedSearch.name: OrderedSearch,
    }
)

ALGORITHM_NAMES: tuple[str, ...] = tuple(ALGORITHMS)


def get_algorithm(name: str) -> type[SearchAlgorithm]:
    try:
        return ALGORITHMS[name]
    except KeyError:
        raise UnknownAlgorithmError(name) from None

## libs/inputs/ (ignore)

In [ ]:
%%writefile src/libs/inputs/exceptions.py
# (ignore)
from __future__ import annotations


class InputError(Exception): ...


class InvalidArgumentError(InputError):
    def __init__(self, detail: str) -> None:
        super().__init__(f"invalid argument: {detail}")
        self.detail = detail


class UnknownSelectionError(InputError):
    def __init__(self, kind: str, name: str, available: tuple[str, ...]) -> None:
        super().__init__(f"unknown {kind}: {name!r}")
        self.kind = kind
        self.name = name
        self.available = available

In [ ]:
%%writefile src/libs/inputs/parser.py
# (ignore)
from __future__ import annotations

import argparse
from collections.abc import Iterable, Sequence
from pathlib import Path
from typing import NoReturn

from config import settings
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEM_IDS
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs.exceptions import InvalidArgumentError
from libs.inputs.selection import ExecutionRequest


class _HelpFormatter(argparse.HelpFormatter):
    def add_usage(
        self,
        usage: str | None,
        actions: Iterable[argparse.Action],
        groups: Iterable[argparse._MutuallyExclusiveGroup],
        prefix: str | None = None,
    ) -> None:
        super().add_usage(usage, actions, groups, settings.USAGE_PREFIX)


class _Parser(argparse.ArgumentParser):
    def error(self, message: str) -> NoReturn:
        raise InvalidArgumentError(message)

    def exit(self, status: int = 0, message: str | None = None) -> NoReturn:
        raise InvalidArgumentError(message or "")


def build_parser() -> argparse.ArgumentParser:
    parser = _Parser(
        prog=settings.PROGRAM_NAME,
        description=settings.DESCRIPTION,
        epilog=settings.EPILOG,
        add_help=False,
        formatter_class=_HelpFormatter,
    )
    group = parser.add_argument_group(settings.OPTIONS_TITLE)
    group.add_argument("-h", "--help", action="store_true", help=settings.HELP_HELP)
    group.add_argument("--problem", action="append", help=settings.HELP_PROBLEM)
    group.add_argument("--algorithm", action="append", help=settings.HELP_ALGORITHM)
    group.add_argument("--strategy", action="append", help=settings.HELP_STRATEGY)
    group.add_argument("--all", action="store_true", help=settings.HELP_ALL)
    group.add_argument("--all-goals", action="store_true", help=settings.HELP_ALL_GOALS)
    group.add_argument("--no-prune", action="store_true", help=settings.HELP_NO_PRUNE)
    group.add_argument("--show-tree", action="store_true", help=settings.HELP_SHOW_TREE)
    group.add_argument(
        "--show-trace", action="store_true", help=settings.HELP_SHOW_TRACE
    )
    group.add_argument(
        "--show-states", action="store_true", help=settings.HELP_SHOW_STATES
    )
    group.add_argument(
        "--format", default=settings.DEFAULT_OUTPUT_FORMAT, help=settings.HELP_FORMAT
    )
    group.add_argument("--output", help=settings.HELP_OUTPUT)
    group.add_argument("--svg-dir", help=settings.HELP_SVG_DIR)
    group.add_argument(
        "--rank-mode", default=settings.DEFAULT_RANK_MODE, help=settings.HELP_RANK_MODE
    )
    group.add_argument(
        "--max-iterations",
        default=str(settings.MAX_ITERATIONS),
        help=settings.HELP_MAX_ITERATIONS,
    )
    group.add_argument("--seed", help=settings.HELP_SEED)
    group.add_argument("--list", action="store_true", help=settings.HELP_LIST)
    group.add_argument("--verbose", action="store_true", help=settings.HELP_VERBOSE)
    return parser


def help_text() -> str:
    return build_parser().format_help()


def parse(argv: Sequence[str]) -> ExecutionRequest:
    namespace = build_parser().parse_args(list(argv))

    select_all: bool = namespace.all
    problems = _selected(namespace.problem, PROBLEM_IDS, select_all, upper=True)
    algorithms = _selected(namespace.algorithm, ALGORITHM_NAMES, select_all)
    strategies = _selected(namespace.strategy, STRATEGY_NAMES, select_all)

    return ExecutionRequest(
        problem_ids=problems,
        algorithm_names=algorithms,
        strategy_names=strategies,
        show_tree=namespace.show_tree,
        show_trace=namespace.show_trace,
        show_states=namespace.show_states,
        output_format=namespace.format.lower(),
        rank_mode=namespace.rank_mode.lower(),
        max_iterations=_positive_int("--max-iterations", namespace.max_iterations),
        output_path=Path(namespace.output) if namespace.output else None,
        svg_dir=Path(namespace.svg_dir) if namespace.svg_dir else None,
        list_only=namespace.list,
        show_help=namespace.help,
        verbose=namespace.verbose,
        all_goals=namespace.all_goals,
        prune=not namespace.no_prune,
        seed=None
        if namespace.seed is None
        else _positive_int("--seed", namespace.seed),
    )


def _selected(
    raw: list[str] | None,
    available: tuple[str, ...],
    select_all: bool,
    *,
    upper: bool = False,
) -> tuple[str, ...]:
    if select_all or not raw:
        return available
    names = [
        token.strip().upper() if upper else token.strip().lower()
        for entry in raw
        for token in entry.split(",")
        if token.strip()
    ]
    return tuple(dict.fromkeys(names))


def _positive_int(flag: str, raw: str) -> int:
    try:
        value = int(raw)
    except ValueError:
        raise InvalidArgumentError(f"{flag}={raw!r}") from None
    if value <= 0:
        raise InvalidArgumentError(f"{flag}={raw!r}")
    return value

In [ ]:
%%writefile src/libs/inputs/selection.py
# (ignore)
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from config import settings


@dataclass(frozen=True, slots=True)
class ExecutionRequest:
    problem_ids: tuple[str, ...]
    algorithm_names: tuple[str, ...]
    strategy_names: tuple[str, ...]
    show_tree: bool = False
    show_trace: bool = False
    show_states: bool = False
    output_format: str = settings.DEFAULT_OUTPUT_FORMAT
    rank_mode: str = settings.DEFAULT_RANK_MODE
    max_iterations: int = settings.MAX_ITERATIONS
    output_path: Path | None = None
    svg_dir: Path | None = None
    list_only: bool = False
    show_help: bool = False
    verbose: bool = False
    seed: int | None = None
    all_goals: bool = False
    prune: bool = True

    @property
    def execution_count(self) -> int:
        return (
            len(self.problem_ids) * len(self.algorithm_names) * len(self.strategy_names)
        )

In [ ]:
%%writefile src/libs/inputs/validator.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterable

from config import settings
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEM_IDS
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs.exceptions import UnknownSelectionError
from libs.inputs.selection import ExecutionRequest


def validate(request: ExecutionRequest) -> ExecutionRequest:
    _require_known("problem", request.problem_ids, PROBLEM_IDS)
    _require_known("algorithm", request.algorithm_names, ALGORITHM_NAMES)
    _require_known("strategy", request.strategy_names, STRATEGY_NAMES)
    _require_known("format", (request.output_format,), settings.OUTPUT_FORMATS)
    _require_known("rank_mode", (request.rank_mode,), settings.RANK_MODES)
    return request


def _require_known(
    kind: str, selected: Iterable[str], available: tuple[str, ...]
) -> None:
    known = set(available)
    for name in selected:
        if name not in known:
            raise UnknownSelectionError(kind, name, available)

## libs/outputs/ (ignore)

In [ ]:
%%writefile src/libs/outputs/console.py
# (ignore)
from __future__ import annotations

import sys
import textwrap
from typing import TextIO

from core.search_tree.result import SearchResult
from libs.outputs import state_render, theme, trace_render, tree_render
from libs.outputs.formatter import ProblemReport, Report
from libs.outputs.trace_render import render_table
from libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow


class ConsoleFormatter:
    name = "console"

    def render(self, report: Report) -> str:
        blocks = (
            []
            if report.summary_only
            else [self._render_problem(entry, report) for entry in report.problems]
        )
        if report.summary is not None and (
            report.summary_only or len(report.problems) > 1
        ):
            blocks.append(self._render_summary(report.summary))
        return "\n\n".join(block for block in blocks if block)

    def _render_summary(self, summary: Summary) -> str:
        sections = [render_section(theme.SUMMARY_HEADER, render_summary(summary))]
        if summary.highlights:
            sections.append(
                render_section(theme.HIGHLIGHTS_HEADER, render_highlights(summary))
            )
        return "\n\n".join(sections)

    def _render_problem(self, entry: ProblemReport, report: Report) -> str:
        sections = [
            f"{theme.PROBLEM_HEADER} {entry.problem_id}",
            "",
            f"{theme.INITIAL_STATE_HEADER}: {state_render.render_inline(entry.initial_state)}",
            f"{theme.GOAL_STATE_HEADER}: {state_render.render_inline(entry.goal_state)}",
        ]
        for result in entry.results:
            sections.append("")
            sections.append(render_result_box(result))
            sections.extend(_render_details(result, report))
        if entry.leaderboard is not None:
            sections.extend(
                [
                    "",
                    render_section(
                        theme.LEADERBOARD_HEADER, render_leaderboard(entry.leaderboard)
                    ),
                ]
            )
        return "\n".join(sections)


def render_result_box(result: SearchResult) -> str:
    metrics = result.metrics
    moves = (
        theme.ABSENT if result.solution_length is None else str(result.solution_length)
    )
    cost = theme.ABSENT if result.solution_cost is None else str(result.solution_cost)
    path = (
        theme.ARROW.join(result.applied_rules) if result.applied_rules else theme.ABSENT
    )
    rows = (
        (theme.REPORT_LABELS["outcome"], theme.outcome_label(result.outcome)),
        (theme.REPORT_LABELS["moves"], moves),
        (theme.REPORT_LABELS["cost"], cost),
        (theme.REPORT_LABELS["path"], path),
        (theme.REPORT_LABELS["iterations"], str(metrics.iterations)),
        (theme.REPORT_LABELS["nodes_generated"], str(metrics.nodes_generated)),
        (theme.REPORT_LABELS["nodes_visited"], str(metrics.nodes_visited)),
        (theme.REPORT_LABELS["rules_tested"], str(metrics.rules_tested)),
        (theme.REPORT_LABELS["backtracks"], str(metrics.backtracks)),
        (theme.REPORT_LABELS["deadlocks"], str(metrics.deadlocks)),
        (theme.REPORT_LABELS["max_depth"], str(metrics.max_depth)),
        (theme.REPORT_LABELS["max_frontier"], str(metrics.max_frontier)),
        (theme.REPORT_LABELS["elapsed"], f"{metrics.elapsed_ms:.3f}"),
    )
    lines = [line for label, value in rows for line in _wrap_row(label, value)]
    return render_box(result.label, tuple(lines))


def _wrap_row(label: str, value: str) -> list[str]:
    available = theme.MAX_BOX_WIDTH - 2 - theme.LABEL_WIDTH
    chunks = textwrap.wrap(value, width=available) or [""]
    head = f"{label.ljust(theme.LABEL_WIDTH)}{chunks[0]}"
    tail = [f"{''.ljust(theme.LABEL_WIDTH)}{chunk}" for chunk in chunks[1:]]
    return [head, *tail]


def render_leaderboard(leaderboard: Leaderboard) -> str:
    scored = leaderboard.mode == "score"
    headers = theme.LEADERBOARD_HEADERS if scored else theme.LEADERBOARD_HEADERS[:-1]
    rows = []
    for row in leaderboard.rows:
        result = row.result
        cells = [
            str(row.position),
            result.algorithm,
            result.strategy,
            theme.outcome_label(result.outcome),
            theme.ABSENT
            if result.solution_length is None
            else str(result.solution_length),
            str(result.metrics.iterations),
            str(result.metrics.nodes_generated),
        ]
        if scored:
            cells.append(theme.ABSENT if row.score is None else f"{row.score:.1f}")
        rows.append(tuple(cells))
    return render_table(headers, tuple(rows))


def render_summary(summary: Summary) -> str:
    rows = tuple(
        (
            row.algorithm,
            row.strategy,
            f"{row.successes}/{row.runs}",
            str(row.deadlocks),
            _render_stat(row.moves),
            _render_stat(row.cost),
            _render_stat(row.iterations),
            _render_stat(row.nodes_generated),
        )
        for row in summary.rows
    )
    table = render_table(theme.SUMMARY_HEADERS, rows)
    return f"{theme.SUMMARY_LEGEND}\n\n{table}"


def render_highlights(summary: Summary) -> str:
    strategies: dict[str, int] = {}
    for row in summary.rows:
        strategies[row.algorithm] = strategies.get(row.algorithm, 0) + 1
    rows = tuple(
        (
            theme.CRITERION_LABELS[highlight.criterion],
            _render_winners(highlight.by_mean, strategies),
            _render_winners(highlight.by_median, strategies),
        )
        for highlight in summary.highlights
    )
    table = render_table(theme.HIGHLIGHTS_HEADERS, rows)
    return f"{theme.HIGHLIGHTS_LEGEND}\n\n{table}"


def _render_winners(winners: tuple[SummaryRow, ...], strategies: dict[str, int]) -> str:
    grouped: dict[str, list[str]] = {}
    for row in winners:
        grouped.setdefault(row.algorithm, []).append(row.strategy)
    return "; ".join(
        f"{algorithm} {theme.ALL_STRATEGIES}"
        if len(names) == strategies[algorithm]
        else f"{algorithm} ({', '.join(names)})"
        for algorithm, names in grouped.items()
    )


def _render_stat(stat: Stat | None) -> str:
    if stat is None:
        return theme.ABSENT
    return f"{stat.mean:.2f} / {stat.median:g}"


def render_box(title: str, lines: tuple[str, ...]) -> str:
    content_width = max((len(line) for line in lines), default=0)
    width = min(
        max(content_width + 2, len(title) + 6, theme.MIN_BOX_WIDTH),
        max(theme.MAX_BOX_WIDTH, len(title) + 6, content_width + 2),
    )

    header = f"{theme.BOX_HORIZONTAL} {title} "
    top = (
        theme.BOX_TOP_LEFT
        + header
        + theme.BOX_HORIZONTAL * (width - len(header))
        + theme.BOX_TOP_RIGHT
    )
    body = [
        f"{theme.BOX_VERTICAL} {line.ljust(width - 2)} {theme.BOX_VERTICAL}"
        for line in lines
    ]
    bottom = (
        theme.BOX_BOTTOM_LEFT + theme.BOX_HORIZONTAL * width + theme.BOX_BOTTOM_RIGHT
    )
    return "\n".join([top, *body, bottom])


def render_section(title: str, body: str) -> str:
    return f"{title}\n{theme.BOX_HORIZONTAL * len(title)}\n{body}"


def _render_details(result: SearchResult, report: Report) -> list[str]:
    sections: list[str] = []
    if report.show_states and result.solution_path:
        drawings = "\n\n".join(
            state_render.render_pegs(node.state) for node in result.solution_path
        )
        sections.extend(["", render_section(theme.STATES_HEADER, drawings)])
    if report.show_tree:
        sections.extend(
            [
                "",
                render_section(
                    theme.TREE_HEADER, tree_render.render_tree(result.trace)
                ),
            ]
        )
    if report.show_trace:
        sections.extend(
            [
                "",
                render_section(
                    theme.TRACE_HEADER, trace_render.render_trace(result.trace)
                ),
            ]
        )
    return sections


def write(text: str, stream: TextIO | None = None) -> None:
    target = sys.stdout if stream is None else stream
    target.write(f"{text}\n")


def write_error(text: str) -> None:
    sys.stderr.write(f"{text}\n")

In [ ]:
%%writefile src/libs/outputs/dot_render.py
# (ignore)
from __future__ import annotations

from core.search_tree.result import SearchResult
from core.search_tree.trace import TraceEvent, TraceStep
from libs.outputs import state_render, theme
from libs.outputs.tree_render import removed_edges


def render_dot(result: SearchResult) -> str:
    trace = result.trace
    root_steps = trace.of_event(TraceEvent.ROOT)
    if not root_steps:
        return ""

    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}
    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}
    visited_orders = {step.node_order for step in trace.of_event(TraceEvent.VISIT)}
    solution_orders = {node.order for node in result.solution_path}
    removed = removed_edges(trace)

    lines = [
        "digraph search {",
        f'  label="{_title(result)}"; labelloc=t; fontname="{theme.GRAPH_FONT}";',
        f'  node [shape=box, style="rounded,filled", fontname="{theme.GRAPH_MONO_FONT}", fontsize=10];',
        f'  edge [fontname="{theme.GRAPH_FONT}", fontsize=9];',
        _node(root_steps[0], theme.GRAPH_ROOT_FILL, on_path=bool(solution_orders)),
    ]

    for step in trace.of_event(TraceEvent.GENERATE):
        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:
            continue
        on_path = step.node_order in solution_orders
        if step.node_order in goal_orders:
            fill = theme.GRAPH_GOAL_FILL
        elif step.node_order in deadlock_orders:
            fill = theme.GRAPH_DEADLOCK_FILL
        elif step.node_order in visited_orders:
            fill = theme.GRAPH_VISITED_FILL
        else:
            fill = theme.GRAPH_GENERATED_FILL
        lines.append(_node(step, fill, on_path=on_path))
        lines.append(_edge(step, on_path=on_path))

    lines.extend(_pruned(step) for step in trace.of_event(TraceEvent.PRUNE))
    lines.append("}")
    return "\n".join(lines)


def _title(result: SearchResult) -> str:
    parts = [result.problem, result.label]
    if not result.pruned:
        parts.append(theme.GRAPH_NO_PRUNE_LABEL)
    parts.append(theme.outcome_label(result.outcome))
    if result.solution_length is not None:
        parts.append(f"{result.solution_length} {theme.GRAPH_MOVES_LABEL}")
    if result.solution_cost is not None:
        parts.append(f"{theme.GRAPH_COST_LABEL} {result.solution_cost}")
    return " · ".join(parts)


def _node(step: TraceStep, fill: str, *, on_path: bool) -> str:
    label = f"#{step.node_order}\\n{state_render.render_inline(step.state)}"
    width = theme.GRAPH_PATH_WIDTH if on_path else 1
    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR
    return (
        f'  n{step.node_order} [label="{label}", fillcolor="{fill}", '
        f'color="{color}", penwidth={width}];'
    )


def _edge(step: TraceStep, *, on_path: bool) -> str:
    width = theme.GRAPH_PATH_WIDTH if on_path else 1
    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR
    return (
        f'  n{step.parent_order} -> n{step.node_order} [label="{step.rule_id}", '
        f'color="{color}", fontcolor="{color}", penwidth={width}];'
    )


def _pruned(step: TraceStep) -> str:
    ghost = f"p{step.node_order}_{step.rule_id}"
    return (
        f'  {ghost} [label="{step.rule_id}", shape=plaintext, style=solid, '
        f'fontcolor="{theme.GRAPH_PRUNED_COLOR}", fontsize=8];\n'
        f"  n{step.node_order} -> {ghost} [style=dotted, "
        f'color="{theme.GRAPH_PRUNED_COLOR}", arrowhead=none];'
    )

In [ ]:
%%writefile src/libs/outputs/formatter.py
# (ignore)
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol

from core.domain.state import State
from core.search_tree.result import SearchResult
from libs.ranking.leaderboard import Leaderboard, Summary


@dataclass(frozen=True, slots=True)
class ProblemReport:
    problem_id: str
    initial_state: State
    goal_state: State
    results: tuple[SearchResult, ...]
    leaderboard: Leaderboard | None = None


@dataclass(frozen=True, slots=True)
class Report:
    problems: tuple[ProblemReport, ...]
    summary: Summary | None = None
    show_tree: bool = False
    show_trace: bool = False
    show_states: bool = False
    summary_only: bool = False


class Formatter(Protocol):
    def render(self, report: Report) -> str: ...

In [ ]:
%%writefile src/libs/outputs/graph_writer.py
# (ignore)
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

from libs.outputs import dot_render, theme
from libs.outputs.formatter import Report

DOT_EXECUTABLE = "dot"


def graphviz_available() -> bool:
    return shutil.which(DOT_EXECUTABLE) is not None


def write_graphs(report: Report, directory: Path) -> tuple[Path, ...]:
    render = graphviz_available()
    written: list[Path] = []
    for entry in report.problems:
        target = directory / entry.problem_id
        target.mkdir(parents=True, exist_ok=True)
        for result in entry.results:
            source = dot_render.render_dot(result)
            suffix = "" if result.pruned else theme.GRAPH_NO_PRUNE_SUFFIX
            stem = target / f"{result.algorithm}_{result.strategy}{suffix}"
            dot_path = stem.with_suffix(".dot")
            dot_path.write_text(source, encoding="utf-8")
            written.append(dot_path)
            if render:
                written.append(_render_svg(dot_path, stem.with_suffix(".svg")))
    return tuple(written)


def _render_svg(dot_path: Path, svg_path: Path) -> Path:
    subprocess.run(
        [DOT_EXECUTABLE, "-Tsvg", str(dot_path), "-o", str(svg_path)],
        check=True,
        capture_output=True,
    )
    return svg_path

In [ ]:
%%writefile src/libs/outputs/json_writer.py
# (ignore)
from __future__ import annotations

import json
from pathlib import Path

from core.domain.state import Peg, State
from core.search_tree.result import SearchResult
from libs.outputs import theme
from libs.outputs.formatter import ProblemReport, Report
from libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow

type JsonValue = (
    str | int | float | bool | list["JsonValue"] | dict[str, "JsonValue"] | None
)


class JsonFormatter:
    name = "json"

    def render(self, report: Report) -> str:
        payload: dict[str, JsonValue] = {
            "problemas": [_problem_to_json(entry) for entry in report.problems]
        }
        if report.summary is not None:
            payload["resumo"] = _summary_to_json(report.summary)
        return json.dumps(payload, ensure_ascii=False, indent=2)


def write_file(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _problem_to_json(entry: ProblemReport) -> JsonValue:
    payload: dict[str, JsonValue] = {
        "id": entry.problem_id,
        "estado_inicial": _state_to_json(entry.initial_state),
        "objetivo": _state_to_json(entry.goal_state),
        "execucoes": [_result_to_json(result) for result in entry.results],
    }
    if entry.leaderboard is not None:
        payload["placar"] = _leaderboard_to_json(entry.leaderboard)
    return payload


def _leaderboard_to_json(leaderboard: Leaderboard) -> JsonValue:
    return {
        "modo": leaderboard.mode,
        "posicoes": [
            {
                "posicao": row.position,
                "algoritmo": row.result.algorithm,
                "estrategia": row.result.strategy,
                "desfecho": theme.outcome_label(row.result.outcome),
                "movimentos": row.result.solution_length,
                "escore": row.score,
            }
            for row in leaderboard.rows
        ],
    }


def _summary_to_json(summary: Summary) -> JsonValue:
    return {
        "linhas": [
            {
                "algoritmo": row.algorithm,
                "estrategia": row.strategy,
                "execucoes": row.runs,
                "sucessos": row.successes,
                "impasses": row.deadlocks,
                "movimentos": _stat_to_json(row.moves),
                "custo": _stat_to_json(row.cost),
                "iteracoes": _stat_to_json(row.iterations),
                "nos_gerados": _stat_to_json(row.nodes_generated),
            }
            for row in summary.rows
        ],
        "melhores": [
            {
                "criterio": highlight.criterion,
                "pela_media": [_row_label(row) for row in highlight.by_mean],
                "pela_mediana": [_row_label(row) for row in highlight.by_median],
            }
            for highlight in summary.highlights
        ],
    }


def _row_label(row: SummaryRow) -> JsonValue:
    return {"algoritmo": row.algorithm, "estrategia": row.strategy}


def _stat_to_json(stat: Stat | None) -> JsonValue:
    if stat is None:
        return None
    return {"media": round(stat.mean, 2), "mediana": stat.median}


def _result_to_json(result: SearchResult) -> JsonValue:
    metrics = result.metrics
    return {
        "algoritmo": result.algorithm,
        "estrategia": result.strategy,
        "poda": result.pruned,
        "desfecho": theme.outcome_label(result.outcome),
        "movimentos": result.solution_length,
        "custo": result.solution_cost,
        "caminho": list(result.applied_rules),
        "metricas": {
            "iteracoes": metrics.iterations,
            "nos_gerados": metrics.nodes_generated,
            "nos_visitados": metrics.nodes_visited,
            "regras_testadas": metrics.rules_tested,
            "retrocessos": metrics.backtracks,
            "impasses": metrics.deadlocks,
            "profundidade_maxima": metrics.max_depth,
            "pico_abertos": metrics.max_frontier,
            "tempo_ms": round(metrics.elapsed_ms, 3),
        },
        "estados": [_state_to_json(node.state) for node in result.solution_path],
    }


def _state_to_json(state: State) -> JsonValue:
    return {
        theme.PEG_NAMES[peg]: [theme.disk_name(disk) for disk in state[peg]]
        for peg in Peg
    }

In [ ]:
%%writefile src/libs/outputs/state_render.py
# (ignore)
from __future__ import annotations

from core.domain.state import CAPACITIES, Peg, State
from libs.outputs import theme


def render_inline(state: State) -> str:
    parts = []
    for peg in Peg:
        stack = state[peg]
        content = (
            ",".join(theme.disk_symbol(disk) for disk in stack)
            if stack
            else theme.EMPTY_PEG
        )
        parts.append(f"{theme.PEG_NAMES[peg]}[{content}]")
    return " ".join(parts)


def render_pegs(state: State) -> str:
    height = max(CAPACITIES.values())
    columns = [_column(state, peg, height) for peg in Peg]
    width = max(len(cell) for column in columns for cell in column)

    lines = [
        "  ".join(column[level].center(width) for column in columns).rstrip()
        for level in range(height)
    ]
    footer = "  ".join(
        f"{theme.PEG_NAMES[peg]}({CAPACITIES[peg]})".center(width) for peg in Peg
    )
    lines.append(footer.rstrip())
    return "\n".join(lines)


def _column(state: State, peg: Peg, height: int) -> list[str]:
    stack = state[peg]
    cells = ["" for _ in range(height - len(stack))]
    cells.extend(f"[{theme.disk_name(disk)}]" for disk in reversed(stack))
    capacity = CAPACITIES[peg]
    for level in range(height):
        if not cells[level] and height - level <= capacity:
            cells[level] = "|"
    return cells

In [ ]:
%%writefile src/libs/outputs/theme.py
# (ignore)
from __future__ import annotations

from collections.abc import Mapping
from typing import Final

from core.rules.domain.base import Disk, Peg
from core.search_tree.outcome import Outcome
from core.search_tree.trace import TraceEvent

DISK_NAMES: Final[Mapping[Disk, str]] = {
    Disk.GREEN: "VERDE",
    Disk.RED: "VERMELHO",
    Disk.BLUE: "AZUL",
}

DISK_SYMBOLS: Final[Mapping[Disk, str]] = {
    Disk.GREEN: "V",
    Disk.RED: "R",
    Disk.BLUE: "A",
}

PEG_NAMES: Final[Mapping[Peg, str]] = {
    Peg.H1: "H1",
    Peg.H2: "H2",
    Peg.H3: "H3",
}

OUTCOME_LABELS: Final[Mapping[Outcome, str]] = {
    Outcome.SUCCESS: "SUCESSO",
    Outcome.DEADLOCK: "IMPASSE",
    Outcome.CUTOFF: "LIMITE",
    Outcome.FAILURE: "FALHA",
}

EVENT_LABELS: Final[Mapping[TraceEvent, str]] = {
    TraceEvent.ROOT: "raiz",
    TraceEvent.VISIT: "visita",
    TraceEvent.GENERATE: "gera",
    TraceEvent.PRUNE: "poda",
    TraceEvent.GOAL: "objetivo",
    TraceEvent.DEADLOCK: "impasse",
    TraceEvent.BACKTRACK: "retrocesso",
    TraceEvent.CUTOFF: "limite",
    TraceEvent.EXHAUSTED: "esgotado",
}

REPORT_LABELS: Final[Mapping[str, str]] = {
    "outcome": "Desfecho",
    "moves": "Movimentos",
    "cost": "Custo",
    "path": "Caminho",
    "iterations": "Iterações",
    "nodes_generated": "Nós gerados",
    "nodes_visited": "Nós visitados",
    "rules_tested": "Regras testadas",
    "backtracks": "Retrocessos",
    "deadlocks": "Impasses",
    "max_depth": "Profundidade máx",
    "max_frontier": "Pico de ABERTOS",
    "elapsed": "Tempo (ms)",
}

LEADERBOARD_HEADERS: Final[tuple[str, ...]] = (
    "#",
    "ALGORITMO",
    "ESTRATÉGIA",
    "DESFECHO",
    "MOV",
    "ITER",
    "GERADOS",
    "ESCORE",
)

SUMMARY_HEADERS: Final[tuple[str, ...]] = (
    "ALGORITMO",
    "ESTRATÉGIA",
    "SUCESSOS",
    "IMPASSES",
    "MOVIMENTOS",
    "CUSTO",
    "ITERAÇÕES",
    "GERADOS",
)

SUMMARY_LEGEND: Final = (
    "Cada métrica: média / mediana. Movimentos e custo contam só os sucessos."
)

HIGHLIGHTS_HEADERS: Final[tuple[str, ...]] = (
    "CRITÉRIO",
    "MELHOR PELA MÉDIA",
    "MELHOR PELA MEDIANA",
)

HIGHLIGHTS_LEGEND: Final = (
    "Menor valor vence. Só concorre quem tem o maior número de sucessos."
)

ALL_STRATEGIES: Final = "(todas)"

CRITERION_LABELS: Final[Mapping[str, str]] = {
    "moves": "Movimentos",
    "cost": "Custo",
    "iterations": "Iterações",
    "nodes_generated": "Nós gerados",
}

TREE_HEADER: Final = "Árvore de busca"
TRACE_HEADER: Final = "Passo a passo"
STATES_HEADER: Final = "Estados do caminho solução"
LEADERBOARD_HEADER: Final = "Placar"
SUMMARY_HEADER: Final = "Resumo consolidado"
HIGHLIGHTS_HEADER: Final = "Melhor por critério"
PROBLEM_HEADER: Final = "PROBLEMA"
INITIAL_STATE_HEADER: Final = "Posição inicial"
GOAL_STATE_HEADER: Final = "Objetivo"

ABSENT: Final = "—"
ARROW: Final = " → "
EMPTY_PEG: Final = "·"

BOX_TOP_LEFT: Final = "╭"
BOX_TOP_RIGHT: Final = "╮"
BOX_BOTTOM_LEFT: Final = "╰"
BOX_BOTTOM_RIGHT: Final = "╯"
BOX_HORIZONTAL: Final = "─"
BOX_VERTICAL: Final = "│"

TREE_BRANCH: Final = "├── "
TREE_LAST_BRANCH: Final = "└── "
TREE_TRUNK: Final = "│   "
TREE_GAP: Final = "    "

GRAPH_FONT: Final = "Helvetica"
GRAPH_MONO_FONT: Final = "monospace"
GRAPH_MOVES_LABEL: Final = "movimentos"
GRAPH_COST_LABEL: Final = "custo"
GRAPH_NO_PRUNE_LABEL: Final = "sem poda"
GRAPH_NO_PRUNE_SUFFIX: Final = "_sem_poda"
GRAPH_ROOT_FILL: Final = "#dbe9ff"
GRAPH_GOAL_FILL: Final = "#c6f0c6"
GRAPH_DEADLOCK_FILL: Final = "#f7c6c6"
GRAPH_VISITED_FILL: Final = "#ffffff"
GRAPH_GENERATED_FILL: Final = "#eeeeee"
GRAPH_NODE_COLOR: Final = "#333333"
GRAPH_PATH_COLOR: Final = "#1a56b8"
GRAPH_PRUNED_COLOR: Final = "#999999"
GRAPH_PATH_WIDTH: Final = 2.5

LABEL_WIDTH: Final = 18
MIN_BOX_WIDTH: Final = 56
MAX_BOX_WIDTH: Final = 76


def disk_symbol(disk: Disk) -> str:
    return DISK_SYMBOLS[disk]


def disk_name(disk: Disk) -> str:
    return DISK_NAMES[disk]


def outcome_label(outcome: Outcome) -> str:
    return OUTCOME_LABELS[outcome]


def event_label(event: TraceEvent) -> str:
    return EVENT_LABELS[event]

In [ ]:
%%writefile src/libs/outputs/trace_render.py
# (ignore)
from __future__ import annotations

from core.search_tree.trace import Trace
from libs.outputs import state_render, theme

_HEADERS = ("ITER", "EVENTO", "NÓ", "PROF", "REGRA", "ESTADO")


def render_trace(trace: Trace) -> str:
    rows = [
        (
            str(step.iteration),
            theme.event_label(step.event),
            f"#{step.node_order}",
            str(step.depth),
            step.rule_id or theme.ABSENT,
            state_render.render_inline(step.state),
        )
        for step in trace
    ]
    return render_table(_HEADERS, tuple(rows))


def render_table(headers: tuple[str, ...], rows: tuple[tuple[str, ...], ...]) -> str:
    widths = [
        max(len(headers[column]), *(len(row[column]) for row in rows))
        if rows
        else len(headers[column])
        for column in range(len(headers))
    ]
    lines = [
        "  ".join(header.ljust(widths[i]) for i, header in enumerate(headers)).rstrip()
    ]
    lines.extend(
        "  ".join(cell.ljust(widths[i]) for i, cell in enumerate(row)).rstrip()
        for row in rows
    )
    return "\n".join(lines)

In [ ]:
%%writefile src/libs/outputs/tree_render.py
# (ignore)
from __future__ import annotations

from core.search_tree.trace import Trace, TraceEvent
from libs.outputs import state_render, theme


def render_tree(trace: Trace) -> str:
    root_steps = trace.of_event(TraceEvent.ROOT)
    if not root_steps:
        return ""

    root = root_steps[0]
    removed = removed_edges(trace)
    children: dict[int, list[tuple[int, str | None, str]]] = {}
    for step in trace.of_event(TraceEvent.GENERATE):
        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:
            continue
        label = state_render.render_inline(step.state)
        children.setdefault(step.parent_order, []).append(
            (step.node_order, step.rule_id, label)
        )

    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}
    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}

    lines = [f"#{root.node_order} {state_render.render_inline(root.state)}"]
    _append_children(lines, root.node_order, children, goal_orders, deadlock_orders, "")
    return "\n".join(lines)


def _append_children(
    lines: list[str],
    parent_order: int,
    children: dict[int, list[tuple[int, str | None, str]]],
    goal_orders: set[int],
    deadlock_orders: set[int],
    prefix: str,
) -> None:
    siblings = children.get(parent_order, ())
    for position, (order, rule_id, label) in enumerate(siblings):
        is_last = position == len(siblings) - 1
        connector = theme.TREE_LAST_BRANCH if is_last else theme.TREE_BRANCH
        marks = ""
        if order in goal_orders:
            marks = f"  ({theme.event_label(TraceEvent.GOAL)})"
        elif order in deadlock_orders:
            marks = f"  ({theme.event_label(TraceEvent.DEADLOCK)})"
        lines.append(f"{prefix}{connector}{rule_id} → #{order} {label}{marks}")
        _append_children(
            lines,
            order,
            children,
            goal_orders,
            deadlock_orders,
            prefix + (theme.TREE_GAP if is_last else theme.TREE_TRUNK),
        )


def removed_edges(trace: Trace) -> set[tuple[int, str | None]]:
    return {
        (step.node_order, step.rule_id) for step in trace.of_event(TraceEvent.PRUNE)
    }

## libs/ranking/ (ignore)

In [ ]:
%%writefile src/libs/ranking/comparator.py
# (ignore)
from __future__ import annotations

from collections.abc import Iterable

from core.search_tree.result import SearchResult
from libs.ranking import criteria


def lexicographic_key(result: SearchResult) -> tuple[int, int, int, int, str]:
    return (
        criteria.outcome_rank(result),
        criteria.solution_length(result),
        criteria.iterations(result),
        criteria.nodes_generated(result),
        criteria.label(result),
    )


def sort_results(results: Iterable[SearchResult]) -> tuple[SearchResult, ...]:
    return tuple(sorted(results, key=lexicographic_key))

In [ ]:
%%writefile src/libs/ranking/criteria.py
# (ignore)
from __future__ import annotations

from collections.abc import Callable, Mapping
from dataclasses import dataclass
from typing import Final

from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult

OUTCOME_ORDER: Final[Mapping[Outcome, int]] = {
    Outcome.SUCCESS: 0,
    Outcome.DEADLOCK: 1,
    Outcome.CUTOFF: 2,
    Outcome.FAILURE: 3,
}

NO_SOLUTION: Final = 10**6


def outcome_rank(result: SearchResult) -> int:
    return OUTCOME_ORDER[result.outcome]


def solution_length(result: SearchResult) -> int:
    length = result.solution_length
    return NO_SOLUTION if length is None else length


def iterations(result: SearchResult) -> int:
    return result.metrics.iterations


def nodes_generated(result: SearchResult) -> int:
    return result.metrics.nodes_generated


def label(result: SearchResult) -> str:
    return f"{result.algorithm}:{result.strategy}"


@dataclass(frozen=True, slots=True)
class Criterion:
    name: str
    measure: Callable[[SearchResult], int]
    weight: float


CRITERIA: Final[tuple[Criterion, ...]] = (
    Criterion("outcome", outcome_rank, 0.50),
    Criterion("solution_length", solution_length, 0.25),
    Criterion("iterations", iterations, 0.15),
    Criterion("nodes_generated", nodes_generated, 0.10),
)

In [ ]:
%%writefile src/libs/ranking/leaderboard.py
# (ignore)
from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from statistics import fmean, median

from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult
from libs.ranking import criteria
from libs.ranking.comparator import lexicographic_key, sort_results
from libs.ranking.scorer import score_all


@dataclass(frozen=True, slots=True)
class LeaderboardRow:
    position: int
    result: SearchResult
    score: float | None


@dataclass(frozen=True, slots=True)
class Leaderboard:
    problem_id: str
    mode: str
    rows: tuple[LeaderboardRow, ...]

    @property
    def winner(self) -> LeaderboardRow | None:
        return self.rows[0] if self.rows else None


@dataclass(frozen=True, slots=True)
class Stat:
    mean: float
    median: float


@dataclass(frozen=True, slots=True)
class SummaryRow:
    algorithm: str
    strategy: str
    runs: int
    successes: int
    deadlocks: int
    moves: Stat | None
    cost: Stat | None
    iterations: Stat
    nodes_generated: Stat


@dataclass(frozen=True, slots=True)
class Highlight:
    criterion: str
    by_mean: tuple[SummaryRow, ...]
    by_median: tuple[SummaryRow, ...]


@dataclass(frozen=True, slots=True)
class Summary:
    rows: tuple[SummaryRow, ...]
    highlights: tuple[Highlight, ...] = ()


SUMMARY_CRITERIA: tuple[tuple[str, Callable[[SummaryRow], Stat | None]], ...] = (
    ("moves", lambda row: row.moves),
    ("cost", lambda row: row.cost),
    ("iterations", lambda row: row.iterations),
    ("nodes_generated", lambda row: row.nodes_generated),
)


def build_leaderboard(
    problem_id: str, results: Sequence[SearchResult], mode: str
) -> Leaderboard:
    scores = score_all(results) if mode == "score" else {}
    ordered = (
        _sorted_by_score(results, scores) if mode == "score" else sort_results(results)
    )
    rows = tuple(
        LeaderboardRow(
            position=position,
            result=result,
            score=scores.get(criteria.label(result)),
        )
        for position, result in enumerate(ordered, start=1)
    )
    return Leaderboard(problem_id=problem_id, mode=mode, rows=rows)


def build_summary(results: Sequence[SearchResult]) -> Summary:
    grouped: dict[tuple[str, str], list[SearchResult]] = {}
    for result in results:
        grouped.setdefault((result.algorithm, result.strategy), []).append(result)

    rows = tuple(
        _summarise(algorithm, strategy, group)
        for (algorithm, strategy), group in grouped.items()
    )
    return Summary(
        rows=tuple(sorted(rows, key=_summary_key)), highlights=_highlights(rows)
    )


def _highlights(rows: Sequence[SummaryRow]) -> tuple[Highlight, ...]:
    if not rows:
        return ()
    most = max(row.successes for row in rows)
    contenders = [row for row in rows if row.successes == most]
    highlights: list[Highlight] = []
    for name, measure in SUMMARY_CRITERIA:
        measured: list[tuple[SummaryRow, Stat]] = []
        for row in contenders:
            stat = measure(row)
            if stat is not None:
                measured.append((row, stat))
        if not measured:
            continue
        best_mean = min(stat.mean for _, stat in measured)
        best_median = min(stat.median for _, stat in measured)
        highlights.append(
            Highlight(
                criterion=name,
                by_mean=tuple(r for r, s in measured if s.mean == best_mean),
                by_median=tuple(r for r, s in measured if s.median == best_median),
            )
        )
    return tuple(highlights)


def _stat(values: Sequence[int]) -> Stat | None:
    if not values:
        return None
    return Stat(mean=fmean(values), median=float(median(values)))


def _sorted_by_score(
    results: Sequence[SearchResult], scores: Mapping[str, float]
) -> tuple[SearchResult, ...]:
    return tuple(
        sorted(
            results,
            key=lambda result: (
                -scores[criteria.label(result)],
                lexicographic_key(result),
            ),
        )
    )


def _summarise(
    algorithm: str, strategy: str, group: Sequence[SearchResult]
) -> SummaryRow:
    moves = [r.solution_length for r in group if r.solution_length is not None]
    costs = [r.solution_cost for r in group if r.solution_cost is not None]
    iterations = _stat([result.metrics.iterations for result in group])
    generated = _stat([result.metrics.nodes_generated for result in group])
    assert iterations is not None and generated is not None
    return SummaryRow(
        algorithm=algorithm,
        strategy=strategy,
        runs=len(group),
        successes=sum(result.outcome.is_success for result in group),
        deadlocks=sum(result.outcome is Outcome.DEADLOCK for result in group),
        moves=_stat(moves),
        cost=_stat(costs),
        iterations=iterations,
        nodes_generated=generated,
    )


def _summary_key(row: SummaryRow) -> tuple[int, float, float, float, str, str]:
    no_solution = float(criteria.NO_SOLUTION)
    return (
        -row.successes,
        row.moves.mean if row.moves is not None else no_solution,
        row.moves.median if row.moves is not None else no_solution,
        row.iterations.mean,
        row.algorithm,
        row.strategy,
    )

In [ ]:
%%writefile src/libs/ranking/scorer.py
# (ignore)
from __future__ import annotations

from collections.abc import Mapping, Sequence

from core.search_tree.result import SearchResult
from libs.ranking import criteria
from libs.ranking.criteria import Criterion

MAX_SCORE = 100.0


def score_all(results: Sequence[SearchResult]) -> Mapping[str, float]:
    if not results:
        return {}
    best = {
        criterion.name: min(criterion.measure(result) for result in results)
        for criterion in criteria.CRITERIA
    }
    return {criteria.label(result): _score(result, best) for result in results}


def _score(result: SearchResult, best: Mapping[str, int]) -> float:
    total = sum(
        criterion.weight * _normalise(criterion, result, best[criterion.name])
        for criterion in criteria.CRITERIA
    )
    return round(total * MAX_SCORE, 2)


def _normalise(criterion: Criterion, result: SearchResult, best: int) -> float:
    value = criterion.measure(result)
    if value == best:
        return 1.0
    if value <= 0:
        return 1.0
    return best / value if best > 0 else 1.0 / (1.0 + value)

## runner/ (ignore)

In [ ]:
%%writefile src/runner/cli.py
# (ignore)
from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

from config import settings
from config.logging import configure_logging
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEMS
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs import parser, validator
from libs.inputs.exceptions import (
    InputError,
    InvalidArgumentError,
    UnknownSelectionError,
)
from libs.inputs.selection import ExecutionRequest
from libs.outputs import console, graph_writer, state_render
from libs.outputs.console import ConsoleFormatter
from libs.outputs.formatter import Formatter, Report
from libs.outputs.json_writer import JsonFormatter, write_file
from runner.pipeline import build_report

_FORMATTERS: dict[str, Formatter] = {
    ConsoleFormatter.name: ConsoleFormatter(),
    JsonFormatter.name: JsonFormatter(),
}


def main(argv: Sequence[str] | None = None) -> int:
    arguments = [] if argv is None else list(argv)
    try:
        request = validator.validate(parser.parse(arguments))
    except InputError as error:
        console.write_error(_error_message(error))
        console.write_error(settings.ERROR_HINT)
        return settings.EXIT_INVALID_INPUT

    configure_logging(verbose=request.verbose)

    if request.show_help:
        console.write(parser.help_text())
        return settings.EXIT_OK

    if request.list_only:
        console.write(_render_catalogues())
        return settings.EXIT_OK

    report = build_report(request)
    _emit(report, request)
    if request.svg_dir is not None:
        _write_graphs(report, request.svg_dir)
    return settings.EXIT_OK


def _write_graphs(report: Report, directory: Path) -> None:
    written = graph_writer.write_graphs(report, directory)
    console.write(settings.GRAPHS_WRITTEN.format(count=len(written), path=directory))
    if not graph_writer.graphviz_available():
        console.write_error(settings.GRAPHVIZ_MISSING)


def _emit(report: Report, request: ExecutionRequest) -> None:
    text = _FORMATTERS[request.output_format].render(report)
    if request.output_path is None:
        console.write(text)
        return
    write_file(request.output_path, text)
    console.write(settings.OUTPUT_WRITTEN.format(path=request.output_path))


def _render_catalogues() -> str:
    problems = "\n".join(
        f"  {problem.id}  {state_render.render_inline(problem.goal)}"
        for problem in PROBLEMS
    )
    return "\n\n".join(
        [
            console.render_section(settings.LIST_PROBLEMS_HEADER, problems),
            console.render_section(
                settings.LIST_ALGORITHMS_HEADER,
                "\n".join(f"  {name}" for name in ALGORITHM_NAMES),
            ),
            console.render_section(
                settings.LIST_STRATEGIES_HEADER,
                "\n".join(f"  {name}" for name in STRATEGY_NAMES),
            ),
        ]
    )


def _error_message(error: Exception) -> str:
    match error:
        case UnknownSelectionError():
            detail = settings.ERROR_UNKNOWN_SELECTION.format(
                kind=settings.KIND_LABELS[error.kind],
                name=error.name,
                available=", ".join(error.available),
            )
        case InvalidArgumentError():
            detail = settings.ERROR_INVALID_ARGUMENT.format(detail=error.detail)
        case _:
            detail = str(error)
    return f"{settings.ERROR_PREFIX} {detail}"

In [ ]:
%%writefile src/runner/executor.py
# (ignore)
from __future__ import annotations

from core.algorithms.domain.registry import get_algorithm
from core.domain.problem import Problem, all_goal_problems, get_problem
from core.rules.strategies.domain.registry import get_strategy
from core.search_tree.result import SearchResult
from core.search_tree.tree import SearchTree
from libs.inputs.selection import ExecutionRequest


def run_problem(
    problem: Problem, request: ExecutionRequest
) -> tuple[SearchResult, ...]:
    return tuple(
        get_algorithm(algorithm_name)(
            SearchTree(),
            get_strategy(strategy_name),
            max_iterations=request.max_iterations,
            prune=request.prune,
        ).solve(problem)
        for algorithm_name in request.algorithm_names
        for strategy_name in request.strategy_names
    )


def run_matrix(
    request: ExecutionRequest,
) -> tuple[tuple[Problem, tuple[SearchResult, ...]], ...]:
    problems = (
        all_goal_problems()
        if request.all_goals
        else tuple(get_problem(problem_id) for problem_id in request.problem_ids)
    )
    return tuple((problem, run_problem(problem, request)) for problem in problems)

In [ ]:
%%writefile src/runner/pipeline.py
# (ignore)
from __future__ import annotations

from libs.inputs.selection import ExecutionRequest
from libs.outputs.formatter import ProblemReport, Report
from libs.ranking.leaderboard import build_leaderboard, build_summary
from runner.executor import run_matrix


def build_report(request: ExecutionRequest) -> Report:
    executed = run_matrix(request)
    problems = tuple(
        ProblemReport(
            problem_id=problem.id,
            initial_state=problem.initial,
            goal_state=problem.goal,
            results=results,
            leaderboard=build_leaderboard(problem.id, results, request.rank_mode),
        )
        for problem, results in executed
    )
    every_result = [result for _, results in executed for result in results]
    return Report(
        problems=problems,
        summary=build_summary(every_result),
        show_tree=request.show_tree,
        show_trace=request.show_trace,
        show_states=request.show_states,
        summary_only=request.all_goals,
    )

## Módulos dos slides (ignore)

In [ ]:
#@title Grava os módulos que aparecem nos slides (o código de cada um está no ponto do slide) (ignore)
from pathlib import Path

FILES = {
    'src/core/rules/domain/base.py': '"""Legenda do problema.\n\nDiscos: VERDE, VERMELHO, AZUL. Distintos por cor; NÃO há ordenação por tamanho.\nHastes: H1, H2, H3.\n\nPesos, usados só no custo das regras (busca ordenada):\n    VERDE: 1    VERMELHO: 2    AZUL: 3\n\nCapacidades:\n    H1: até 3 discos\n    H2: até 2 discos\n    H3: até 1 disco\n\nA capacidade é a única restrição estrutural: é o que substitui a ordenação por\ntamanho da Torre de Hanói.\n\nEstado: tupla de três pilhas, cada uma da base para o topo. Imutável, hashable e\ncanônica: uma configuração física tem exatamente uma escrita.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom enum import IntEnum, unique\nfrom types import MappingProxyType\nfrom typing import Protocol\n\n\n@unique\nclass Disk(IntEnum):\n    GREEN = 0\n    RED = 1\n    BLUE = 2\n\n\n@unique\nclass Peg(IntEnum):\n    H1 = 0\n    H2 = 1\n    H3 = 2\n\n\nCAPACITIES: Mapping[Peg, int] = MappingProxyType({Peg.H1: 3, Peg.H2: 2, Peg.H3: 1})\n\nDISK_WEIGHTS: Mapping[Disk, int] = MappingProxyType(\n    {Disk.GREEN: 1, Disk.RED: 2, Disk.BLUE: 3}\n)\n\nTOTAL_DISKS: int = len(Disk)\n\ntype Stack = tuple[Disk, ...]\ntype State = tuple[Stack, Stack, Stack]\n\n\nclass TransitionRule(Protocol):\n    """Um movimento: tira o disco do topo de uma haste e põe no topo de outra.\n\n        is_applicable(estado)  ->  posso mover agora? (origem tem disco e\n                                   destino tem espaço)\n        apply(estado)          ->  o estado que resulta do movimento\n        distance               ->  quantas hastes o disco atravessa (1 ou 2)\n        cost(estado)           ->  quanto custa mover agora (só a busca\n                                   ordenada usa; depende do disco movido)\n\n    `apply` não mexe no estado recebido, devolve um novo -- por isso um nó da\n    árvore nunca perde a configuração que guardava. São seis regras fixas, R1\n    a R6, uma por par origem/destino, definidas em `moves.py`.\n\n    A busca não sabe o que é disco nem haste: ela só testa a regra, aplica a\n    que passou, e segue. Trocar o problema é trocar as regras, não o motor.\n    """\n\n    @property\n    def id(self) -> str: ...\n\n    @property\n    def origin(self) -> Peg: ...\n\n    @property\n    def destination(self) -> Peg: ...\n\n    @property\n    def distance(self) -> int: ...\n\n    def cost(self, state: State) -> int: ...\n\n    def is_applicable(self, state: State) -> bool: ...\n\n    def apply(self, state: State) -> State: ...\n',
    'src/core/rules/domain/catalog.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom types import MappingProxyType\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.exceptions import UnknownRuleError\nfrom core.rules.moves import R1, R2, R3, R4, R5, R6\n\nRULES: tuple[TransitionRule, ...] = (R1, R2, R3, R4, R5, R6)\n\nRULE_BY_ID: Mapping[str, TransitionRule] = MappingProxyType(\n    {rule.id: rule for rule in RULES}\n)\n\nINVERSE_RULE_ID: Mapping[str, str] = MappingProxyType(\n    {"R1": "R3", "R3": "R1", "R2": "R5", "R5": "R2", "R4": "R6", "R6": "R4"}\n)\n\n\ndef get_rule(rule_id: str) -> TransitionRule:\n    try:\n        return RULE_BY_ID[rule_id]\n    except KeyError:\n        raise UnknownRuleError(rule_id) from None\n\n\ndef inverse_of(rule: TransitionRule) -> TransitionRule:\n    return RULE_BY_ID[INVERSE_RULE_ID[rule.id]]\n',
    'src/core/rules/domain/constraints.py': '"""Condições de validade.\n\n    1. Origem: a haste de origem possui pelo menos um disco no momento exato da\n       ação.\n    2. Destino: a haste de destino não ultrapassa sua capacidade estrita de\n       volume (H1 até 3, H2 até 2, H3 até 1).\n\nAs duas são estáticas: dependem apenas do estado corrente.\n\nA terceira condição do material, que proíbe produzir estado já presente no\ncaminho, é contextual: depende do percurso, e por isso vive no algoritmo\n(`core/search_tree/path.py`), não aqui.\n"""\n\nfrom __future__ import annotations\n\nfrom core.rules.domain.base import CAPACITIES, Peg, State\n\n\ndef has_disk_to_move(state: State, origin: Peg) -> bool:\n    return len(state[origin]) >= 1\n\n\ndef has_room_for_disk(state: State, destination: Peg) -> bool:\n    return len(state[destination]) + 1 <= CAPACITIES[destination]\n\n\ndef is_move_allowed(state: State, origin: Peg, destination: Peg) -> bool:\n    return has_disk_to_move(state, origin) and has_room_for_disk(state, destination)\n',
    'src/core/rules/moves.py': '"""Regras de transição de estado.\n\n    R1: mover o disco do topo da H1 para a H2.\n    R2: mover o disco do topo da H1 para a H3.\n    R3: mover o disco do topo da H2 para a H1.\n    R4: mover o disco do topo da H2 para a H3.\n    R5: mover o disco do topo da H3 para a H1.\n    R6: mover o disco do topo da H3 para a H2.\n\nToda regra tem inversa exata: R1<->R3, R2<->R5, R4<->R6.\n\nCusto de aplicar uma regra num estado:\n\n    custo = 10 + peso do disco movido x distância entre as hastes\n\n    10           toda jogada custa o mesmo tanto, e esse tanto pesa mais que o\n                 esforço (de 1 a 6 por jogada). Em todos os 1.260 pares início\n                 -> objetivo deste espaço, o caminho mais barato é um dos mais\n                 curtos: a busca ordenada não troca movimentos por esforço.\n                 Sem o 10, isso falha em 3,7% das execuções.\n    distância    hastes vizinhas (H1 <-> H2, H2 <-> H3): 1; H1 <-> H3, que pula\n                 a do meio: 2.\n    peso         VERDE 1, VERMELHO 2, AZUL 3 (`DISK_WEIGHTS`).\n\nEntre caminhos com o mesmo número de movimentos, vence o que carrega os discos\nmais pesados por menos distância. Uma regra e sua inversa custam o mesmo: movem\no mesmo disco pela mesma distância. Só a busca ordenada olha o custo.\n\nExemplo: em ([V, R], [A], []), R4 leva o azul de H2 para H3: 10 + 3 x 1 = 13.\n\n`apply` exige `is_applicable` como pré-condição: aplicar regra inválida é erro de\nprogramação e falha alto.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Final\n\nfrom core.rules.domain.base import DISK_WEIGHTS, Peg, State\nfrom core.rules.domain.constraints import is_move_allowed\nfrom core.rules.domain.exceptions import RuleNotApplicableError\n\nMOVE_COST: Final = 10\n\n\n@dataclass(frozen=True, slots=True)\nclass Move:\n    id: str\n    origin: Peg\n    destination: Peg\n\n    @property\n    def distance(self) -> int:\n        return abs(self.destination - self.origin)\n\n    def cost(self, state: State) -> int:\n        if not self.is_applicable(state):\n            raise RuleNotApplicableError(self.id)\n        disk = state[self.origin][-1]\n        return MOVE_COST + DISK_WEIGHTS[disk] * self.distance\n\n    def is_applicable(self, state: State) -> bool:\n        return is_move_allowed(state, self.origin, self.destination)\n\n    def apply(self, state: State) -> State:\n        if not self.is_applicable(state):\n            raise RuleNotApplicableError(self.id)\n\n        stacks = list(state)\n        disk = stacks[self.origin][-1]\n        stacks[self.origin] = stacks[self.origin][:-1]\n        stacks[self.destination] = (*stacks[self.destination], disk)\n        return (stacks[0], stacks[1], stacks[2])\n\n\nR1 = Move("R1", Peg.H1, Peg.H2)\nR2 = Move("R2", Peg.H1, Peg.H3)\nR3 = Move("R3", Peg.H2, Peg.H1)\nR4 = Move("R4", Peg.H2, Peg.H3)\nR5 = Move("R5", Peg.H3, Peg.H1)\nR6 = Move("R6", Peg.H3, Peg.H2)\n',
    'src/core/rules/strategies/ascending.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\n\n_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}\n\n\n@dataclass(frozen=True, slots=True)\nclass AscendingStrategy:\n    name: str = "ascending"\n\n    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:\n        return tuple(sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id]))\n\n\nASCENDING = AscendingStrategy()\n',
    'src/core/rules/strategies/descending.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\n\n_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}\n\n\n@dataclass(frozen=True, slots=True)\nclass DescendingStrategy:\n    name: str = "descending"\n\n    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:\n        return tuple(\n            sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id], reverse=True)\n        )\n\n\nDESCENDING = DescendingStrategy()\n',
    'src/core/domain/problem.py': '"""Estado inicial e catálogo de cartas.\n\nInicial, fixo pelo aparato:  H1[VERDE, VERMELHO]  H2[AZUL]  H3[]\nCada carta define um objetivo a alcançar a partir dele.\n\n`Problem` valida inicial e objetivo na construção: carta impossível falha na\nimportação do módulo, não no meio de uma busca. O campo `initial` tem padrão para\nque testes montem cenários sintéticos sem tocar no catálogo.\n\n`all_goal_problems` monta uma carta sintética por estado do espaço (G01 a\nG36), todas a partir do mesmo inicial: é a bateria usada para comparar os\nalgoritmos por média e mediana em vez de por uma carta só.\n\nÚnico ponto a alterar quando o conjunto de cartas muda: os testes usam\npropriedades relativas, nunca o literal do objetivo.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom dataclasses import dataclass\nfrom types import MappingProxyType\n\nfrom core.domain.state import State, build_state, validate\nfrom core.domain.state_space import all_states\nfrom core.rules.domain.base import Disk\n\nGREEN = Disk.GREEN\nRED = Disk.RED\nBLUE = Disk.BLUE\n\n\nclass UnknownProblemError(Exception):\n    def __init__(self, problem_id: str) -> None:\n        super().__init__(f"unknown problem identifier: {problem_id!r}")\n        self.problem_id = problem_id\n\n\nINITIAL_STATE: State = build_state((GREEN, RED), (BLUE,), ())\n\n\n@dataclass(frozen=True, slots=True)\nclass Problem:\n    id: str\n    goal: State\n    initial: State = INITIAL_STATE\n\n    def __post_init__(self) -> None:\n        validate(self.goal)\n        validate(self.initial)\n\n    def is_goal(self, state: State) -> bool:\n        return state == self.goal\n\n\nPROBLEMS: tuple[Problem, ...] = (Problem("P1", build_state((), (RED, GREEN), (BLUE,))),)\n\nPROBLEM_BY_ID: Mapping[str, Problem] = MappingProxyType(\n    {problem.id: problem for problem in PROBLEMS}\n)\n\nPROBLEM_IDS: tuple[str, ...] = tuple(PROBLEM_BY_ID)\n\n\ndef get_problem(problem_id: str) -> Problem:\n    try:\n        return PROBLEM_BY_ID[problem_id]\n    except KeyError:\n        raise UnknownProblemError(problem_id) from None\n\n\ndef all_goal_problems(initial: State = INITIAL_STATE) -> tuple[Problem, ...]:\n    return tuple(\n        Problem(f"G{index:02d}", goal, initial=initial)\n        for index, goal in enumerate(all_states(), start=1)\n    )\n',
    'src/core/search_tree/frontier.py': '"""FRONTEIRA — a lista de ABERTOS.\n\nOs nós já gerados que ainda não foram processados. A busca tira um daqui,\nolha em volta e devolve os filhos. Fronteira vazia, busca encerrada.\n\nQuem sai primeiro define o algoritmo: pilha afunda, fila varre por nível,\nfila de prioridade sai pelo menor custo.\n\nAqui fica só o contrato. Cada implementação mora ao lado do algoritmo que a\nusa: core/algorithms/<algoritmo>/frontier.py.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Protocol\n\nfrom core.search_tree.node import Node\n\n\nclass EmptyFrontierError(Exception):\n    """Tentou tirar um nó de uma fronteira vazia."""\n\n    def __init__(self) -> None:\n        super().__init__("cannot pop from an empty frontier")\n\n\nclass Frontier(Protocol):\n    """Guardar um nó, tirar um nó, contar quantos faltam. Só isso.\n\n    Sem índice, sem busca por estado, sem percorrer o conteúdo: o único\n    acesso é pela ponta. O contrato não diz QUAL nó o pop devolve — é a\n    única coisa que muda entre as implementações, e é o que define a busca.\n    """\n\n    def push(self, node: Node) -> None:\n        """Guarda um nó. Ele sempre entra pelo fim."""\n        ...\n\n    def pop(self) -> Node:\n        """Tira um nó e devolve. QUAL nó é o que muda entre as fronteiras."""\n        ...\n\n    def __len__(self) -> int:\n        """Quantos nós ainda esperam. Zero encerra a busca."""\n        ...\n',
    'src/core/algorithms/domain/base.py': 'from __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass, field\nfrom time import perf_counter\nfrom typing import ClassVar\n\nfrom config import settings\nfrom core.domain.problem import Problem\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.search_tree import path\nfrom core.search_tree.metrics import MetricsCollector\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.result import SearchResult\nfrom core.search_tree.trace import TraceEvent, TraceRecorder\nfrom core.search_tree.tree import SearchTree\n\n\nclass SearchNotStartedError(Exception):\n    def __init__(self) -> None:\n        super().__init__("search context is only available during solve()")\n\n\n@dataclass(slots=True)\nclass _RunContext:\n    problem: Problem\n    root: Node\n    metrics: MetricsCollector = field(default_factory=MetricsCollector)\n    trace: TraceRecorder = field(default_factory=TraceRecorder)\n    goal_node: Node | None = None\n\n\nclass SearchAlgorithm(ABC):\n    """Motor compartilhado pelos algoritmos de busca.\n\n    Visitar, gerar, podar, contar e registrar vivem aqui. Cada algoritmo\n    implementa só o próprio laço, em `_search`.\n    """\n\n    name: ClassVar[str] = "abstract"\n\n    def __init__(\n        self,\n        tree: SearchTree,\n        strategy: ControlStrategy,\n        *,\n        max_iterations: int = settings.MAX_ITERATIONS,\n        prune: bool = True,\n    ) -> None:\n        self._tree = tree\n        self._strategy = strategy\n        self._max_iterations = max_iterations\n        self._prune = prune\n        self._context: _RunContext | None = None\n\n    @property\n    def strategy(self) -> ControlStrategy:\n        return self._strategy\n\n    @property\n    def max_iterations(self) -> int:\n        return self._max_iterations\n\n    @property\n    def prune(self) -> bool:\n        return self._prune\n\n    def solve(self, problem: Problem) -> SearchResult:\n        """Porta de entrada da busca.\n\n        Monta o contexto da execução, cronometra o `_search` do algoritmo\n        concreto e sela métricas, trace e caminho no resultado.\n        """\n        context = _RunContext(problem=problem, root=self._tree.root(problem.initial))\n        self._context = context\n        context.metrics.count_generated(context.root.depth)\n        context.trace.record(iteration=0, event=TraceEvent.ROOT, node=context.root)\n\n        started = perf_counter()\n        try:\n            outcome = self._search(problem)\n        finally:\n            elapsed_ms = (perf_counter() - started) * 1000.0\n            self._context = None\n\n        goal_node = context.goal_node if outcome.is_success else None\n        solution = () if goal_node is None else path.path_to(goal_node)\n        return SearchResult(\n            algorithm=self.name,\n            strategy=self._strategy.name,\n            problem=problem.id,\n            outcome=outcome,\n            solution_path=solution,\n            applied_rules=() if goal_node is None else path.applied_rules(goal_node),\n            metrics=context.metrics.seal(elapsed_ms),\n            trace=context.trace.seal(),\n            pruned=self._prune,\n        )\n\n    @abstractmethod\n    def _search(self, problem: Problem) -> Outcome:\n        """O laço de cada algoritmo. A única coisa que eles não compartilham."""\n        ...\n\n    def _applicable_rules(self, node: Node) -> tuple[TransitionRule, ...]:\n        """As regras que este nó pode usar, prontas para serem tentadas.\n\n        Faz os três passos em ordem: testa quais são aplicáveis, ordena pela\n        estratégia e poda as que repetiriam um estado. Sem poda, o terceiro\n        passo não roda e a árvore aceita estados repetidos.\n        """\n        context = self._require_context()\n        applicable: list[TransitionRule] = []\n        for rule in self._tree.rules:\n            context.metrics.count_rule_test()\n            if rule.is_applicable(node.state):\n                applicable.append(rule)\n\n        allowed: list[TransitionRule] = []\n        for rule in self._strategy.order(tuple(applicable)):\n            successor = rule.apply(node.state)\n            if self._prune and self._is_repetition(node, rule, successor):\n                context.trace.record_prune(\n                    iteration=context.metrics.iterations,\n                    node=node,\n                    rule_id=rule.id,\n                    state=successor,\n                )\n                continue\n            allowed.append(rule)\n        return tuple(allowed)\n\n    def _is_repetition(\n        self, node: Node, rule: TransitionRule, successor: State\n    ) -> bool:\n        """A política de repetição: aqui, um estado já presente no caminho.\n\n        A busca em largura sobrescreve para usar o conjunto global de\n        fechados, que é uma poda mais forte. A ordenada sobrescreve para\n        comparar o custo do novo caminho com o menor já conhecido.\n        """\n        return path.contains_state(node, successor)\n\n    def _require_context(self) -> _RunContext:\n        """O contexto da execução. Só existe durante o `solve`."""\n        if self._context is None:\n            raise SearchNotStartedError\n        return self._context\n\n    def _next_iteration(self) -> bool:\n        """Conta mais uma iteração e aplica o teto.\n\n        Devolve `False` quando o limite estourou, e aí a busca encerra em\n        LIMITE. Chame no topo do laço, antes de qualquer trabalho.\n        """\n        context = self._require_context()\n        if context.metrics.iterations >= self._max_iterations:\n            context.trace.record(\n                iteration=context.metrics.iterations,\n                event=TraceEvent.CUTOFF,\n                node=context.root,\n            )\n            return False\n        context.metrics.count_iteration()\n        return True\n\n    def _observe_frontier(self, size: int) -> None:\n        """Só contabilidade: guarda o maior tamanho que ABERTOS já teve."""\n        self._require_context().metrics.observe_frontier(size)\n\n    def _visit(self, node: Node) -> None:\n        """Só contabilidade: registra que o nó foi olhado. Uma vez por nó."""\n        context = self._require_context()\n        context.metrics.count_visited()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.VISIT, node=node\n        )\n\n    def _expand(self, node: Node, rule: TransitionRule) -> Node:\n        """Aplica a regra e cria o nó filho.\n\n        É o único ponto do motor que faz a árvore crescer.\n        """\n        context = self._require_context()\n        child = self._tree.expand(node, rule)\n        context.metrics.count_generated(child.depth)\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.GENERATE, node=child\n        )\n        return child\n\n    def _succeed(self, node: Node) -> Outcome:\n        """Guarda o nó objetivo, de onde o caminho solução é reconstruído."""\n        context = self._require_context()\n        context.goal_node = node\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.GOAL, node=node\n        )\n        return Outcome.SUCCESS\n\n    def _deadlock(self, node: Node) -> None:\n        """Só contabilidade: este nó nasceu sem saída.\n\n        Não encerra busca alguma. Quem decide o que fazer com o impasse é o\n        algoritmo: a irrevogável desiste, o backtracking retrocede.\n        """\n        context = self._require_context()\n        context.metrics.count_deadlock()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.DEADLOCK, node=node\n        )\n\n    def _backtrack(self, node: Node) -> None:\n        """Só contabilidade: um retrocesso aconteceu.\n\n        Nada é desfeito aqui. O retrocesso em si é o algoritmo deixando de\n        devolver o nó à pilha.\n        """\n        context = self._require_context()\n        context.metrics.count_backtrack()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.BACKTRACK, node=node\n        )\n\n    def _exhausted(self) -> Outcome:\n        """Fim de linha: nada mais a explorar e nenhum objetivo encontrado."""\n        context = self._require_context()\n        context.trace.record(\n            iteration=context.metrics.iterations,\n            event=TraceEvent.EXHAUSTED,\n            node=context.root,\n        )\n        return Outcome.FAILURE\n',
    'src/core/algorithms/irrevocable/algorithm.py': '"""BUSCA IRREVOGÁVEL\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n       H1      H2      H3              H1      H2      H3\n      ┌───┐   ┌───┐   ┌───┐           ┌───┐   ┌───┐   ┌───┐\n      │ R │   │   │   │   │           │   │   │ V │   │   │\n      │ V │   │ A │   │   │           │   │   │ R │   │ A │\n      └───┘   └───┘   └───┘           └───┘   └───┘   └───┘\n\n2. Regras válidas em S0\n\n    R1 = H1 → H2  ✓          R4 = H2 → H3  ✓\n    R2 = H1 → H3  ✓          R5 = H3 → H1  ✗  H3 vazia\n    R3 = H2 → H1  ✓          R6 = H3 → H2  ✗  H3 vazia\n\n3. Critério\n\n    A estratégia ordena as regras válidas e a busca aplica a primeira.\n    Não guarda as outras: cada passo é definitivo e não há retorno.\n    Uma regra que leva a um estado já percorrido é descartada (poda).\n\n4. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1\n\n    Passo 1   S0 = ([V, R], [A], [])\n              válidas em ordem: R4, R3, R2, R1\n              aplica R4 (azul: H2 → H3)\n              ↓\n    Passo 2   S1 = ([V, R], [], [A])\n              válidas em ordem: R6, R5, R1\n              R6 volta a S0: poda.  Aplica R5 (azul: H3 → H1)\n              ↓\n    Passo 3   S2 = ([V, R, A], [], [])\n              válidas em ordem: R2, R1\n              R2 volta a S1: poda.  R1 volta a S0: poda.\n              Nenhuma regra sobrou.\n              ↓\n              ✗ IMPASSE\n\n5. Árvore de busca\n\n    S0 ([V,R], [A], [])\n    └── R4 → S1 ([V,R], [], [A])\n        └── R5 → S2 ([V,R,A], [], [])   (impasse)\n\n6. A mesma carta com a ordem crescente R1 → R2 → R3 → R4 → R5 → R6\n\n    S0 ([V,R],[A],[]) → R1 → R2 → R3 → R3 → R5 → R1 → R1 → R2 → R3 → R3\n       → R5 → R1 → R1 → R2 → Sf ([],[R,V],[A])\n\n    ✓ SUCESSO com 14 movimentos.  O caminho ótimo tem 3.\n\n7. Conclusão\n\n    O caminho nunca repete estado, então a busca sempre para, no máximo\n    depois dos 36 estados. Parar não é chegar: a ordem decrescente termina\n    em impasse e a crescente só chega depois de 14 movimentos.\n    É o único método em que a estratégia decide o desfecho.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import ClassVar\n\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.search_tree.outcome import Outcome\n\n\nclass IrrevocableSearch(SearchAlgorithm):\n    name: ClassVar[str] = "irrevocable"\n\n    def _search(self, problem: Problem) -> Outcome:\n        node = self._require_context().root\n        self._observe_frontier(1)\n        while True:\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            if not candidates:\n                self._deadlock(node)\n                return Outcome.DEADLOCK\n\n            node = self._expand(node, candidates[0])\n',
    'src/core/algorithms/backtracking/algorithm.py': '"""BUSCA COM BACKTRACKING\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n2. Critério\n\n    A mesma descida da busca irrevogável, mas as regras válidas de cada\n    estado ficam guardadas numa pilha. Quando um estado não tem mais regra,\n    ele sai da pilha (retrocesso) e o estado anterior tenta a próxima regra\n    que tinha guardado. Uma regra que leva a um estado já no caminho atual\n    é descartada (poda).\n\n3. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1\n\n    Passo 1   S0 = ([V, R], [A], [])\n              guarda [R4, R3, R2, R1], aplica R4\n              ↓\n    Passo 2   S1 = ([V, R], [], [A])\n              R6 volta a S0: poda.  Guarda [R5, R1], aplica R5\n              ↓\n    Passo 3   S2 = ([V, R, A], [], [])\n              R2 volta a S1 e R1 volta a S0: poda.  Nada guardado.\n              ✗ impasse → retrocesso: S2 sai da pilha\n              ↑\n    Passo 4   S1 = ([V, R], [], [A])   ainda tem [R1], aplica R1\n              ↓\n    Passo 5   S3 = ([V], [R], [A])\n              R3 volta a S1: poda.  Guarda [R6, R5, R1], aplica R6\n              ↓\n              ...  continua descendo; a cada beco sem saída, um retrocesso\n\n    Passo 51  S36 = ([], [R, V], [A])\n              ✓ SUCESSO\n\n4. Árvore de busca (início)\n\n    S0 ([V,R], [A], [])\n    └── R4 → S1 ([V,R], [], [A])\n        ├── R5 → S2 ([V,R,A], [], [])   (impasse, retrocesso)\n        └── R1 → S3 ([V], [R], [A])\n            └── R6 → S4 ([V], [R,A], [])\n                └── R3 → S5 ([V,A], [R], [])\n                    └── R4 → S6 ([V,A], [], [R])\n                        ├── R5 → S7 ([V,A,R], [], [])   (impasse, retrocesso)\n                        └── R1 → S8 ([V], [A], [R])\n                            └── ...  até S36 = Sf\n\n5. Caminho encontrado\n\n    R4 R1 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2\n\n    Número de movimentos: 22       Iterações: 51\n    Retrocessos: 14                Impasses: 8\n\n    Com a ordem crescente: 14 movimentos em 15 iterações, sem retrocesso.\n    O caminho ótimo tem 3.\n\n6. Conclusão\n\n    O impasse da busca irrevogável vira um desvio: nenhum ramo é abandonado\n    sem ser explorado e, como todo estado alcança todo estado, a solução\n    sempre aparece. Só não é a melhor. A busca para no primeiro objetivo\n    que encontra, tenha o caminho o comprimento que tiver.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import ClassVar\n\nfrom core.algorithms.backtracking.frontier import StackFrontier\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.rules.domain.base import TransitionRule\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\n\n\nclass BacktrackingSearch(SearchAlgorithm):\n    name: ClassVar[str] = "backtracking"\n\n    def _search(self, problem: Problem) -> Outcome:\n        """Desce enquanto houver regra; sem regra, retrocede.\n\n        Cada iteração olha a ponta do caminho e faz uma coisa OU a outra:\n        desce, e o nó continua no caminho com um filho por cima; ou retrocede,\n        e o nó sai do caminho, deixando o pai como ponta.\n\n        Sempre encontra solução, se existir — mas é a primeira que aparecer,\n        não a mais curta.\n        """\n        context = self._require_context()\n\n        path = StackFrontier()\n        path.push(context.root)\n        self._observe_frontier(len(path))\n\n        # Para cada nó do caminho, as regras que ele ainda não tentou. Sem\n        # isso não há retrocesso: o nó não saberia por onde continuar.\n        untried: dict[int, deque[TransitionRule]] = {}\n\n        while len(path):\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n\n            # Topo da pilha = ponta do caminho. Ele só volta para a pilha se\n            # ainda tiver regra -- ver o if lá embaixo.\n            node = path.pop()\n\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            remaining = self._remaining_rules(node, untried)\n            if remaining:\n                # DESCE: o nó continua no caminho e o filho entra por cima.\n                path.push(node)\n                path.push(self._expand(node, remaining.popleft()))\n                self._observe_frontier(len(path))\n            else:\n                # RETROCEDE: sem regras, o nó não volta e sai do caminho. Na\n                # próxima iteração o topo é o pai, que tenta a regra seguinte.\n                self._backtrack(node)\n\n        return self._exhausted()\n\n    def _remaining_rules(\n        self, node: Node, untried: dict[int, deque[TransitionRule]]\n    ) -> deque[TransitionRule]:\n        """Na primeira visita monta a lista do nó; nas voltas seguintes apenas\n        devolve o que sobrou dela. _applicable_rules já poda a regra que\n        repetiria um estado do caminho.\n\n        As seis regras que entram nessa lista:\n\n        ```\n        regra | de | para | inversa\n        ------+----+------+--------\n        R1    | H1 | H2   | R3\n        R2    | H1 | H3   | R5\n        R3    | H2 | H1   | R1\n        R4    | H2 | H3   | R6\n        R5    | H3 | H1   | R2\n        R6    | H3 | H2   | R4\n        ```\n\n        Toda regra tem inversa exata, e é daí que vem a necessidade da poda:\n        sem ela, R1 seguido de R3 devolveria a busca ao mesmo estado, sempre.\n        """\n        if node.order not in untried:\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            untried[node.order] = deque(candidates)\n            if not candidates:\n                self._deadlock(node)\n        return untried[node.order]\n',
    'src/core/algorithms/backtracking/frontier.py': '"""FRONTEIRA DO BACKTRACKING — pilha.\n\nO contrato (push, pop, len) está em core/search_tree/frontier.py.\n"""\n\nfrom __future__ import annotations\n\nfrom core.search_tree.frontier import EmptyFrontierError\nfrom core.search_tree.node import Node\n\n\nclass StackFrontier:\n    """PILHA (LIFO): sai o último que entrou. Usada pelo backtracking.\n\n    ```\n    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S3\n    ```\n\n    Entra e sai pelo mesmo lado, então o filho recém-gerado é sempre o\n    próximo a ser olhado: a busca afunda num ramo até o fim.\n    """\n\n    __slots__ = ("_nodes",)\n\n    def __init__(self) -> None:\n        self._nodes: list[Node] = []\n\n    def push(self, node: Node) -> None:\n        """Empilha o nó no topo."""\n        self._nodes.append(node)\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó do topo: o ÚLTIMO que entrou."""\n        if not self._nodes:\n            raise EmptyFrontierError\n        return self._nodes.pop()\n\n    def __len__(self) -> int:\n        """Quantos nós estão empilhados."""\n        return len(self._nodes)\n',
    'src/core/algorithms/breadth_first/algorithm.py': '"""BUSCA EM LARGURA\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n2. Critério\n\n    As regras são analisadas na ordem crescente R1 → R2 → R3 → R4 → R5 → R6.\n    Em vez de escolher uma, a busca aplica todas as válidas e coloca os\n    filhos no fim de ABERTOS. O estado visitado vai para FECHADOS. Uma\n    regra cujo resultado já está em ABERTOS ou FECHADOS é descartada (poda).\n\n3. Execução\n\n    Nível 0\n        ABERTOS = [S0]               FECHADOS = []\n\n    Expansão de S0 = ([V, R], [A], [])\n        R1 → S1 = ([V], [A, R], [])\n        R2 → S2 = ([V], [A], [R])\n        R3 → S3 = ([V, R, A], [], [])\n        R4 → S4 = ([V, R], [], [A])\n        R5, R6: inválidas, H3 vazia\n        ABERTOS = [S1, S2, S3, S4]   FECHADOS = [S0]\n\n    Nível 1\n\n    Expansão de S1 = ([V], [A, R], [])\n        R1: inválida, H2 cheia\n        R2 → S5 = ([], [A, R], [V])\n        R3 → S0: poda\n        R4 → S2: poda\n        ABERTOS = [S2, S3, S4, S5]   FECHADOS = [S0, S1]\n\n    Expansão de S2 = ([V], [A], [R])\n        R1 → S6 = ([], [A, V], [R])\n        R3 → S7 = ([V, A], [], [R])\n        R5 → S0, R6 → S1: poda\n        ABERTOS = [S3, S4, S5, S6, S7]\n\n    Expansão de S3 = ([V, R, A], [], [])\n        R1 → S0, R2 → S4: poda.  Nada novo.\n\n    Expansão de S4 = ([V, R], [], [A])\n        R1 → S8 = ([V], [R], [A])\n        R5 → S3, R6 → S0: poda\n        ABERTOS = [S5, S6, S7, S8]   FECHADOS = [S0, S1, S2, S3, S4]\n\n    Nível 2\n        S5 gera S9.  S6 gera S10.  S7 gera S11 e S12.\n        S8 = ([V], [R], [A]) gera:\n            R1 → S13 = ([], [R, V], [A])   ← é Sf, entra na fila\n            R6 → S14 = ([V], [R, A], [])\n        ABERTOS = [S9, S10, S11, S12, S13, S14]\n\n    Nível 3\n        S9 gera S15 e S16.  S10, S11 e S12 só geram repetidos.\n        Chega a vez de S13 = ([], [R, V], [A])\n        ✓ SUCESSO\n\n4. Árvore de busca\n\n    S0 ([V,R], [A], [])\n    ├── R1 → S1 ([V], [A,R], [])\n    │   └── R2 → S5 ([], [A,R], [V])\n    │       └── R3 → S9 ([R], [A], [V])\n    │           ├── R3 → S15\n    │           └── R5 → S16\n    ├── R2 → S2 ([V], [A], [R])\n    │   ├── R1 → S6 ([], [A,V], [R])\n    │   │   └── R5 → S10\n    │   └── R3 → S7 ([V,A], [], [R])\n    │       ├── R5 → S11\n    │       └── R6 → S12\n    ├── R3 → S3 ([V,R,A], [], [])\n    └── R4 → S4 ([V,R], [], [A])\n        └── R1 → S8 ([V], [R], [A])\n            ├── R1 → S13 ([], [R,V], [A])   (objetivo)\n            └── R6 → S14 ([V], [R,A], [])\n\n5. Caminho solução\n\n    S0 ([V,R], [A], [])\n     │ R4: azul H2 → H3\n     ↓\n    S4 ([V,R], [], [A])\n     │ R1: vermelho H1 → H2\n     ↓\n    S8 ([V], [R], [A])\n     │ R1: verde H1 → H2\n     ↓\n    Sf ([], [R,V], [A])\n\n    Número de movimentos: 3      Iterações: 14      Gerados: 17\n\n6. Conclusão\n\n    Nada de profundidade d+1 é visitado antes de esgotar a profundidade d,\n    então o primeiro caminho até um estado é o mais curto até ele. O objetivo\n    encontrado é o ótimo em movimentos, e por isso este método serve de\n    referência de comprimento para os outros. A estratégia só muda a ordem\n    dentro de um nível.\n\n    A poda aqui é global: um estado descoberto por qualquer ramo nunca é\n    gerado de novo. A irrevogável e o backtracking só evitam repetir dentro\n    do próprio caminho.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import ClassVar\n\nfrom config import settings\nfrom core.algorithms.breadth_first.frontier import QueueFrontier\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.tree import SearchTree\n\n\nclass BreadthFirstSearch(SearchAlgorithm):\n    name: ClassVar[str] = "breadth_first"\n\n    def __init__(\n        self,\n        tree: SearchTree,\n        strategy: ControlStrategy,\n        *,\n        max_iterations: int = settings.MAX_ITERATIONS,\n        prune: bool = True,\n    ) -> None:\n        super().__init__(tree, strategy, max_iterations=max_iterations, prune=prune)\n        self._closed: set[State] = set()\n        self._open_states: set[State] = set()\n\n    def _search(self, problem: Problem) -> Outcome:\n        context = self._require_context()\n        frontier = QueueFrontier()\n        frontier.push(context.root)\n\n        self._closed = set()\n        self._open_states = {context.root.state}\n        self._observe_frontier(len(frontier))\n\n        while len(frontier):\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n\n            node = frontier.pop()\n            self._open_states.discard(node.state)\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            self._closed.add(node.state)\n            self._visit(node)\n            for rule in self._applicable_rules(node):\n                child = self._expand(node, rule)\n                self._open_states.add(child.state)\n                frontier.push(child)\n            self._observe_frontier(len(frontier))\n\n        return self._exhausted()\n\n    def _is_repetition(\n        self, node: Node, rule: TransitionRule, successor: State\n    ) -> bool:\n        return successor in self._closed or successor in self._open_states\n',
    'src/core/algorithms/breadth_first/frontier.py': '"""FRONTEIRA DA BUSCA EM LARGURA — fila.\n\nO contrato (push, pop, len) está em core/search_tree/frontier.py.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import deque\n\nfrom core.search_tree.frontier import EmptyFrontierError\nfrom core.search_tree.node import Node\n\n\nclass QueueFrontier:\n    """FILA (FIFO): sai o primeiro que entrou. Usada pela busca em largura.\n\n    ```\n    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S1\n    ```\n\n    Entra por um lado, sai pelo outro, então os filhos esperam atrás de todo\n    o nível atual: a busca varre nível por nível e acha o caminho mais curto.\n\n    Usa deque, não list, porque popleft é O(1) e list.pop(0) é O(n).\n    """\n\n    __slots__ = ("_nodes",)\n\n    def __init__(self) -> None:\n        self._nodes: deque[Node] = deque()\n\n    def push(self, node: Node) -> None:\n        """Enfileira o nó no fim da fila."""\n        self._nodes.append(node)\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó do início: o PRIMEIRO que entrou.\n\n        É a única linha que difere da StackFrontier, e é ela que troca\n        profundidade por largura.\n        """\n        if not self._nodes:\n            raise EmptyFrontierError\n        return self._nodes.popleft()\n\n    def __len__(self) -> int:\n        """Quantos nós estão na fila."""\n        return len(self._nodes)\n',
    'src/core/algorithms/ordered/algorithm.py': '"""BUSCA ORDENADA\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n2. Custos\n\n    custo da regra = 10 + peso do disco movido x distância entre as hastes\n\n        10          por jogada; pesa mais que o esforço, e em todo par do\n                    espaço o caminho mais barato é um dos mais curtos\n        distância   hastes vizinhas: 1; H1 <-> H3, que pula a do meio: 2\n        peso        verde 1, vermelho 2, azul 3\n\n    Na raiz: R1 (vermelho, 1) 12, R2 (vermelho, 2) 14, R3 (azul, 1) 13,\n    R4 (azul, 1) 13. O custo de um nó é a soma das regras desde a raiz.\n\n3. Critério\n\n    ABERTOS é ordenada pelo MENOR custo. No empate, sai o nó gerado primeiro;\n    entre irmãos de mesmo custo, decide a estratégia (aqui, a ordem crescente\n    R1 → R2 → R3 → R4 → R5 → R6). A busca só encerra quando o objetivo vira\n    o estado atual, não quando ele é gerado.\n\n    Poda pelo vetor de menor custo: o filho cujo estado já foi gerado com\n    custo menor ou igual é descartado. Se o novo custo for menor, o nó\n    antigo sai de ABERTOS e da árvore e o novo entra no lugar.\n\n4. Execução\n\n    Expansão de S0 = ([V, R], [A], []), custo 0\n        R1 → S1 = ([V], [A, R], [])      12\n        R2 → S2 = ([V], [A], [R])        14\n        R3 → S3 = ([V, R, A], [], [])    13\n        R4 → S4 = ([V, R], [], [A])      13\n        ABERTOS  = [S1(12), S3(13), S4(13), S2(14)]\n        FECHADOS = [S0]\n\n    Expansão de S1 = ([V], [A, R], []), custo 12\n        R2 → S5 = ([], [A, R], [V])      24\n        R3 → S0, R4 → estado de S2 (24 ≥ 14): poda\n        ABERTOS  = [S3(13), S4(13), S2(14), S5(24)]\n\n    Expansão de S3 = ([V, R, A], [], []), custo 13\n        R1 → S0, R2 → S4: poda.  ✗ impasse\n        ABERTOS  = [S4(13), S2(14), S5(24)]\n\n    Expansão de S4 = ([V, R], [], [A]), custo 13\n        R1 → S6 = ([V], [R], [A])        25\n        R5, R6: poda\n        ABERTOS  = [S2(14), S5(24), S6(25)]\n\n    Expansão de S2 = ([V], [A], [R]), custo 14\n        R1 → S7 = ([], [A, V], [R])      25\n        R3 → S8 = ([V, A], [], [R])      27\n        ABERTOS  = [S5(24), S6(25), S7(25), S8(27)]\n\n    Expansão de S5 = ([], [A, R], [V]), custo 24\n        R3 → S9 = ([R], [A], [V])        36\n\n    Expansão de S6 = ([V], [R], [A]), custo 25\n        R1 → S10 = ([], [R, V], [A])     36   ← é Sf, mas só entra na fila\n        R5 → S11 = ([V, A], [R], [])     41   (azul, 2 hastes: 10 + 6)\n        R6 → S12 = ([V], [R, A], [])     38\n        ABERTOS  = [S7(25), S8(27), S9(36), S10(36), S12(38), S11(41)]\n\n    Expansão de S7 = ([], [A, V], [R]), custo 25\n        R5 → S13 = ([R], [A, V], [])     39\n\n    Expansão de S8 = ([V, A], [], [R]), custo 27\n        R5 → S14 = ([V, A, R], [], [])   41\n        R6 → S15 = ([V, A], [R], [])     39   (vermelho, 1 haste: 10 + 2)\n        ✓ TROCA: S15 chega ao estado de S11 por 39, menos que 41. S11 sai\n          de ABERTOS e da árvore; S15 entra no lugar.\n        ABERTOS  = [S9(36), S10(36), S12(38), S13(39), S15(39), S14(41)]\n\n    Expansão de S9 = ([R], [A], [V]), custo 36\n        S9 e S10 custam 36; S9 foi gerado antes e sai primeiro.\n        R3 → S16 (49), R5 → S17 (48)\n\n    Chega a vez de S10 = ([], [R, V], [A])\n        ✓ SUCESSO\n\n5. Árvore de busca (custo acumulado entre parênteses)\n\n    S0 ([V,R], [A], [])  (0)\n    ├── R1 → S1 ([V], [A,R], [])  (12)\n    │   └── R2 → S5 ([], [A,R], [V])  (24)\n    │       └── R3 → S9 ([R], [A], [V])  (36)\n    │           ├── R3 → S16  (49)\n    │           └── R5 → S17  (48)\n    ├── R2 → S2 ([V], [A], [R])  (14)\n    │   ├── R1 → S7 ([], [A,V], [R])  (25)\n    │   │   └── R5 → S13  (39)\n    │   └── R3 → S8 ([V,A], [], [R])  (27)\n    │       ├── R5 → S14  (41)\n    │       └── R6 → S15 ([V,A], [R], [])  (39)   (substituiu S11)\n    ├── R3 → S3 ([V,R,A], [], [])  (13)   (impasse)\n    └── R4 → S4 ([V,R], [], [A])  (13)\n        └── R1 → S6 ([V], [R], [A])  (25)\n            ├── R1 → S10 ([], [R,V], [A])  (36)   (objetivo)\n            ├── R5 → S11  (41)   (removido pela troca)\n            └── R6 → S12 ([V], [R,A], [])  (38)\n\n6. Caminho solução\n\n    S0 ([V,R], [A], [])\n     │ R4: azul H2 → H3        10 + 3 x 1 = 13\n     ↓\n    S4 ([V,R], [], [A])\n     │ R1: vermelho H1 → H2    10 + 2 x 1 = 12\n     ↓\n    S6 ([V], [R], [A])\n     │ R1: verde H1 → H2       10 + 1 x 1 = 11\n     ↓\n    Sf ([], [R,V], [A])\n\n    Custo da solução: 13 + 12 + 11 = 36\n    Número de movimentos: 3      Iterações: 11      Gerados: 18\n\n    Com a ordem decrescente: o mesmo caminho, o mesmo custo e também 11\n    iterações.\n\n7. Conclusão\n\n    Um nó só é expandido depois de todos os mais baratos, então quando o\n    objetivo vira o estado atual nenhum nó aberto chega nele mais barato:\n    a solução é a de MENOR CUSTO. Como cada jogada custa 10 e o esforço de\n    uma jogada varia só de 1 a 6, a de menor custo é, em todo par do\n    espaço, uma das de menos movimentos; entre as de mesmo tamanho, vence a\n    que carrega os discos mais pesados por menos distância.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import ClassVar\n\nfrom config import settings\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.algorithms.ordered.frontier import PriorityFrontier\nfrom core.domain.problem import Problem\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.tree import SearchTree\n\n\nclass OrderedSearch(SearchAlgorithm):\n    name: ClassVar[str] = "ordered"\n\n    def __init__(\n        self,\n        tree: SearchTree,\n        strategy: ControlStrategy,\n        *,\n        max_iterations: int = settings.MAX_ITERATIONS,\n        prune: bool = True,\n    ) -> None:\n        super().__init__(tree, strategy, max_iterations=max_iterations, prune=prune)\n        self._best_cost: dict[State, int] = {}\n        self._open: dict[State, Node] = {}\n\n    def _search(self, problem: Problem) -> Outcome:\n        """Sempre expande o nó aberto de menor custo acumulado.\n\n        O objetivo só encerra a busca quando vira o estado atual, não quando\n        é gerado: só aí nenhum nó aberto pode chegar nele mais barato.\n        """\n        context = self._require_context()\n        frontier = PriorityFrontier()\n        frontier.push(context.root)\n\n        # Vetor de menor custo: o custo do caminho mais barato já gerado até\n        # cada estado. Cobre ABERTOS e FECHADOS de uma vez.\n        self._best_cost = {context.root.state: 0}\n        # O nó de cada estado que ainda espera em ABERTOS, para poder tirá-lo\n        # de lá quando um caminho mais barato até o mesmo estado aparecer.\n        self._open = {context.root.state: context.root}\n        self._observe_frontier(len(frontier))\n\n        while len(frontier):\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n\n            node = frontier.pop()\n            if self._open.get(node.state) is node:\n                del self._open[node.state]\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            if not candidates:\n                self._deadlock(node)\n\n            for rule in candidates:\n                child = self._expand(node, rule)\n                older = self._open.get(child.state) if self.prune else None\n                if older is not None:\n                    # Chegou mais barato a um estado que ainda está aberto:\n                    # o nó antigo sai de ABERTOS e da árvore.\n                    frontier.remove(older)\n                    self._discard(older)\n                self._best_cost[child.state] = child.cost\n                self._open[child.state] = child\n                frontier.push(child)\n            self._observe_frontier(len(frontier))\n\n        return self._exhausted()\n\n    def _is_repetition(\n        self, node: Node, rule: TransitionRule, successor: State\n    ) -> bool:\n        """Poda o filho que não melhora o menor custo conhecido do estado.\n\n        Com custos positivos, um estado já fechado nunca é alcançado mais\n        barato depois, então ele sempre cai aqui.\n        """\n        best = self._best_cost.get(successor)\n        return best is not None and node.cost + rule.cost(node.state) >= best\n\n    def _discard(self, node: Node) -> None:\n        """Só contabilidade: registra o nó substituído como poda do pai."""\n        context = self._require_context()\n        parent, rule = node.parent, node.rule\n        assert parent is not None and rule is not None\n        context.trace.record_prune(\n            iteration=context.metrics.iterations,\n            node=parent,\n            rule_id=rule.id,\n            state=node.state,\n        )\n',
    'src/core/algorithms/ordered/frontier.py': '"""FRONTEIRA DA BUSCA ORDENADA — fila de prioridade por custo.\n\nO contrato (push, pop, len) está em core/search_tree/frontier.py.\n"""\n\nfrom __future__ import annotations\n\nimport heapq\n\nfrom core.search_tree.frontier import EmptyFrontierError\nfrom core.search_tree.node import Node\n\n\nclass PriorityFrontier:\n    """FILA ORDENADA POR CUSTO: sai o de menor custo acumulado. Usada pela\n    busca ordenada.\n\n    ```\n    push S1(5), S2(6), S3(6), S4(1)  ->  pop devolve S4, S1, S2, S3\n    ```\n\n    Empate de custo sai o gerado primeiro (menor `order`). Irmãos de mesmo\n    custo nascem na ordem da estratégia, então ela é o desempate entre eles.\n\n    Heap de (custo, ordem, nó): a ordem é única, então o nó nunca é comparado.\n    """\n\n    __slots__ = ("_heap", "_removed")\n\n    def __init__(self) -> None:\n        self._heap: list[tuple[int, int, Node]] = []\n        self._removed: set[int] = set()\n\n    def push(self, node: Node) -> None:\n        """Insere o nó na posição do seu custo."""\n        heapq.heappush(self._heap, (node.cost, node.order, node))\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó de MENOR custo; no empate, o mais antigo."""\n        while self._heap:\n            _, order, node = heapq.heappop(self._heap)\n            if order in self._removed:\n                self._removed.discard(order)\n                continue\n            return node\n        raise EmptyFrontierError\n\n    def remove(self, node: Node) -> None:\n        """Tira um nó que ainda está na fila, sem devolvê-lo.\n\n        A remoção é preguiçosa: o nó só é descartado quando chegaria a vez\n        dele no pop. O nó precisa estar na fila.\n        """\n        self._removed.add(node.order)\n\n    def __len__(self) -> int:\n        """Quantos nós ainda esperam, sem contar os removidos."""\n        return len(self._heap) - len(self._removed)\n',
}
for name, text in FILES.items():
    Path(name).write_text(text, encoding="utf-8")

## Apresentação (ignore)

In [ ]:
#@title Funções de apresentação (ignore)
import heapq
import shutil
import subprocess
from collections import Counter
from functools import cache
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

from config import settings
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import INITIAL_STATE, get_problem
from core.algorithms.breadth_first.algorithm import BreadthFirstSearch
from core.algorithms.ordered.algorithm import OrderedSearch
from core.algorithms.backtracking.frontier import StackFrontier
from core.algorithms.breadth_first.frontier import QueueFrontier
from core.algorithms.ordered.frontier import PriorityFrontier
from core.domain.problem import Problem
from core.domain.state_space import all_states, applicable_rules, shortest_distance, successors
from core.rules.domain.base import CAPACITIES, DISK_WEIGHTS, Disk
from core.rules.domain.catalog import INVERSE_RULE_ID, RULE_BY_ID, RULES
from core.rules.moves import MOVE_COST
from core.rules.strategies.domain.registry import STRATEGIES as STRATEGY_REGISTRY
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from core.search_tree.outcome import Outcome
from core.search_tree.trace import TraceEvent
from core.search_tree.tree import SearchTree
from libs.inputs.selection import ExecutionRequest
from libs.outputs import graph_writer, state_render, theme, trace_render, tree_render
from libs.outputs.formatter import ProblemReport, Report
from runner.pipeline import build_report

LABELS = theme.REPORT_LABELS
ALGORITHM_LABELS = {
    "irrevocable": "Irrevogável",
    "backtracking": "Backtracking",
    "breadth_first": "Largura",
    "ordered": "Ordenada",
}
CHART_ALGORITHMS = tuple(name for name in ALGORITHM_LABELS if name != "irrevocable")
STRATEGY_LABELS = {"ascending": "crescente", "descending": "decrescente"}
SHORT = {"ascending": "cresc.", "descending": "decr."}
SLIDE_STRATEGIES = ("ascending", "descending")
GRAPH_DIR = Path("data")
P1 = get_problem("P1")
LIST_MODES = {
    "irrevocable": "single",
    "backtracking": "stack",
    "breadth_first": "queue",
    "ordered": "priority",
}
LONG_LIST = 8


def run(problem_ids=("P1",), algorithms=ALGORITHM_NAMES, strategies=STRATEGY_NAMES, *, all_goals=False, prune=True, max_iterations=settings.MAX_ITERATIONS):
    return build_report(
        ExecutionRequest(
            problem_ids=tuple(problem_ids),
            algorithm_names=tuple(algorithms),
            strategy_names=tuple(strategies),
            all_goals=all_goals,
            prune=prune,
            max_iterations=max_iterations,
        )
    )


@cache
def solve(algorithm, strategy, prune=True, max_iterations=settings.MAX_ITERATIONS):
    return run(("P1",), (algorithm,), (strategy,), prune=prune, max_iterations=max_iterations).problems[0].results[0]


@cache
def p1_report():
    return run()


@cache
def goals_report():
    return run((), all_goals=True)


def compact(state):
    return "/".join("".join(theme.disk_symbol(disk) for disk in stack) or "–" for stack in state)


def side_by_side(blocks, gap="      "):
    columns = [block.splitlines() for block in blocks]
    height = max(len(lines) for lines in columns)
    columns = [[""] * (height - len(lines)) + lines for lines in columns]
    widths = [max(len(line) for line in lines) for lines in columns]
    return "\n".join(
        gap.join(lines[row].ljust(width) for lines, width in zip(columns, widths)).rstrip()
        for row in range(height)
    )


def show_problem(problem=P1):
    print(
        side_by_side(
            [
                f"INICIAL  {compact(problem.initial)}\n\n{state_render.render_pegs(problem.initial)}",
                "\n\n\n  ——→",
                f"OBJETIVO {problem.id}  {compact(problem.goal)}\n\n{state_render.render_pegs(problem.goal)}",
            ]
        )
    )


def rules_table():
    return pd.DataFrame(
        [
            {
                "Regra": rule.id,
                "Move de": rule.origin.name,
                "Para": rule.destination.name,
                "Inversa": INVERSE_RULE_ID[rule.id],
                "Distância": rule.distance,
                "Custo V / R / A": " / ".join(str(MOVE_COST + DISK_WEIGHTS[disk] * rule.distance) for disk in Disk),
            }
            for rule in RULES
        ]
    ).set_index("Regra")


def show_rule_example(rule_id="R4", state=INITIAL_STATE):
    rule = RULE_BY_ID[rule_id]
    after = rule.apply(state)
    print(
        side_by_side(
            [
                f"S0  {compact(state)}\n\n{state_render.render_pegs(state)}",
                f"\n\n\n  —{rule.id}→",
                f"{compact(after)}\n\n{state_render.render_pegs(after)}",
            ]
        )
    )
    moved = theme.disk_name(state[rule.origin][-1]).lower()
    print(f"\nExemplo: {rule.id} em S0 leva o {moved} de {rule.origin.name} para {rule.destination.name}.")


def strategies_table():
    return pd.DataFrame(
        [
            {
                "Estratégia": name,
                "Ordem das regras": "  ".join(rule.id for rule in STRATEGY_REGISTRY[name].order(RULES)),
                "Na linha de comando": f"--strategy {name}",
            }
            for name in STRATEGY_NAMES
        ]
    ).set_index("Estratégia")


def _nodes(result):
    return {
        step.node_order: step
        for step in result.trace
        if step.event in (TraceEvent.ROOT, TraceEvent.GENERATE)
    }


def _costs(nodes):
    costs = {}
    for order in sorted(nodes):
        step = nodes[order]
        costs[order] = 0 if step.parent_order is None else costs[step.parent_order] + RULE_BY_ID[step.rule_id].cost(nodes[step.parent_order].state)
    return costs


def _names(orders, costs=None):
    if costs is None:
        return " ".join(f"S{order}" for order in orders)
    return " ".join(f"S{order}({costs[order]})" for order in orders)


def _shorten(orders, costs=None):
    if len(orders) <= LONG_LIST:
        return _names(orders, costs)
    ordered = sorted(orders)
    if costs is None and ordered == list(range(ordered[0], ordered[-1] + 1)):
        return f"S{ordered[0]} … S{ordered[-1]}"
    return f"{len(orders)}: {_names(orders[:3], costs)} …"


def lists_table(result):
    mode = LIST_MODES[result.algorithm]
    nodes = _nodes(result)
    costs = _costs(nodes) if mode == "priority" else None
    steps = {}
    for step in result.trace:
        steps.setdefault(step.iteration, []).append(step)

    titles = {
        "stack": ("TOPO DA PILHA", "PILHA", "RETROCEDIDOS"),
        "single": ("NÓ ATUAL", "ABERTOS", "FECHADOS"),
    }.get(mode, ("SAI DE ABERTOS", "ABERTOS", "FECHADOS"))
    opened, closed = [0], []
    swaps = []
    rows = [{"IT": 0, titles[0]: "", "NÍV": "", "GERA": "", "PODA": "", titles[1]: _names(opened, costs), titles[2]: ""}]
    for iteration in range(1, result.metrics.iterations + 1):
        events = steps.get(iteration, [])
        generated = [step for step in events if step.event is TraceEvent.GENERATE]
        goal = next((step for step in events if step.event is TraceEvent.GOAL), None)
        visit = next((step for step in events if step.event is TraceEvent.VISIT), None)
        back = next((step for step in events if step.event is TraceEvent.BACKTRACK), None)
        if goal is not None:
            current = goal.node_order
        elif visit is not None:
            current = visit.node_order
        elif back is not None:
            current = back.node_order
        else:
            current = generated[0].parent_order
        pruned = [step.rule_id for step in events if step.event is TraceEvent.PRUNE and step.node_order == current]
        replaced = [step for step in events if step.event is TraceEvent.PRUNE and step.node_order != current]
        node = nodes[current]
        row = {
            "IT": iteration,
            titles[0]: f"S{current}  {compact(node.state)}",
            "NÍV": node.depth,
            "PODA": " ".join(pruned),
        }
        if goal is not None:
            rows.append({**row, "GERA": "objetivo: SUCESSO", titles[1]: "", titles[2]: ""})
            break
        if mode == "stack":
            if back is not None:
                opened.remove(current)
                closed.append(current)
            opened.extend(step.node_order for step in generated)
        elif mode == "single":
            closed.append(current)
            opened = [step.node_order for step in generated]
        else:
            opened.remove(current)
            closed.append(current)
            swaps = []
            for step in replaced:
                old = next(o for o in opened if nodes[o].parent_order == step.node_order and nodes[o].rule_id == step.rule_id)
                opened.remove(old)
                swaps.append(old)
            opened.extend(step.node_order for step in generated)
            if mode == "priority":
                opened.sort(key=lambda order: (costs[order], order))
        made = "  ".join(f"{step.rule_id}: S{step.node_order}" for step in generated)
        if mode == "priority" and swaps:
            made += "  (troca " + " ".join(f"S{order}" for order in swaps) + ")"
        deadlock = any(step.event is TraceEvent.DEADLOCK for step in events)
        if back is not None:
            made = "impasse, retrocesso" if deadlock else "retrocesso"
        elif not generated:
            made = "nada (impasse)" if deadlock else "nada"
        rows.append({**row, "GERA": made, titles[1]: _shorten(opened, costs), titles[2]: _shorten(closed)})
        swaps = []
    frame = pd.DataFrame(rows).set_index("IT")
    if result.outcome is Outcome.CUTOFF:
        print(f"Parou no limite de {result.metrics.iterations} iterações (LIMITE).")
    return frame


def case(algorithm, strategy, *, prune=True, max_iterations=settings.MAX_ITERATIONS):
    result = solve(algorithm, strategy, prune, max_iterations)
    report = Report(problems=(ProblemReport(P1.id, P1.initial, P1.goal, (result,)),))
    graph_writer.write_graphs(report, GRAPH_DIR)
    suffix = "" if prune else theme.GRAPH_NO_PRUNE_SUFFIX
    svg = GRAPH_DIR / P1.id / f"{algorithm}_{strategy}{suffix}.svg"
    if svg.exists():
        display(SVG(filename=str(svg)))
    else:
        print(tree_render.render_tree(result.trace))
    print(summary_line(result))
    return result


def summary_line(result):
    metrics = result.metrics
    parts = [
        theme.outcome_label(result.outcome),
        f"{metrics.iterations} iterações",
        f"{metrics.nodes_generated} nós gerados",
        f"{metrics.nodes_visited} expandidos",
        f"pico de ABERTOS {metrics.max_frontier}",
    ]
    if result.solution_length is not None:
        parts.append(f"caminho de {result.solution_length} movimentos, custo {result.solution_cost}")
    return " · ".join(parts)


def frontier_demo(algorithm):
    tree = SearchTree()
    root = tree.root(INITIAL_STATE)
    children = [tree.expand(root, rule) for rule in applicable_rules(INITIAL_STATE)]
    frontier = {"backtracking": StackFrontier, "breadth_first": QueueFrontier, "ordered": PriorityFrontier}[algorithm]()
    for child in children:
        frontier.push(child)
    print("entram, nesta ordem:", "  ".join(f"S{c.order} ({c.rule.id}, custo {c.cost})" for c in children))
    print("saem, nesta ordem:  ", "  ".join(f"S{frontier.pop().order}" for _ in children))


def root_children(algorithm):
    rows = []
    for strategy in SLIDE_STRATEGIES:
        ordered = STRATEGY_REGISTRY[strategy].order(applicable_rules(INITIAL_STATE))
        first = lists_table(solve(algorithm, strategy)).loc[1]
        opened_title = "PILHA" if LIST_MODES[algorithm] == "stack" else "ABERTOS"
        rows.append(
            {
                "Estratégia": STRATEGY_LABELS[strategy],
                "Regras válidas na ordem": "  ".join(f"{rule.id} → {compact(rule.apply(INITIAL_STATE))}" for rule in ordered),
                "Gera na iteração 1": first["GERA"],
                f"{opened_title} após a iteração 1": first[opened_title],
            }
        )
    return pd.DataFrame(rows).set_index("Estratégia")


def _levels(result, depth=None):
    nodes = _nodes(result)
    if depth is None:
        depth = max(step.depth for step in nodes.values())
    counts = Counter(step.depth for step in nodes.values() if step.depth <= depth)
    states = {step.state for step in nodes.values() if step.depth <= depth}
    return [counts[level] for level in range(depth + 1)], len(states)


def growth(algorithm, strategy="ascending", max_iterations=settings.MAX_ITERATIONS):
    result = solve(algorithm, strategy, False, max_iterations)
    depth = result.solution_length
    counts, distinct = _levels(result, depth)
    total = sum(counts)
    if len(counts) <= 8:
        print("   →   ".join(f"{count} no nível {level}" for level, count in enumerate(counts)))
    else:
        print(f"{len(counts)} níveis: " + " ".join(str(count) for count in counts))
    where = f"até o nível {depth}" if depth is not None else f"em {result.metrics.iterations} iterações"
    print(f"\n{total} nós {where}, só {distinct} estados distintos.")
    print(f"{1 - distinct / total:.0%} da árvore é repetição.")


def levels_chart(algorithm, strategy="ascending", max_iterations=settings.MAX_ITERATIONS):
    pruned = solve(algorithm, strategy, True)
    loose = solve(algorithm, strategy, False, max_iterations)
    depth = pruned.solution_length if pruned.solution_length is not None else pruned.metrics.max_depth
    with_prune, _ = _levels(pruned, depth)
    without, _ = _levels(loose, depth)
    levels = range(depth + 1)
    width = 0.4
    fig, ax = plt.subplots(figsize=(min(1.2 * len(levels) + 3, 13), 3.6))
    gray = ax.bar([level - width / 2 for level in levels], without, width * 0.95, color="#c5c7cc", label="sem poda")
    blue = ax.bar([level + width / 2 for level in levels], with_prune, width * 0.95, color=ALGORITHM_COLORS[algorithm], label="com poda")
    ax.bar_label(gray, fontsize=8, color=INK, padding=2)
    ax.bar_label(blue, fontsize=8, color=INK, padding=2)
    ax.set_xticks(list(levels), [f"nível {level}" for level in levels])
    ax.set_title(f"nós gerados por nível, {STRATEGY_LABELS[strategy]}", loc="left")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left")
    plt.show()


def show_path(result):
    if not result.solution_path:
        print(f"{theme.outcome_label(result.outcome)}: não há caminho solução.")
        return
    drawings = []
    for node in result.solution_path:
        title = f"nível {node.depth}" if node.rule is None else f"nível {node.depth}  ({node.rule.id})"
        drawings.append(f"{title}\n{compact(node.state)}\n\n{state_render.render_pegs(node.state)}")
    for start in range(0, len(drawings), 4):
        print(side_by_side(drawings[start:start + 4]))
        print()
    for node in result.solution_path[1:]:
        moved = theme.disk_name(node.state[node.rule.destination][-1]).lower()
        print(f"{node.rule.id}: {moved} de {node.rule.origin.name} para {node.rule.destination.name}  (custo acumulado {node.cost})")
    print(f"\n{result.solution_length} movimentos, custo {result.solution_cost}.")


def _bold_columns(frame, algorithm):
    label = ALGORITHM_LABELS[algorithm]
    columns = [column for column in frame.columns if column.startswith(label)]
    return frame.style.set_properties(subset=columns, **{"font-weight": "bold"})


def _metrics_column(result):
    metrics = result.metrics
    return {
        "Desfecho": theme.outcome_label(result.outcome),
        "Nível da solução": theme.ABSENT if result.solution_length is None else str(result.solution_length),
        "Caminho": " ".join(result.applied_rules) or theme.ABSENT,
        "Custo": theme.ABSENT if result.solution_cost is None else str(result.solution_cost),
        "Iterações": str(metrics.iterations),
        "Nós gerados": str(metrics.nodes_generated),
        "Nós expandidos": str(metrics.nodes_visited),
        "Pico de ABERTOS": str(metrics.max_frontier),
        "Profundidade máx": str(metrics.max_depth),
        "Retrocessos / impasses": f"{metrics.backtracks} / {metrics.deadlocks}",
    }


def strategy_table(algorithm):
    columns = {}
    for prune in (False, True):
        for strategy in SLIDE_STRATEGIES:
            label = f"{SHORT[strategy]} {'com' if prune else 'sem'} poda"
            columns[label] = _metrics_column(solve(algorithm, strategy, prune))
    return pd.DataFrame(columns)


def algorithms_table(algorithm):
    columns = {
        f"{ALGORITHM_LABELS[name]} {SHORT[strategy]}": _metrics_column(solve(name, strategy))
        for name in ALGORITHM_NAMES
        for strategy in SLIDE_STRATEGIES
    }
    frame = pd.DataFrame(columns)
    contenders = [
        f"{ALGORITHM_LABELS[name]} {SHORT[strategy]}"
        for name in CHART_ALGORITHMS
        for strategy in SLIDE_STRATEGIES
        if solve(name, strategy).outcome.is_success
    ]
    return _bold_columns(frame, algorithm).apply(_champions, subset=contenders, axis=1)


CHAMPION_ROWS = ("Custo", "Nível da solução", "Iterações", "Nós gerados", "Nós expandidos", "Pico de ABERTOS", "Retrocessos / impasses")


def _champions(row):
    if row.name not in CHAMPION_ROWS:
        return ["" for _ in row]
    values = [int(str(value).split(" / ")[0]) for value in row]
    return ["color: #1a8746; font-weight: bold" if value == min(values) else "" for value in values]


def _charted(algorithm):
    return [name for name in ALGORITHM_NAMES if name in CHART_ALGORITHMS or name == algorithm]


def paths_plot(algorithm):
    rows = [(name, strategy, solve(name, strategy)) for name in _charted(algorithm) for strategy in SLIDE_STRATEGIES]
    colors = {"ascending": "#2a78d6", "descending": "#c87400"}
    longest = max(len(result.applied_rules) for _, _, result in rows)
    fig, ax = plt.subplots(figsize=(min(3 + 0.45 * longest, 14), 0.75 * len(rows) + 0.6))
    for row, (name, strategy, result) in enumerate(rows):
        y = len(rows) - row
        rules = result.applied_rules
        label = f"{ALGORITHM_LABELS[name]} {SHORT[strategy]}"
        weight = "bold" if name == algorithm else "normal"
        ax.text(-3.6, y, label, ha="right", va="center", color=colors[strategy], fontweight=weight, fontsize=10)
        count = f"{len(rules)} · custo {result.solution_cost}" if result.outcome.is_success else theme.outcome_label(result.outcome).lower()
        ax.text(-0.3, y, count, ha="right", va="center", fontweight="bold", fontsize=10, color="#0b0b0b")
        if rules:
            ax.plot(range(len(rules)), [y] * len(rules), color="#c5c7cc", linewidth=1, zorder=1)
        for position, rule_id in enumerate(rules):
            last = position == len(rules) - 1
            ax.scatter(position, y, s=60, color="#1baf7a" if last else colors[strategy], zorder=2)
            ax.text(position, y + 0.28, rule_id, ha="center", fontsize=7, color=MUTED)
    ax.set_xlim(-7.5, longest)
    ax.set_ylim(0.3, len(rows) + 0.7)
    ax.axis("off")
    ax.set_title("caminhos encontrados em P1 (com poda): movimentos · custo", loc="left", fontsize=10, color=INK)
    plt.show()


def expanded_chart(algorithm):
    bars = []
    for prune in (True, False):
        for strategy in SLIDE_STRATEGIES:
            result = solve(algorithm, strategy, prune)
            label = f"{ALGORITHM_LABELS[algorithm]} {SHORT[strategy]} {'com' if prune else 'sem'} poda"
            bars.append((label, result, algorithm))
    for name in _charted(algorithm):
        if name == algorithm:
            continue
        for strategy in SLIDE_STRATEGIES:
            bars.append((f"{ALGORITHM_LABELS[name]} {SHORT[strategy]} com poda", solve(name, strategy), name))
    finished = [bar for bar in bars if bar[1].outcome is not Outcome.CUTOFF]
    finished.sort(key=lambda bar: bar[1].metrics.nodes_visited)
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(finished) + 1))
    shown = ax.barh(
        [label for label, _, _ in finished],
        [result.metrics.nodes_visited for _, result, _ in finished],
        color=[ALGORITHM_COLORS[name] for _, _, name in finished],
        height=0.7,
    )
    ax.bar_label(shown, fontsize=8, color=INK, padding=3)
    for tick, (_, _, name) in zip(ax.get_yticklabels(), finished):
        tick.set_fontweight("bold" if name == algorithm else "normal")
    ax.invert_yaxis()
    ax.set_title("nós expandidos em P1", loc="left")
    ax.grid(axis="y", visible=False)
    plt.show()
    for label, result, _ in bars:
        if result.outcome is Outcome.CUTOFF:
            print(f"{label}: fora do gráfico, gira até o limite de {result.metrics.iterations} iterações.")
    solved = [bar for bar in finished if bar[1].outcome.is_success]
    cheapest = min(result.solution_cost for _, result, _ in solved)
    print("Menor custo:", "; ".join(label for label, result, _ in solved if result.solution_cost == cheapest), f"(custo {cheapest})")
    shortest = min(result.solution_length for _, result, _ in solved)
    print("Menor nível:", "; ".join(label for label, result, _ in solved if result.solution_length == shortest), f"(nível {shortest})")
    fewest = min(result.metrics.nodes_visited for _, result, _ in solved)
    print("Menos nós expandidos:", "; ".join(label for label, result, _ in solved if result.metrics.nodes_visited == fewest), f"({fewest})")


SYMBOLS = {
    "b": ("fator de ramificação", "quantas regras valem, em média, em cada nó expandido"),
    "d": ("nível da solução", "profundidade do objetivo mais raso"),
    "m": ("profundidade máxima", "até onde o caminho desce"),
    "V": ("vértices", "os estados possíveis"),
    "E": ("arestas", "movimentos entre estados"),
    "C*": ("custo ótimo", "custo da solução mais barata"),
    "ε": ("menor custo de regra", "a regra mais barata"),
}


def _max_depths(algorithm):
    if algorithm is None:
        return "; ".join(
            f"{ALGORITHM_LABELS[name]} " + "/".join(str(solve(name, strategy).metrics.max_depth) for strategy in SLIDE_STRATEGIES)
            for name in ALGORITHM_NAMES
        ) + " (cresc./decr.)"
    return " · ".join(f"{solve(algorithm, strategy).metrics.max_depth} no {SHORT[strategy]}" for strategy in SLIDE_STRATEGIES)


def complexity_values(algorithm, symbols):
    states = all_states()
    loose = solve("breadth_first", "ascending", False)
    expanded = [step.state for step in loose.trace.of_event(TraceEvent.VISIT)]
    values = {
        "b": f"cerca de {sum(len(successors(state)) for state in expanded) / len(expanded):.1f}".replace(".", ","),
        "d": str(solve("breadth_first", "ascending").solution_length),
        "m": _max_depths(algorithm),
        "V": str(len(states)),
        "E": str(sum(len(successors(state)) for state in states) // 2),
        "C*": str(solve("ordered", "ascending").solution_cost),
        "ε": str(min(rule.cost(state) for state in states for rule in RULES if rule.is_applicable(state))),
    }
    return pd.DataFrame(
        [{"Símbolo": symbol, "Nome": SYMBOLS[symbol][0], "O que é": SYMBOLS[symbol][1], "Em P1": values[symbol]} for symbol in symbols]
    ).set_index("Símbolo")


def _stat(value):
    return theme.ABSENT if value is None else f"{value.mean:.2f} / {value.median:g}"


def _bold_rows(frame, algorithm):
    label = ALGORITHM_LABELS[algorithm]
    return frame.style.apply(
        lambda row: ["font-weight: bold" if row["Algoritmo"] == label else "" for _ in row],
        axis=1,
    ).hide(axis="index")


def p1_comparison(algorithm=None):
    frame = pd.DataFrame(
        [
            {
                "#": row.position,
                "Algoritmo": ALGORITHM_LABELS[row.result.algorithm],
                "Estratégia": STRATEGY_LABELS[row.result.strategy],
                LABELS["outcome"]: theme.outcome_label(row.result.outcome),
                LABELS["moves"]: row.result.solution_length,
                LABELS["cost"]: row.result.solution_cost,
                LABELS["iterations"]: row.result.metrics.iterations,
                LABELS["nodes_generated"]: row.result.metrics.nodes_generated,
            }
            for row in p1_report().problems[0].leaderboard.rows
        ]
    ).astype({LABELS["moves"]: "Int64", LABELS["cost"]: "Int64"})
    return frame if algorithm is None else _bold_rows(frame, algorithm)


def goals_comparison(algorithm=None):
    frame = pd.DataFrame(
        [
            {
                "Algoritmo": ALGORITHM_LABELS[row.algorithm],
                "Estratégia": STRATEGY_LABELS[row.strategy],
                "Sucessos": f"{row.successes}/{row.runs}",
                "Impasses": row.deadlocks,
                LABELS["moves"]: _stat(row.moves),
                LABELS["cost"]: _stat(row.cost),
                LABELS["iterations"]: _stat(row.iterations),
                LABELS["nodes_generated"]: _stat(row.nodes_generated),
            }
            for row in goals_report().summary.rows
        ]
    )
    return frame if algorithm is None else _bold_rows(frame, algorithm)


def cost_table():
    return pd.DataFrame(
        [
            {
                "Regra": rule.id,
                "Move de": rule.origin.name,
                "Para": rule.destination.name,
                "Distância": rule.distance,
                **{
                    f"{theme.disk_name(disk).lower()} (peso {DISK_WEIGHTS[disk]})": MOVE_COST + DISK_WEIGHTS[disk] * rule.distance
                    for disk in Disk
                },
            }
            for rule in RULES
        ]
    ).set_index("Regra")


def explain_cost(rule_id, state=INITIAL_STATE):
    rule = RULE_BY_ID[rule_id]
    disk = state[rule.origin][-1]
    weight = DISK_WEIGHTS[disk]
    print(
        f"{rule.id} em {compact(state)}: leva o {theme.disk_name(disk).lower()} (peso {weight}) "
        f"de {rule.origin.name} para {rule.destination.name} (distância {rule.distance})"
    )
    print(f"custo = {MOVE_COST} + {weight} × {rule.distance} = {rule.cost(state)}")


def effort_range():
    efforts = [rule.cost(state) - MOVE_COST for state in all_states() for rule in RULES if rule.is_applicable(state)]
    print(f"esforço por jogada (peso × distância): de {min(efforts)} a {max(efforts)}")
    print(f"custo por jogada: de {MOVE_COST + min(efforts)} a {MOVE_COST + max(efforts)}")


def _with_steps(result):
    path = result.solution_path
    steps = [f"{child.rule.id}({child.cost - parent.cost})" for parent, child in zip(path, path[1:])]
    return " ".join(steps)


def show_hop(before, rule_id, title=""):
    state = INITIAL_STATE
    for previous in before:
        state = RULE_BY_ID[previous].apply(state)
    rule = RULE_BY_ID[rule_id]
    after = rule.apply(state)
    disk = state[rule.origin][-1]
    if title:
        print(title)
    print(
        side_by_side(
            [
                f"{compact(state)}\n\n{state_render.render_pegs(state)}",
                f"\n\n\n  —{rule.id}→",
                f"{compact(after)}\n\n{state_render.render_pegs(after)}",
            ]
        )
    )
    print(
        f"\n{rule.id} leva o {theme.disk_name(disk).lower()} (peso {DISK_WEIGHTS[disk]}) de {rule.origin.name} para "
        f"{rule.destination.name} (distância {rule.distance}): {MOVE_COST} + {DISK_WEIGHTS[disk]} × {rule.distance} = {rule.cost(state)}\n"
    )


def compare_with_breadth(goal_id, strategy):
    problem = next(p for p in goals_report().problems if p.problem_id == goal_id)
    rows = []
    for name in ("breadth_first", "ordered"):
        result = next(r for r in problem.results if r.algorithm == name and r.strategy == strategy)
        rows.append(
            {
                "Algoritmo": ALGORITHM_LABELS[name],
                "Caminho (custo de cada jogada)": _with_steps(result),
                "Movimentos": result.solution_length,
                "Custo": result.solution_cost,
            }
        )
    breadth_cost = rows[0]["Custo"]
    for row in rows:
        row["Melhoria em relação à largura"] = f"{100 * (breadth_cost - row['Custo']) / breadth_cost:.1f}%".replace(".", ",")
    print(f"{goal_id}: de {compact(problem.initial_state)} até {compact(problem.goal_state)}, ordem {STRATEGY_LABELS[strategy]}")
    return pd.DataFrame(rows).set_index("Algoritmo")


class PricedRule:
    def __init__(self, rule, price):
        self.rule = rule
        self.price = price

    id = property(lambda self: self.rule.id)
    origin = property(lambda self: self.rule.origin)
    destination = property(lambda self: self.rule.destination)
    distance = property(lambda self: self.rule.distance)

    def is_applicable(self, state):
        return self.rule.is_applicable(state)

    def apply(self, state):
        return self.rule.apply(state)

    def cost(self, state):
        return self.price(state, self.rule)


def _arm(state, rule):
    lift = CAPACITIES[rule.origin] - len(state[rule.origin]) + 1
    lower = CAPACITIES[rule.destination] - len(state[rule.destination])
    return lift + rule.distance + lower


def _weight(state, rule):
    return DISK_WEIGHTS[state[rule.origin][-1]]


COST_MODELS = {
    "uniforme (1 por jogada)": lambda state, rule: 1,
    "distância (1 ou 2)": lambda state, rule: rule.distance,
    "distância² (1 ou 4)": lambda state, rule: rule.distance ** 2,
    "braço físico (sobe + anda + desce)": _arm,
    "peso × distância, sem o 10": lambda state, rule: _weight(state, rule) * rule.distance,
    "10 + peso do disco": lambda state, rule: 10 + _weight(state, rule),
    "10 + peso × distância (escolhido)": lambda state, rule: rule.cost(state),
}


def _optimal_counts(source, price):
    dist, count, done = {source: 0}, {source: 1}, set()
    heap, pushed = [(0, 0, source)], 1
    while heap:
        cost, _, state = heapq.heappop(heap)
        if state in done:
            continue
        done.add(state)
        for rule in RULES:
            if not rule.is_applicable(state):
                continue
            nxt, new = rule.apply(state), cost + price(state, rule)
            if nxt not in dist or new < dist[nxt]:
                dist[nxt], count[nxt] = new, count[state]
                heapq.heappush(heap, (new, pushed, nxt))
                pushed += 1
            elif new == dist[nxt]:
                count[nxt] += count[state]
    return count


def _study(models):
    states = all_states()
    pairs = [(s, g) for s in states for g in states if s != g]
    shortest = {(s, g): shortest_distance(s, g) for s, g in pairs}
    breadth = {
        (s, g, name): BreadthFirstSearch(SearchTree(), STRATEGY_REGISTRY[name]).solve(Problem("PAR", g, initial=s))
        for s, g in pairs
        for name in SLIDE_STRATEGIES
    }
    breadth_generated = sum(result.metrics.nodes_generated for result in breadth.values()) / len(breadth)
    rows = []
    for label, price in models:
        rules = tuple(PricedRule(rule, price) for rule in RULES)

        def path_cost(state, rule_ids):
            total = 0
            for rule_id in rule_ids:
                rule = RULE_BY_ID[rule_id]
                total += price(state, rule)
                state = rule.apply(state)
            return total

        runs = minimal = cheaper = unique = depends = swaps = generated_total = extra_moves = 0
        savings = []
        for s in states:
            counts = _optimal_counts(s, price)
            for g in states:
                if g == s:
                    continue
                unique += counts[g] == 1
                paths = set()
                for name in SLIDE_STRATEGIES:
                    result = OrderedSearch(SearchTree(rules), STRATEGY_REGISTRY[name]).solve(Problem("PAR", g, initial=s))
                    generated = [step.state for step in result.trace.of_event(TraceEvent.GENERATE)]
                    reference = breadth[(s, g, name)]
                    reference_cost = path_cost(s, reference.applied_rules)
                    runs += 1
                    minimal += result.solution_length == shortest[(s, g)]
                    cheaper += result.solution_cost < reference_cost
                    swaps += len(generated) - len(set(generated))
                    generated_total += result.metrics.nodes_generated
                    extra_moves += result.solution_length - reference.solution_length
                    savings.append((reference_cost - result.solution_cost) / reference_cost)
                    paths.add(result.applied_rules)
                depends += len(paths) > 1
        rows.append(
            {
                "Modelo de custo": label,
                "Mínimo de movimentos": 100 * minimal / runs,
                "Mais barato que a largura": 100 * cheaper / runs,
                "Ótimo único": 100 * unique / len(pairs),
                "Caminho muda com a estratégia": 100 * depends / len(pairs),
                "Trocas de nó": swaps,
                "Economia média de custo": 100 * sum(savings) / len(savings),
                "Maior economia": 100 * max(savings),
                "Nós gerados vs largura": 100 * (generated_total / runs - breadth_generated) / breadth_generated,
                "Trocas por execução": swaps / runs,
                "Movimentos a mais": extra_moves / runs,
            }
        )
    return pd.DataFrame(rows).set_index("Modelo de custo")


@cache
def cost_study():
    return _study(COST_MODELS.items())


def _path_cost(state, rule_ids, price):
    total = 0
    for rule_id in rule_ids:
        rule = RULE_BY_ID[rule_id]
        total += price(state, rule)
        state = rule.apply(state)
    return total


def _long_hops(state, rule_ids):
    hops = []
    for rule_id in rule_ids:
        rule = RULE_BY_ID[rule_id]
        if rule.distance == 2:
            hops.append(f"{rule.id} leva o {theme.disk_name(state[rule.origin][-1]).lower()}")
        state = rule.apply(state)
    return ", ".join(hops)


def distance_example(goal_id="G17", paths=(("R1", "R2", "R3", "R5"), ("R2", "R1", "R5", "R3"))):
    problem = next(p for p in goals_report().problems if p.problem_id == goal_id)
    weight_only = COST_MODELS["10 + peso do disco"]
    chosen = COST_MODELS["10 + peso × distância (escolhido)"]
    print(f"{goal_id}: de {compact(problem.initial_state)} até {compact(problem.goal_state)}")
    return pd.DataFrame(
        [
            {
                "Caminho": " ".join(path),
                "Viagens longas (H1 ↔ H3)": _long_hops(problem.initial_state, path),
                "Custo com 10 + peso": _path_cost(problem.initial_state, path, weight_only),
                "Custo com 10 + peso × distância": _path_cost(problem.initial_state, path, chosen),
            }
            for path in paths
        ]
    ).set_index("Caminho")


def distance_tiebreak(goal_id="G17"):
    problem = next(p for p in goals_report().problems if p.problem_id == goal_id)
    rows = []
    for label in ("10 + peso do disco", "10 + peso × distância (escolhido)"):
        rules = tuple(PricedRule(rule, COST_MODELS[label]) for rule in RULES)
        for name in SLIDE_STRATEGIES:
            result = OrderedSearch(SearchTree(rules), STRATEGY_REGISTRY[name]).solve(Problem("PAR", problem.goal_state, initial=problem.initial_state))
            rows.append({"Custo": label, "Estratégia": STRATEGY_LABELS[name], "Caminho achado": " ".join(result.applied_rules), "Custo do caminho": result.solution_cost})
    return pd.DataFrame(rows).set_index(["Custo", "Estratégia"])


def show_dot(source):
    if shutil.which("dot") is None:
        print(source)
        return
    display(SVG(subprocess.run(["dot", "-Tsvg"], input=source, capture_output=True, text=True, check=True).stdout))


def show_trace(result):
    print(trace_render.render_trace(result.trace))

In [ ]:
#@title Estilo e funções dos gráficos (ignore)
from functools import cache

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from core.domain.state_space import cheapest_cost, shortest_distance
from core.rules.domain.base import CAPACITIES, Disk, Peg

MATPLOTLIB_VERSION = tuple(int(part) for part in matplotlib.__version__.split(".")[:2])
HORIZONTAL = {"orientation": "horizontal"} if MATPLOTLIB_VERSION >= (3, 10) else {"vert": False}

ALGORITHMS = CHART_ALGORITHMS
STRATEGIES = tuple(STRATEGY_LABELS)
ALGORITHM_COLORS = {
    "irrevocable": "#2a78d6",
    "backtracking": "#eb6834",
    "breadth_first": "#1baf7a",
    "ordered": "#eda100",
}
DISK_COLORS = {Disk.GREEN: "#2e9e44", Disk.RED: "#d63b3b", Disk.BLUE: "#2f6fd6"}
OPTIMUM_COLOR = "#b8b8b4"
INK = "#52514e"
MUTED = "#8a8984"

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "axes.titlesize": 11,
        "axes.titlecolor": "#0b0b0b",
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#e6e6e3",
        "grid.linewidth": 0.8,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
    }
)

BOX_STYLE = {
    "patch_artist": True,
    "showmeans": True,
    "medianprops": {"color": "#0b0b0b", "linewidth": 1.5},
    "meanprops": {"marker": "D", "markerfacecolor": "#ffffff", "markeredgecolor": "#0b0b0b", "markersize": 5},
    "flierprops": {"marker": "o", "markersize": 3, "markerfacecolor": MUTED, "markeredgecolor": "none"},
    "whiskerprops": {"color": MUTED},
    "capprops": {"color": MUTED},
}


def combination_label(algorithm, strategy):
    return f"{ALGORITHM_LABELS[algorithm]} · {STRATEGY_LABELS[strategy]}"


def algorithm_legend(fig, *, optimum=False, y=0.0):
    names = [ALGORITHM_LABELS[name] for name in ALGORITHMS]
    colors = [ALGORITHM_COLORS[name] for name in ALGORITHMS]
    if optimum:
        names.append("ótimo")
        colors.append(OPTIMUM_COLOR)
    handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in colors]
    fig.legend(handles, names, loc="lower center", ncol=len(names), bbox_to_anchor=(0.5, y))


def paint(parts, colors):
    for patch, color in zip(parts["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
        patch.set_alpha(0.85)


def draw_state(ax, state, left, title):
    for peg in Peg:
        x = left + 1.2 * peg.value
        ax.plot([x, x], [0, 0.9 * CAPACITIES[peg] + 0.15], color=MUTED, linewidth=3, solid_capstyle="round", zorder=1)
        ax.text(x, -0.45, peg.name, ha="center", va="center", fontsize=9, color=INK)
        for level, disk in enumerate(state[peg]):
            ax.add_patch(plt.Circle((x, 0.45 + 0.9 * level), 0.4, color=DISK_COLORS[disk], zorder=2))
            ax.text(x, 0.45 + 0.9 * level, theme.disk_symbol(disk), ha="center", va="center", fontsize=9, fontweight="bold", color="#ffffff", zorder=3)
    ax.plot([left - 0.6, left + 3.0], [0, 0], color=INK, linewidth=2)
    ax.text(left + 1.2, 3.25, title, ha="center", va="center", fontsize=11, color="#0b0b0b")


def draw_problem(ax, problem):
    draw_state(ax, problem.initial, 0, f"INICIAL  {compact(problem.initial)}")
    ax.annotate("", xy=(5.2, 1.3), xytext=(3.8, 1.3), arrowprops={"arrowstyle": "-|>", "color": INK, "linewidth": 1.5})
    draw_state(ax, problem.goal, 6.4, f"OBJETIVO {problem.id}  {compact(problem.goal)}")
    ax.set_xlim(-1, 10)
    ax.set_ylim(-0.8, 3.6)
    ax.set_aspect("equal")
    ax.axis("off")


def grouped_bars(ax, frame, column, title, missing="—"):
    width = 0.8 / len(ALGORITHMS)
    positions = np.arange(len(STRATEGIES))
    for index, algorithm in enumerate(ALGORITHMS):
        values = [frame.loc[(algorithm, strategy), column] for strategy in STRATEGIES]
        heights = [0 if pd.isna(value) else value for value in values]
        bars = ax.bar(
            positions + (index - (len(ALGORITHMS) - 1) / 2) * width,
            heights,
            width * 0.9,
            color=ALGORITHM_COLORS[algorithm],
        )
        labels = [missing if pd.isna(value) else f"{value:g}" for value in values]
        ax.bar_label(bars, labels=labels, fontsize=8, color=INK, padding=2)
    ax.set_xticks(positions, [STRATEGY_LABELS[name] for name in STRATEGIES])
    ax.set_title(title, loc="left")
    ax.grid(axis="x", visible=False)


def p1_chart():
    frame = pd.DataFrame(
        [
            {
                "algorithm": result.algorithm,
                "strategy": result.strategy,
                "moves": result.solution_length,
                "iterations": result.metrics.iterations,
                "generated": result.metrics.nodes_generated,
            }
            for result in p1_report().problems[0].results
        ]
    ).set_index(["algorithm", "strategy"]).astype({"moves": "Int64"})
    fig = plt.figure(figsize=(13, 6.8))
    grid = fig.add_gridspec(2, 3, height_ratios=(1, 2))
    draw_problem(fig.add_subplot(grid[0, :]), P1)
    axes = [fig.add_subplot(grid[1, column]) for column in range(3)]
    grouped_bars(axes[0], frame, "moves", "Movimentos da solução (tracejado: ótimo)")
    axes[0].axhline(shortest_distance(P1.initial, P1.goal), color=MUTED, linestyle="--", linewidth=1, zorder=0)
    grouped_bars(axes[1], frame, "iterations", "Iterações")
    grouped_bars(axes[2], frame, "generated", "Nós gerados")
    algorithm_legend(fig)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()


@cache
def goals_runs():
    rows = []
    for entry in goals_report().problems:
        best_moves = shortest_distance(entry.initial_state, entry.goal_state)
        best_cost = cheapest_cost(entry.initial_state, entry.goal_state)
        for result in entry.results:
            rows.append(
                {
                    "goal": entry.problem_id,
                    "algorithm": result.algorithm,
                    "strategy": result.strategy,
                    "success": result.outcome.is_success,
                    "moves": result.solution_length,
                    "cost": result.solution_cost,
                    "iterations": result.metrics.iterations,
                    "generated": result.metrics.nodes_generated,
                    "expanded": result.metrics.nodes_visited,
                    "best_moves": best_moves,
                    "best_cost": best_cost,
                }
            )
    runs = pd.DataFrame(rows)
    runs["extra_moves"] = runs.moves - runs.best_moves
    runs["extra_cost"] = runs.cost - runs.best_cost
    return runs


def _values(runs, algorithm, strategy, column):
    return runs.loc[(runs.algorithm == algorithm) & (runs.strategy == strategy), column].dropna().to_numpy()


def goals_chart():
    runs = goals_runs()
    panels = (
        ("moves", "Movimentos da solução", False),
        ("iterations", "Iterações (escala log)", True),
        ("generated", "Nós gerados (escala log)", True),
    )
    width = 0.8 / len(ALGORITHMS)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
    for ax, (column, title, log) in zip(axes, panels):
        for index, algorithm in enumerate(ALGORITHMS):
            positions = np.arange(len(STRATEGIES)) + (index - (len(ALGORITHMS) - 1) / 2) * width
            data = [_values(runs, algorithm, strategy, column) for strategy in STRATEGIES]
            paint(ax.boxplot(data, positions=positions, widths=width * 0.85, **BOX_STYLE), [ALGORITHM_COLORS[algorithm]] * len(data))
        ticks = [STRATEGY_LABELS[name] for name in STRATEGIES]
        if column == "moves":
            optimum = runs.drop_duplicates("goal").best_moves.to_numpy()
            paint(ax.boxplot([optimum], positions=[len(STRATEGIES)], widths=width * 0.85, **BOX_STYLE), [OPTIMUM_COLOR])
            ticks.append("ótimo")
        ax.set_xticks(range(len(ticks)), ticks)
        if log:
            ax.set_yscale("log")
        ax.set_title(title, loc="left")
        ax.grid(axis="x", visible=False)
    algorithm_legend(fig, optimum=True)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()


def boxplots(ax, runs, column, title):
    data, labels, colors = [], [], []
    for algorithm in ALGORITHMS:
        for strategy in STRATEGIES:
            data.append(_values(runs, algorithm, strategy, column))
            labels.append(combination_label(algorithm, strategy))
            colors.append(ALGORITHM_COLORS[algorithm])
    paint(ax.boxplot(data, **HORIZONTAL, widths=0.6, **BOX_STYLE), colors)
    ax.set_yticks(range(1, len(labels) + 1), labels)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)


GOAL_COLUMNS = {
    "iterations": ("Iterações", 120),
    "generated": ("Nós gerados", 80),
    "expanded": ("Nós expandidos", 120),
    "cost": ("Custo da solução", 440),
}


def _goal_rows(algorithm, column):
    names = [algorithm, *(name for name in ALGORITHMS if name != algorithm)]
    rows = [(name, strategy) for name in names for strategy in STRATEGIES]
    runs = goals_runs()
    return sorted(rows, key=lambda row: round(_values(runs, *row, column).mean(), 9))


def goals_boxplot(algorithm, column):
    title, limit = GOAL_COLUMNS[column]
    runs = goals_runs()
    rows = _goal_rows(algorithm, column)
    data = [_values(runs, name, strategy, column) for name, strategy in rows]
    fig, ax = plt.subplots(figsize=(11, 4.2))
    paint(ax.boxplot(data, **HORIZONTAL, widths=0.6, **BOX_STYLE), [ALGORITHM_COLORS[name] for name, _ in rows])
    ax.set_yticks(range(1, len(rows) + 1), [combination_label(name, strategy) for name, strategy in rows])
    for tick, (name, _) in zip(ax.get_yticklabels(), rows):
        tick.set_fontweight("bold" if name == algorithm else "normal")
    ax.set_xlim(0, limit)
    decimals = 2 if column == "cost" else 1
    for position, values in enumerate(data, start=1):
        mean = values.mean()
        ax.annotate(
            f"{mean:.{decimals}f}".replace(".", ","),
            xy=(mean, position),
            xytext=(0, 13),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            color="#0b0b0b",
        )
        beyond = sorted(value for value in values if value > limit)
        if beyond:
            text = " e ".join(f"{value:,.0f}".replace(",", ".") for value in beyond) + " →"
            ax.annotate(text, xy=(limit, position), xytext=(-4, 0), textcoords="offset points", ha="right", va="center", fontsize=9, color=INK)
    ax.invert_yaxis()
    ax.set_title("Custo da solução: todos os 36 objetivos" if column == "cost" else f"{title} nos 36 objetivos", loc="left")
    ax.grid(axis="y", visible=False)
    plt.show()


def p1_cost_chart(algorithm):
    results = {(result.algorithm, result.strategy): result for result in p1_report().problems[0].results}
    names = [algorithm, *(name for name in ALGORITHMS if name != algorithm)]
    rows = [(name, strategy) for name in names for strategy in STRATEGIES if results[(name, strategy)].solution_cost is not None]
    rows.sort(key=lambda row: results[row].solution_cost)
    fig, ax = plt.subplots(figsize=(11, 3.6))
    bars = ax.barh(range(len(rows)), [results[row].solution_cost for row in rows], color=[ALGORITHM_COLORS[name] for name, _ in rows], height=0.6, alpha=0.85)
    ax.bar_label(bars, fontsize=9, fontweight="bold", color="#0b0b0b", padding=4)
    ax.set_yticks(range(len(rows)), [combination_label(name, strategy) for name, strategy in rows])
    for tick, (name, _) in zip(ax.get_yticklabels(), rows):
        tick.set_fontweight("bold" if name == algorithm else "normal")
    ax.invert_yaxis()
    ax.set_title("Custo da solução: só P1", loc="left")
    ax.grid(axis="y", visible=False)
    plt.show()


def goals_stats(algorithm, column):
    runs = goals_runs()
    rows = []
    for name, strategy in _goal_rows(algorithm, column):
        values = pd.Series(_values(runs, name, strategy, column))
        rows.append(
            {
                "Combinação": combination_label(name, strategy),
                "Mínimo": values.min(),
                "1º quartil": values.quantile(0.25),
                "Mediana": values.median(),
                "3º quartil": values.quantile(0.75),
                "Máximo": values.max(),
                "Média": round(values.mean(), 2),
            }
        )
    return pd.DataFrame(rows).set_index("Combinação")


def quality_chart():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    boxplots(axes[0], goals_runs(), "extra_moves", "Movimentos acima do ótimo")
    boxplots(axes[1], goals_runs(), "extra_cost", "Custo acima do ótimo")
    axes[0].invert_yaxis()
    algorithm_legend(fig)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()


def above_optimum():
    runs = goals_runs()
    for algorithm in ("breadth_first", "ordered"):
        selected = runs[(runs.algorithm == algorithm) & runs.success]
        for column, label in (("extra_moves", "movimentos"), ("extra_cost", "custo")):
            above = int((selected[column] > 0).sum())
            print(f"{ALGORITHM_LABELS[algorithm]:<9} acima do ótimo em {label:<10} {above:>2} de {len(selected)} execuções")


def best_by_criterion():
    def names(rows):
        return "; ".join(combination_label(row.algorithm, row.strategy) for row in rows)

    return pd.DataFrame(
        [
            {
                "Critério": theme.CRITERION_LABELS[item.criterion],
                "Melhor pela média": names(item.by_mean),
                "Melhor pela mediana": names(item.by_median),
            }
            for item in goals_report().summary.highlights
        ]
    ).set_index("Critério")


def permutation_check():
    runs = goals_runs()
    for algorithm in ("breadth_first", "ordered"):
        for strategy in STRATEGIES:
            selected = runs[(runs.algorithm == algorithm) & (runs.strategy == strategy)]
            positions = sorted(selected.iterations)
            print(f"{combination_label(algorithm, strategy):<24} iterações = 1..36: {positions == list(range(1, 37))}   média {selected.iterations.mean():.1f}")

In [ ]:
#@title Fluxogramas (ignore)
FLOWCHARTS = {
    'irrevocable': r'''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz", shape=oval, fillcolor="#dbe9ff"];
  goal [label="é o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  visit [label="visita: regras aplicáveis na ordem da estratégia,\nmenos as que repetem estado do caminho"];
  left [label="sobrou regra?", shape=diamond];
  deadlock [label="IMPASSE", shape=oval, fillcolor="#f7c6c6"];
  apply [label="aplica só a primeira e gera o filho"];
  root -> goal;
  goal -> success [label="sim"];
  goal -> visit [label="não"];
  visit -> left;
  left -> deadlock [label="não"];
  left -> apply [label="sim"];
  apply -> goal;
}''',
    'backtracking': r'''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz na pilha", shape=oval, fillcolor="#dbe9ff"];
  goal [label="topo da pilha\né o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  first [label="primeira vez nele?", shape=diamond];
  visit [label="visita e guarda a lista de regras aplicáveis,\nna ordem da estratégia, sem as que repetem estado do caminho"];
  empty [label="lista vazia?", shape=diamond];
  deadlock [label="impasse", fillcolor="#f7c6c6"];
  left [label="ainda tem\nregra guardada?", shape=diamond];
  back [label="retrocesso: sai da pilha,\no pai volta ao topo"];
  apply [label="tira a próxima, gera o filho e o empilha"];
  root -> goal;
  goal -> success [label="sim"];
  goal -> first [label="não"];
  first -> visit [label="sim"];
  first -> left [label="não"];
  visit -> empty;
  empty -> deadlock [label="sim"];
  empty -> left [label="não"];
  deadlock -> left;
  left -> back [label="não"];
  left -> apply [label="sim"];
  back -> goal;
  apply -> goal;
}''',
    'breadth_first': r'''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz na fila", shape=oval, fillcolor="#dbe9ff"];
  pop [label="tira o primeiro da fila"];
  goal [label="é o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  visit [label="visita: regras aplicáveis na ordem da estratégia,\nmenos as que levam a um estado já gerado por qualquer ramo"];
  apply [label="aplica todas e coloca os filhos no fim da fila"];
  root -> pop;
  pop -> goal;
  goal -> success [label="sim"];
  goal -> visit [label="não"];
  visit -> apply;
  apply -> pop;
}''',
    'ordered': r'''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz em ABERTOS", shape=oval, fillcolor="#dbe9ff"];
  pop [label="tira o de menor custo;\nno empate, o gerado primeiro"];
  goal [label="é o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  visit [label="visita: regras aplicáveis na ordem da estratégia,\nmenos as que não baixam o menor custo conhecido do estado"];
  apply [label="aplica todas; se um filho chega mais barato a um estado ainda aberto,\no nó antigo sai de ABERTOS e da árvore"];
  root -> pop;
  pop -> goal;
  goal -> success [label="sim"];
  goal -> visit [label="não"];
  visit -> apply;
  apply -> pop;
}''',
}


def show_flowchart(algorithm):
    show_dot(FLOWCHARTS[algorithm])

# 1. Busca irrevogável

Um caminho só: a regra preferida da estratégia é aplicada e as outras são esquecidas.

## O problema: carta P1, do estado inicial ao objetivo

Cada estado lista as hastes da base para o topo. As seis regras R1 a R6 movem o disco do topo de uma haste para outra.

A legenda (discos, hastes e capacidades) e o catálogo de cartas, como estão no projeto:

In [ ]:
%%writefile src/core/rules/domain/base.py
"""Legenda do problema.

Discos: VERDE, VERMELHO, AZUL. Distintos por cor; NÃO há ordenação por tamanho.
Hastes: H1, H2, H3.

Pesos, usados só no custo das regras (busca ordenada):
    VERDE: 1    VERMELHO: 2    AZUL: 3

Capacidades:
    H1: até 3 discos
    H2: até 2 discos
    H3: até 1 disco

A capacidade é a única restrição estrutural: é o que substitui a ordenação por
tamanho da Torre de Hanói.

Estado: tupla de três pilhas, cada uma da base para o topo. Imutável, hashable e
canônica: uma configuração física tem exatamente uma escrita.
"""

from __future__ import annotations

from collections.abc import Mapping
from enum import IntEnum, unique
from types import MappingProxyType
from typing import Protocol


@unique
class Disk(IntEnum):
    GREEN = 0
    RED = 1
    BLUE = 2


@unique
class Peg(IntEnum):
    H1 = 0
    H2 = 1
    H3 = 2


CAPACITIES: Mapping[Peg, int] = MappingProxyType({Peg.H1: 3, Peg.H2: 2, Peg.H3: 1})

DISK_WEIGHTS: Mapping[Disk, int] = MappingProxyType(
    {Disk.GREEN: 1, Disk.RED: 2, Disk.BLUE: 3}
)

TOTAL_DISKS: int = len(Disk)

type Stack = tuple[Disk, ...]
type State = tuple[Stack, Stack, Stack]


class TransitionRule(Protocol):
    """Um movimento: tira o disco do topo de uma haste e põe no topo de outra.

        is_applicable(estado)  ->  posso mover agora? (origem tem disco e
                                   destino tem espaço)
        apply(estado)          ->  o estado que resulta do movimento
        distance               ->  quantas hastes o disco atravessa (1 ou 2)
        cost(estado)           ->  quanto custa mover agora (só a busca
                                   ordenada usa; depende do disco movido)

    `apply` não mexe no estado recebido, devolve um novo -- por isso um nó da
    árvore nunca perde a configuração que guardava. São seis regras fixas, R1
    a R6, uma por par origem/destino, definidas em `moves.py`.

    A busca não sabe o que é disco nem haste: ela só testa a regra, aplica a
    que passou, e segue. Trocar o problema é trocar as regras, não o motor.
    """

    @property
    def id(self) -> str: ...

    @property
    def origin(self) -> Peg: ...

    @property
    def destination(self) -> Peg: ...

    @property
    def distance(self) -> int: ...

    def cost(self, state: State) -> int: ...

    def is_applicable(self, state: State) -> bool: ...

    def apply(self, state: State) -> State: ...

In [ ]:
%%writefile src/core/domain/problem.py
"""Estado inicial e catálogo de cartas.

Inicial, fixo pelo aparato:  H1[VERDE, VERMELHO]  H2[AZUL]  H3[]
Cada carta define um objetivo a alcançar a partir dele.

`Problem` valida inicial e objetivo na construção: carta impossível falha na
importação do módulo, não no meio de uma busca. O campo `initial` tem padrão para
que testes montem cenários sintéticos sem tocar no catálogo.

`all_goal_problems` monta uma carta sintética por estado do espaço (G01 a
G36), todas a partir do mesmo inicial: é a bateria usada para comparar os
algoritmos por média e mediana em vez de por uma carta só.

Único ponto a alterar quando o conjunto de cartas muda: os testes usam
propriedades relativas, nunca o literal do objetivo.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
from types import MappingProxyType

from core.domain.state import State, build_state, validate
from core.domain.state_space import all_states
from core.rules.domain.base import Disk

GREEN = Disk.GREEN
RED = Disk.RED
BLUE = Disk.BLUE


class UnknownProblemError(Exception):
    def __init__(self, problem_id: str) -> None:
        super().__init__(f"unknown problem identifier: {problem_id!r}")
        self.problem_id = problem_id


INITIAL_STATE: State = build_state((GREEN, RED), (BLUE,), ())


@dataclass(frozen=True, slots=True)
class Problem:
    id: str
    goal: State
    initial: State = INITIAL_STATE

    def __post_init__(self) -> None:
        validate(self.goal)
        validate(self.initial)

    def is_goal(self, state: State) -> bool:
        return state == self.goal


PROBLEMS: tuple[Problem, ...] = (Problem("P1", build_state((), (RED, GREEN), (BLUE,))),)

PROBLEM_BY_ID: Mapping[str, Problem] = MappingProxyType(
    {problem.id: problem for problem in PROBLEMS}
)

PROBLEM_IDS: tuple[str, ...] = tuple(PROBLEM_BY_ID)


def get_problem(problem_id: str) -> Problem:
    try:
        return PROBLEM_BY_ID[problem_id]
    except KeyError:
        raise UnknownProblemError(problem_id) from None


def all_goal_problems(initial: State = INITIAL_STATE) -> tuple[Problem, ...]:
    return tuple(
        Problem(f"G{index:02d}", goal, initial=initial)
        for index, goal in enumerate(all_states(), start=1)
    )

In [ ]:
show_problem()

## A ideia

**Aplique a primeira regra válida e siga em frente, sem guardar alternativas.**

Não há fila nem pilha: ABERTOS tem no máximo um nó, o filho que acabou de ser gerado, e ele vira o próximo estado atual. Cada passo é definitivo. Um beco sem saída encerra a busca em IMPASSE.

## O laço e as listas

1. O estado atual é o objetivo? **SUCESSO**.
2. Não é: coloca o nó em FECHADOS.
3. Ordena as regras válidas pela estratégia e descarta as que voltam a um estado do caminho (poda).
4. Não sobrou regra: **IMPASSE**.
5. Aplica só a primeira: o filho vira o estado atual. Volta ao passo 1.

**ABERTOS**: o único filho gerado, que vira o próximo estado atual.

**FECHADOS**: o caminho percorrido. Com poda, serve para não voltar a um estado do caminho.

O motor que os quatro algoritmos compartilham: testar as regras, ordenar pela estratégia, podar, gerar o filho e contar. Cada algoritmo implementa só o próprio laço, em `_search`.

In [ ]:
%%writefile src/core/algorithms/domain/base.py
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from time import perf_counter
from typing import ClassVar

from config import settings
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree import path
from core.search_tree.metrics import MetricsCollector
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult
from core.search_tree.trace import TraceEvent, TraceRecorder
from core.search_tree.tree import SearchTree


class SearchNotStartedError(Exception):
    def __init__(self) -> None:
        super().__init__("search context is only available during solve()")


@dataclass(slots=True)
class _RunContext:
    problem: Problem
    root: Node
    metrics: MetricsCollector = field(default_factory=MetricsCollector)
    trace: TraceRecorder = field(default_factory=TraceRecorder)
    goal_node: Node | None = None


class SearchAlgorithm(ABC):
    """Motor compartilhado pelos algoritmos de busca.

    Visitar, gerar, podar, contar e registrar vivem aqui. Cada algoritmo
    implementa só o próprio laço, em `_search`.
    """

    name: ClassVar[str] = "abstract"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
        prune: bool = True,
    ) -> None:
        self._tree = tree
        self._strategy = strategy
        self._max_iterations = max_iterations
        self._prune = prune
        self._context: _RunContext | None = None

    @property
    def strategy(self) -> ControlStrategy:
        return self._strategy

    @property
    def max_iterations(self) -> int:
        return self._max_iterations

    @property
    def prune(self) -> bool:
        return self._prune

    def solve(self, problem: Problem) -> SearchResult:
        """Porta de entrada da busca.

        Monta o contexto da execução, cronometra o `_search` do algoritmo
        concreto e sela métricas, trace e caminho no resultado.
        """
        context = _RunContext(problem=problem, root=self._tree.root(problem.initial))
        self._context = context
        context.metrics.count_generated(context.root.depth)
        context.trace.record(iteration=0, event=TraceEvent.ROOT, node=context.root)

        started = perf_counter()
        try:
            outcome = self._search(problem)
        finally:
            elapsed_ms = (perf_counter() - started) * 1000.0
            self._context = None

        goal_node = context.goal_node if outcome.is_success else None
        solution = () if goal_node is None else path.path_to(goal_node)
        return SearchResult(
            algorithm=self.name,
            strategy=self._strategy.name,
            problem=problem.id,
            outcome=outcome,
            solution_path=solution,
            applied_rules=() if goal_node is None else path.applied_rules(goal_node),
            metrics=context.metrics.seal(elapsed_ms),
            trace=context.trace.seal(),
            pruned=self._prune,
        )

    @abstractmethod
    def _search(self, problem: Problem) -> Outcome:
        """O laço de cada algoritmo. A única coisa que eles não compartilham."""
        ...

    def _applicable_rules(self, node: Node) -> tuple[TransitionRule, ...]:
        """As regras que este nó pode usar, prontas para serem tentadas.

        Faz os três passos em ordem: testa quais são aplicáveis, ordena pela
        estratégia e poda as que repetiriam um estado. Sem poda, o terceiro
        passo não roda e a árvore aceita estados repetidos.
        """
        context = self._require_context()
        applicable: list[TransitionRule] = []
        for rule in self._tree.rules:
            context.metrics.count_rule_test()
            if rule.is_applicable(node.state):
                applicable.append(rule)

        allowed: list[TransitionRule] = []
        for rule in self._strategy.order(tuple(applicable)):
            successor = rule.apply(node.state)
            if self._prune and self._is_repetition(node, rule, successor):
                context.trace.record_prune(
                    iteration=context.metrics.iterations,
                    node=node,
                    rule_id=rule.id,
                    state=successor,
                )
                continue
            allowed.append(rule)
        return tuple(allowed)

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        """A política de repetição: aqui, um estado já presente no caminho.

        A busca em largura sobrescreve para usar o conjunto global de
        fechados, que é uma poda mais forte. A ordenada sobrescreve para
        comparar o custo do novo caminho com o menor já conhecido.
        """
        return path.contains_state(node, successor)

    def _require_context(self) -> _RunContext:
        """O contexto da execução. Só existe durante o `solve`."""
        if self._context is None:
            raise SearchNotStartedError
        return self._context

    def _next_iteration(self) -> bool:
        """Conta mais uma iteração e aplica o teto.

        Devolve `False` quando o limite estourou, e aí a busca encerra em
        LIMITE. Chame no topo do laço, antes de qualquer trabalho.
        """
        context = self._require_context()
        if context.metrics.iterations >= self._max_iterations:
            context.trace.record(
                iteration=context.metrics.iterations,
                event=TraceEvent.CUTOFF,
                node=context.root,
            )
            return False
        context.metrics.count_iteration()
        return True

    def _observe_frontier(self, size: int) -> None:
        """Só contabilidade: guarda o maior tamanho que ABERTOS já teve."""
        self._require_context().metrics.observe_frontier(size)

    def _visit(self, node: Node) -> None:
        """Só contabilidade: registra que o nó foi olhado. Uma vez por nó."""
        context = self._require_context()
        context.metrics.count_visited()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.VISIT, node=node
        )

    def _expand(self, node: Node, rule: TransitionRule) -> Node:
        """Aplica a regra e cria o nó filho.

        É o único ponto do motor que faz a árvore crescer.
        """
        context = self._require_context()
        child = self._tree.expand(node, rule)
        context.metrics.count_generated(child.depth)
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.GENERATE, node=child
        )
        return child

    def _succeed(self, node: Node) -> Outcome:
        """Guarda o nó objetivo, de onde o caminho solução é reconstruído."""
        context = self._require_context()
        context.goal_node = node
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.GOAL, node=node
        )
        return Outcome.SUCCESS

    def _deadlock(self, node: Node) -> None:
        """Só contabilidade: este nó nasceu sem saída.

        Não encerra busca alguma. Quem decide o que fazer com o impasse é o
        algoritmo: a irrevogável desiste, o backtracking retrocede.
        """
        context = self._require_context()
        context.metrics.count_deadlock()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.DEADLOCK, node=node
        )

    def _backtrack(self, node: Node) -> None:
        """Só contabilidade: um retrocesso aconteceu.

        Nada é desfeito aqui. O retrocesso em si é o algoritmo deixando de
        devolver o nó à pilha.
        """
        context = self._require_context()
        context.metrics.count_backtrack()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.BACKTRACK, node=node
        )

    def _exhausted(self) -> Outcome:
        """Fim de linha: nada mais a explorar e nenhum objetivo encontrado."""
        context = self._require_context()
        context.trace.record(
            iteration=context.metrics.iterations,
            event=TraceEvent.EXHAUSTED,
            node=context.root,
        )
        return Outcome.FAILURE

## As regras: seis regras, uma para cada par origem e destino

Vale se a origem tem disco e o destino tem espaço. Toda regra tem inversa: aplicar as duas em seguida volta ao mesmo estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

In [ ]:
%%writefile src/core/rules/domain/constraints.py
"""Condições de validade.

    1. Origem: a haste de origem possui pelo menos um disco no momento exato da
       ação.
    2. Destino: a haste de destino não ultrapassa sua capacidade estrita de
       volume (H1 até 3, H2 até 2, H3 até 1).

As duas são estáticas: dependem apenas do estado corrente.

A terceira condição do material, que proíbe produzir estado já presente no
caminho, é contextual: depende do percurso, e por isso vive no algoritmo
(`core/search_tree/path.py`), não aqui.
"""

from __future__ import annotations

from core.rules.domain.base import CAPACITIES, Peg, State


def has_disk_to_move(state: State, origin: Peg) -> bool:
    return len(state[origin]) >= 1


def has_room_for_disk(state: State, destination: Peg) -> bool:
    return len(state[destination]) + 1 <= CAPACITIES[destination]


def is_move_allowed(state: State, origin: Peg, destination: Peg) -> bool:
    return has_disk_to_move(state, origin) and has_room_for_disk(state, destination)

In [ ]:
%%writefile src/core/rules/moves.py
"""Regras de transição de estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

Toda regra tem inversa exata: R1<->R3, R2<->R5, R4<->R6.

Custo de aplicar uma regra num estado:

    custo = 10 + peso do disco movido x distância entre as hastes

    10           toda jogada custa o mesmo tanto, e esse tanto pesa mais que o
                 esforço (de 1 a 6 por jogada). Em todos os 1.260 pares início
                 -> objetivo deste espaço, o caminho mais barato é um dos mais
                 curtos: a busca ordenada não troca movimentos por esforço.
                 Sem o 10, isso falha em 3,7% das execuções.
    distância    hastes vizinhas (H1 <-> H2, H2 <-> H3): 1; H1 <-> H3, que pula
                 a do meio: 2.
    peso         VERDE 1, VERMELHO 2, AZUL 3 (`DISK_WEIGHTS`).

Entre caminhos com o mesmo número de movimentos, vence o que carrega os discos
mais pesados por menos distância. Uma regra e sua inversa custam o mesmo: movem
o mesmo disco pela mesma distância. Só a busca ordenada olha o custo.

Exemplo: em ([V, R], [A], []), R4 leva o azul de H2 para H3: 10 + 3 x 1 = 13.

`apply` exige `is_applicable` como pré-condição: aplicar regra inválida é erro de
programação e falha alto.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Final

from core.rules.domain.base import DISK_WEIGHTS, Peg, State
from core.rules.domain.constraints import is_move_allowed
from core.rules.domain.exceptions import RuleNotApplicableError

MOVE_COST: Final = 10


@dataclass(frozen=True, slots=True)
class Move:
    id: str
    origin: Peg
    destination: Peg

    @property
    def distance(self) -> int:
        return abs(self.destination - self.origin)

    def cost(self, state: State) -> int:
        if not self.is_applicable(state):
            raise RuleNotApplicableError(self.id)
        disk = state[self.origin][-1]
        return MOVE_COST + DISK_WEIGHTS[disk] * self.distance

    def is_applicable(self, state: State) -> bool:
        return is_move_allowed(state, self.origin, self.destination)

    def apply(self, state: State) -> State:
        if not self.is_applicable(state):
            raise RuleNotApplicableError(self.id)

        stacks = list(state)
        disk = stacks[self.origin][-1]
        stacks[self.origin] = stacks[self.origin][:-1]
        stacks[self.destination] = (*stacks[self.destination], disk)
        return (stacks[0], stacks[1], stacks[2])


R1 = Move("R1", Peg.H1, Peg.H2)
R2 = Move("R2", Peg.H1, Peg.H3)
R3 = Move("R3", Peg.H2, Peg.H1)
R4 = Move("R4", Peg.H2, Peg.H3)
R5 = Move("R5", Peg.H3, Peg.H1)
R6 = Move("R6", Peg.H3, Peg.H2)

In [ ]:
%%writefile src/core/rules/domain/catalog.py
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.rules.domain.base import TransitionRule
from core.rules.domain.exceptions import UnknownRuleError
from core.rules.moves import R1, R2, R3, R4, R5, R6

RULES: tuple[TransitionRule, ...] = (R1, R2, R3, R4, R5, R6)

RULE_BY_ID: Mapping[str, TransitionRule] = MappingProxyType(
    {rule.id: rule for rule in RULES}
)

INVERSE_RULE_ID: Mapping[str, str] = MappingProxyType(
    {"R1": "R3", "R3": "R1", "R2": "R5", "R5": "R2", "R4": "R6", "R6": "R4"}
)


def get_rule(rule_id: str) -> TransitionRule:
    try:
        return RULE_BY_ID[rule_id]
    except KeyError:
        raise UnknownRuleError(rule_id) from None


def inverse_of(rule: TransitionRule) -> TransitionRule:
    return RULE_BY_ID[INVERSE_RULE_ID[rule.id]]

In [ ]:
rules_table()

In [ ]:
show_rule_example("R4")

## Estratégia de controle: a ordem em que as regras são aplicadas

Na irrevogável a estratégia decide tudo: só a primeira regra da ordem é aplicada, então trocar a ordem troca o caminho inteiro e até o desfecho.

In [ ]:
%%writefile src/core/rules/strategies/ascending.py
from __future__ import annotations

from dataclasses import dataclass

from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES

_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}


@dataclass(frozen=True, slots=True)
class AscendingStrategy:
    name: str = "ascending"

    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:
        return tuple(sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id]))


ASCENDING = AscendingStrategy()

In [ ]:
%%writefile src/core/rules/strategies/descending.py
from __future__ import annotations

from dataclasses import dataclass

from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES

_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}


@dataclass(frozen=True, slots=True)
class DescendingStrategy:
    name: str = "descending"

    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:
        return tuple(
            sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id], reverse=True)
        )


DESCENDING = DescendingStrategy()

In [ ]:
strategies_table()

## As duas estratégias na mesma raiz

As regras válidas são as mesmas nas duas ordens, mas a irrevogável gera só o filho da primeira. Em P1 a solução começa por R4: `descending` escolhe R4 já na raiz; `ascending` escolhe R1.

In [ ]:
root_children("irrevocable")

## Sem poda

Sem poda, nada impede a irrevogável de aplicar uma regra e logo depois a inversa: a busca gira entre os mesmos dois estados até o limite de iterações e termina em LIMITE. Para caber no notebook, as execuções sem poda abaixo param em 30 iterações.

In [ ]:
growth("irrevocable", max_iterations=30)

### Sem poda · crescente

In [ ]:
sem_poda_crescente = case("irrevocable", "ascending", prune=False, max_iterations=30)

#### ABERTOS e FECHADOS, iteração a iteração (crescente, sem poda)

In [ ]:
lists_table(sem_poda_crescente)

### Sem poda · decrescente

In [ ]:
sem_poda_decrescente = case("irrevocable", "descending", prune=False, max_iterations=30)

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, sem poda)

In [ ]:
lists_table(sem_poda_decrescente)

## Com poda

A poda descarta a regra que leva a um estado que já está no caminho. Com ela a irrevogável nunca repete estado e sempre para, no máximo depois dos 36 estados. Parar não é chegar: ela ainda pode terminar em IMPASSE.

In [ ]:
levels_chart("irrevocable", max_iterations=30)

### Com poda · crescente

In [ ]:
com_poda_crescente = case("irrevocable", "ascending")

#### ABERTOS e FECHADOS, iteração a iteração (crescente, com poda)

In [ ]:
lists_table(com_poda_crescente)

### Com poda · decrescente

In [ ]:
com_poda_decrescente = case("irrevocable", "descending")

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, com poda)

In [ ]:
lists_table(com_poda_decrescente)

## Pseudocódigo

```text
busca_irrevogavel(problema, estratégia, poda)
    nó = raiz
    enquanto verdadeiro:
        se nó é o objetivo: SUCESSO
        coloca nó em FECHADOS
        regras = regras válidas em nó, na ordem da estratégia
        se poda: descarta as que levam a um estado de FECHADOS
        se não sobrou regra: IMPASSE
        nó = aplica a primeira regra em nó
```

**poda** vem de `--no-prune`: desligada, a linha da poda não roda. A **estratégia** só muda a ordem do *para cada*.

O código do projeto:

In [ ]:
%%writefile src/core/algorithms/irrevocable/algorithm.py
"""BUSCA IRREVOGÁVEL

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

       H1      H2      H3              H1      H2      H3
      ┌───┐   ┌───┐   ┌───┐           ┌───┐   ┌───┐   ┌───┐
      │ R │   │   │   │   │           │   │   │ V │   │   │
      │ V │   │ A │   │   │           │   │   │ R │   │ A │
      └───┘   └───┘   └───┘           └───┘   └───┘   └───┘

2. Regras válidas em S0

    R1 = H1 → H2  ✓          R4 = H2 → H3  ✓
    R2 = H1 → H3  ✓          R5 = H3 → H1  ✗  H3 vazia
    R3 = H2 → H1  ✓          R6 = H3 → H2  ✗  H3 vazia

3. Critério

    A estratégia ordena as regras válidas e a busca aplica a primeira.
    Não guarda as outras: cada passo é definitivo e não há retorno.
    Uma regra que leva a um estado já percorrido é descartada (poda).

4. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1

    Passo 1   S0 = ([V, R], [A], [])
              válidas em ordem: R4, R3, R2, R1
              aplica R4 (azul: H2 → H3)
              ↓
    Passo 2   S1 = ([V, R], [], [A])
              válidas em ordem: R6, R5, R1
              R6 volta a S0: poda.  Aplica R5 (azul: H3 → H1)
              ↓
    Passo 3   S2 = ([V, R, A], [], [])
              válidas em ordem: R2, R1
              R2 volta a S1: poda.  R1 volta a S0: poda.
              Nenhuma regra sobrou.
              ↓
              ✗ IMPASSE

5. Árvore de busca

    S0 ([V,R], [A], [])
    └── R4 → S1 ([V,R], [], [A])
        └── R5 → S2 ([V,R,A], [], [])   (impasse)

6. A mesma carta com a ordem crescente R1 → R2 → R3 → R4 → R5 → R6

    S0 ([V,R],[A],[]) → R1 → R2 → R3 → R3 → R5 → R1 → R1 → R2 → R3 → R3
       → R5 → R1 → R1 → R2 → Sf ([],[R,V],[A])

    ✓ SUCESSO com 14 movimentos.  O caminho ótimo tem 3.

7. Conclusão

    O caminho nunca repete estado, então a busca sempre para, no máximo
    depois dos 36 estados. Parar não é chegar: a ordem decrescente termina
    em impasse e a crescente só chega depois de 14 movimentos.
    É o único método em que a estratégia decide o desfecho.
"""

from __future__ import annotations

from typing import ClassVar

from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.search_tree.outcome import Outcome


class IrrevocableSearch(SearchAlgorithm):
    name: ClassVar[str] = "irrevocable"

    def _search(self, problem: Problem) -> Outcome:
        node = self._require_context().root
        self._observe_frontier(1)
        while True:
            if not self._next_iteration():
                return Outcome.CUTOFF
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._visit(node)
            candidates = self._applicable_rules(node)
            if not candidates:
                self._deadlock(node)
                return Outcome.DEADLOCK

            node = self._expand(node, candidates[0])

## Caminho solução

Com `descending` a busca termina em impasse na 3ª iteração e não há caminho. O caminho abaixo é o da ordem crescente: chega, mas com 14 movimentos, quando o ótimo tem 3.

In [ ]:
show_path(solve("irrevocable", "ascending"))

## Comparativo: os caminhos encontrados

Cada ponto é um movimento; o verde é o que chega ao objetivo.

In [ ]:
paths_plot("irrevocable")

## Comparativo: crescente contra decrescente

In [ ]:
strategy_table("irrevocable")

## Comparativo: busca irrevogável contra os outros, mesmas estratégias

In [ ]:
algorithms_table("irrevocable")

## Comparativo: quem explorou menos nós

In [ ]:
expanded_chart("irrevocable")

## Complexidade

| Algoritmo | Tempo | Memória | Solução | Por quê |
|---|---|---|---|---|
| **Irrevogável sem poda** | não termina | O(1) | pode não achar | pode girar num ciclo para sempre |
| **Irrevogável com poda** | O(b·m²) | O(m) | pode não achar | um nó por iteração, no máximo m; cada regra é comparada com o caminho |

In [ ]:
complexity_values("irrevocable", ('b', 'm', 'V'))

## Conclusão

- Em P1, `descending` trava na 3ª iteração: R4 e R5 levam a `H1[V,R,A]`, e de lá as duas regras aplicáveis voltam a estados do caminho. `ascending` chega, mas com 14 movimentos, quando o ótimo tem 3.
- Nos 36 objetivos, resolve 18 com `ascending` e 3 com `descending`. É o único algoritmo que deixa objetivos sem solução.
- É barata (um nó gerado por iteração), mas não garante solução nem qualidade: a estratégia decide tudo.

## Extras de busca irrevogável

### Fluxograma

In [ ]:
show_flowchart("irrevocable")

### Passo a passo do trace, com poda e ordem crescente

In [ ]:
show_trace(solve("irrevocable", "ascending"))

### Comparação em P1, placar lexicográfico

In [ ]:
p1_comparison("irrevocable")

### Comparação nos 36 objetivos

Cada métrica: média / mediana; movimentos e custo contam só os sucessos.

In [ ]:
goals_comparison("irrevocable")

# 2. Backtracking

A mesma descida da busca irrevogável, com as alternativas de cada nó guardadas numa pilha.

## O problema: carta P1, do estado inicial ao objetivo

Cada estado lista as hastes da base para o topo. As seis regras R1 a R6 movem o disco do topo de uma haste para outra.

In [ ]:
show_problem()

## A ideia

**Desça enquanto houver regra; sem regra, volte ao ancestral mais próximo que ainda tem alternativa.**

Backtracking: pilha, sai o último que entrou. Afunda num ramo.

O contrato da fronteira fica em `core/search_tree/frontier.py`. Cada algoritmo traz a sua implementação ao lado, na própria pasta: a pilha do backtracking, a fila da largura e a fila por custo da ordenada.

In [ ]:
%%writefile src/core/search_tree/frontier.py
"""FRONTEIRA — a lista de ABERTOS.

Os nós já gerados que ainda não foram processados. A busca tira um daqui,
olha em volta e devolve os filhos. Fronteira vazia, busca encerrada.

Quem sai primeiro define o algoritmo: pilha afunda, fila varre por nível,
fila de prioridade sai pelo menor custo.

Aqui fica só o contrato. Cada implementação mora ao lado do algoritmo que a
usa: core/algorithms/<algoritmo>/frontier.py.
"""

from __future__ import annotations

from typing import Protocol

from core.search_tree.node import Node


class EmptyFrontierError(Exception):
    """Tentou tirar um nó de uma fronteira vazia."""

    def __init__(self) -> None:
        super().__init__("cannot pop from an empty frontier")


class Frontier(Protocol):
    """Guardar um nó, tirar um nó, contar quantos faltam. Só isso.

    Sem índice, sem busca por estado, sem percorrer o conteúdo: o único
    acesso é pela ponta. O contrato não diz QUAL nó o pop devolve — é a
    única coisa que muda entre as implementações, e é o que define a busca.
    """

    def push(self, node: Node) -> None:
        """Guarda um nó. Ele sempre entra pelo fim."""
        ...

    def pop(self) -> Node:
        """Tira um nó e devolve. QUAL nó é o que muda entre as fronteiras."""
        ...

    def __len__(self) -> int:
        """Quantos nós ainda esperam. Zero encerra a busca."""
        ...

In [ ]:
%%writefile src/core/algorithms/backtracking/frontier.py
"""FRONTEIRA DO BACKTRACKING — pilha.

O contrato (push, pop, len) está em core/search_tree/frontier.py.
"""

from __future__ import annotations

from core.search_tree.frontier import EmptyFrontierError
from core.search_tree.node import Node


class StackFrontier:
    """PILHA (LIFO): sai o último que entrou. Usada pelo backtracking.

    ```
    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S3
    ```

    Entra e sai pelo mesmo lado, então o filho recém-gerado é sempre o
    próximo a ser olhado: a busca afunda num ramo até o fim.
    """

    __slots__ = ("_nodes",)

    def __init__(self) -> None:
        self._nodes: list[Node] = []

    def push(self, node: Node) -> None:
        """Empilha o nó no topo."""
        self._nodes.append(node)

    def pop(self) -> Node:
        """Tira e devolve o nó do topo: o ÚLTIMO que entrou."""
        if not self._nodes:
            raise EmptyFrontierError
        return self._nodes.pop()

    def __len__(self) -> int:
        """Quantos nós estão empilhados."""
        return len(self._nodes)

In [ ]:
frontier_demo("backtracking")

## O laço e as listas

1. Olha o topo da pilha. É o objetivo? **SUCESSO**.
2. Primeira vez nele: guarda as regras válidas, na ordem da estratégia, sem as que voltam a um estado do caminho (poda). Lista vazia é impasse.
3. Ainda tem regra guardada: aplica a próxima e empilha o filho.
4. Não tem: desempilha o nó (retrocesso) e o pai volta ao topo. Volta ao passo 1.

**PILHA**: o caminho atual, da raiz ao topo. Faz o papel de ABERTOS.

**RETROCEDIDOS**: os nós que saíram da pilha sem levar ao objetivo. Fazem o papel de FECHADOS.

## As regras: seis regras, uma para cada par origem e destino

Vale se a origem tem disco e o destino tem espaço. Toda regra tem inversa: aplicar as duas em seguida volta ao mesmo estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

In [ ]:
rules_table()

In [ ]:
show_rule_example("R4")

## Estratégia de controle: a ordem em que as regras são aplicadas

No backtracking a estratégia decide qual ramo é descido primeiro. A solução é a primeira encontrada, então trocar a ordem troca o caminho e o número de retrocessos.

In [ ]:
strategies_table()

## As duas estratégias na mesma raiz

A raiz guarda as mesmas regras nas duas ordens; muda só qual filho é empilhado primeiro. Em P1 a solução começa por R4, que `descending` tenta primeiro.

In [ ]:
root_children("backtracking")

## Sem poda

Sem poda, a descida aceita voltar a estados do caminho: a busca entra num ciclo entre uma regra e a inversa, nunca chega a um beco sem saída e por isso nunca retrocede. Gira até o limite de iterações e termina em LIMITE. Para caber no notebook, as execuções sem poda abaixo param em 30 iterações.

In [ ]:
growth("backtracking", max_iterations=30)

### Sem poda · crescente

In [ ]:
sem_poda_crescente = case("backtracking", "ascending", prune=False, max_iterations=30)

#### ABERTOS e FECHADOS, iteração a iteração (crescente, sem poda)

In [ ]:
lists_table(sem_poda_crescente)

### Sem poda · decrescente

In [ ]:
sem_poda_decrescente = case("backtracking", "descending", prune=False, max_iterations=30)

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, sem poda)

In [ ]:
lists_table(sem_poda_decrescente)

## Com poda

A poda descarta a regra que leva a um estado que já está no caminho atual. É uma poda local: o mesmo estado pode aparecer em ramos diferentes. Com ela não há ciclo e, como todo estado alcança todo estado, a solução sempre aparece.

In [ ]:
levels_chart("backtracking", max_iterations=30)

### Com poda · crescente

In [ ]:
com_poda_crescente = case("backtracking", "ascending")

#### ABERTOS e FECHADOS, iteração a iteração (crescente, com poda)

In [ ]:
lists_table(com_poda_crescente)

### Com poda · decrescente

In [ ]:
com_poda_decrescente = case("backtracking", "descending")

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, com poda)

In [ ]:
lists_table(com_poda_decrescente)

## Pseudocódigo

```text
backtracking(problema, estratégia, poda)
    PILHA = [raiz]
    enquanto PILHA não estiver vazia:
        nó = topo da PILHA
        se nó é o objetivo: SUCESSO
        se é a primeira vez em nó:
            guarda as regras válidas, na ordem da estratégia
            se poda: descarta as que levam a um estado do caminho
        se nó ainda tem regra guardada:
            tira a próxima, aplica e empilha o filho
        senão:
            desempilha nó (retrocesso)
    FRACASSO
```

**poda** vem de `--no-prune`: desligada, a linha da poda não roda. A **estratégia** só muda a ordem do *para cada*.

O código do projeto:

In [ ]:
%%writefile src/core/algorithms/backtracking/algorithm.py
"""BUSCA COM BACKTRACKING

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Critério

    A mesma descida da busca irrevogável, mas as regras válidas de cada
    estado ficam guardadas numa pilha. Quando um estado não tem mais regra,
    ele sai da pilha (retrocesso) e o estado anterior tenta a próxima regra
    que tinha guardado. Uma regra que leva a um estado já no caminho atual
    é descartada (poda).

3. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1

    Passo 1   S0 = ([V, R], [A], [])
              guarda [R4, R3, R2, R1], aplica R4
              ↓
    Passo 2   S1 = ([V, R], [], [A])
              R6 volta a S0: poda.  Guarda [R5, R1], aplica R5
              ↓
    Passo 3   S2 = ([V, R, A], [], [])
              R2 volta a S1 e R1 volta a S0: poda.  Nada guardado.
              ✗ impasse → retrocesso: S2 sai da pilha
              ↑
    Passo 4   S1 = ([V, R], [], [A])   ainda tem [R1], aplica R1
              ↓
    Passo 5   S3 = ([V], [R], [A])
              R3 volta a S1: poda.  Guarda [R6, R5, R1], aplica R6
              ↓
              ...  continua descendo; a cada beco sem saída, um retrocesso

    Passo 51  S36 = ([], [R, V], [A])
              ✓ SUCESSO

4. Árvore de busca (início)

    S0 ([V,R], [A], [])
    └── R4 → S1 ([V,R], [], [A])
        ├── R5 → S2 ([V,R,A], [], [])   (impasse, retrocesso)
        └── R1 → S3 ([V], [R], [A])
            └── R6 → S4 ([V], [R,A], [])
                └── R3 → S5 ([V,A], [R], [])
                    └── R4 → S6 ([V,A], [], [R])
                        ├── R5 → S7 ([V,A,R], [], [])   (impasse, retrocesso)
                        └── R1 → S8 ([V], [A], [R])
                            └── ...  até S36 = Sf

5. Caminho encontrado

    R4 R1 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2

    Número de movimentos: 22       Iterações: 51
    Retrocessos: 14                Impasses: 8

    Com a ordem crescente: 14 movimentos em 15 iterações, sem retrocesso.
    O caminho ótimo tem 3.

6. Conclusão

    O impasse da busca irrevogável vira um desvio: nenhum ramo é abandonado
    sem ser explorado e, como todo estado alcança todo estado, a solução
    sempre aparece. Só não é a melhor. A busca para no primeiro objetivo
    que encontra, tenha o caminho o comprimento que tiver.
"""

from __future__ import annotations

from collections import deque
from typing import ClassVar

from core.algorithms.backtracking.frontier import StackFrontier
from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.rules.domain.base import TransitionRule
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome


class BacktrackingSearch(SearchAlgorithm):
    name: ClassVar[str] = "backtracking"

    def _search(self, problem: Problem) -> Outcome:
        """Desce enquanto houver regra; sem regra, retrocede.

        Cada iteração olha a ponta do caminho e faz uma coisa OU a outra:
        desce, e o nó continua no caminho com um filho por cima; ou retrocede,
        e o nó sai do caminho, deixando o pai como ponta.

        Sempre encontra solução, se existir — mas é a primeira que aparecer,
        não a mais curta.
        """
        context = self._require_context()

        path = StackFrontier()
        path.push(context.root)
        self._observe_frontier(len(path))

        # Para cada nó do caminho, as regras que ele ainda não tentou. Sem
        # isso não há retrocesso: o nó não saberia por onde continuar.
        untried: dict[int, deque[TransitionRule]] = {}

        while len(path):
            if not self._next_iteration():
                return Outcome.CUTOFF

            # Topo da pilha = ponta do caminho. Ele só volta para a pilha se
            # ainda tiver regra -- ver o if lá embaixo.
            node = path.pop()

            if problem.is_goal(node.state):
                return self._succeed(node)

            remaining = self._remaining_rules(node, untried)
            if remaining:
                # DESCE: o nó continua no caminho e o filho entra por cima.
                path.push(node)
                path.push(self._expand(node, remaining.popleft()))
                self._observe_frontier(len(path))
            else:
                # RETROCEDE: sem regras, o nó não volta e sai do caminho. Na
                # próxima iteração o topo é o pai, que tenta a regra seguinte.
                self._backtrack(node)

        return self._exhausted()

    def _remaining_rules(
        self, node: Node, untried: dict[int, deque[TransitionRule]]
    ) -> deque[TransitionRule]:
        """Na primeira visita monta a lista do nó; nas voltas seguintes apenas
        devolve o que sobrou dela. _applicable_rules já poda a regra que
        repetiria um estado do caminho.

        As seis regras que entram nessa lista:

        ```
        regra | de | para | inversa
        ------+----+------+--------
        R1    | H1 | H2   | R3
        R2    | H1 | H3   | R5
        R3    | H2 | H1   | R1
        R4    | H2 | H3   | R6
        R5    | H3 | H1   | R2
        R6    | H3 | H2   | R4
        ```

        Toda regra tem inversa exata, e é daí que vem a necessidade da poda:
        sem ela, R1 seguido de R3 devolveria a busca ao mesmo estado, sempre.
        """
        if node.order not in untried:
            self._visit(node)
            candidates = self._applicable_rules(node)
            untried[node.order] = deque(candidates)
            if not candidates:
                self._deadlock(node)
        return untried[node.order]

## Caminho solução

A ordem decrescente, a do docstring do módulo: 22 movimentos para um ótimo de 3, depois de 14 retrocessos.

In [ ]:
show_path(solve("backtracking", "descending"))

## Comparativo: os caminhos encontrados

Cada ponto é um movimento; o verde é o que chega ao objetivo.

In [ ]:
paths_plot("backtracking")

## Comparativo: crescente contra decrescente

In [ ]:
strategy_table("backtracking")

## Comparativo: backtracking contra os outros, mesmas estratégias

In [ ]:
algorithms_table("backtracking")

## Comparativo: quem explorou menos nós

In [ ]:
expanded_chart("backtracking")

## Complexidade

| Algoritmo | Tempo | Memória | Solução | Por quê |
|---|---|---|---|---|
| **Backtracking sem poda** | não termina | O(L) | pode não achar | o ciclo cresce a pilha até o limite L de iterações |
| **Backtracking com poda** | O(bᵐ) | O(m) | a primeira achada | só guarda o caminho atual |

In [ ]:
complexity_values("backtracking", ('b', 'm'))

## Conclusão

- Em P1 as duas estratégias chegam ao objetivo. Com `descending` são 51 iterações, 14 retrocessos e 22 movimentos para um ótimo de 3; com `ascending` não há retrocesso e o caminho é o mesmo da irrevogável, com 14 movimentos.
- Nos 36 objetivos resolve todos, mas com média de 12,81 a 16,5 movimentos, contra 4,14 do ótimo. Com `descending`, a média de iterações é 82,92 e a mediana 21,5: um objetivo passa de 2 mil iterações.
- Resolve o problema da irrevogável (o impasse vira desvio), mas para no primeiro objetivo que encontra, tenha o caminho o comprimento que tiver.

## Extras de backtracking

### Fluxograma

In [ ]:
show_flowchart("backtracking")

### Passo a passo do trace, com poda e ordem crescente

In [ ]:
show_trace(solve("backtracking", "ascending"))

### Comparação em P1, placar lexicográfico

In [ ]:
p1_comparison("backtracking")

### Comparação nos 36 objetivos

Cada métrica: média / mediana; movimentos e custo contam só os sucessos.

In [ ]:
goals_comparison("backtracking")

# 3. Busca em largura

Torre de Londres: três hastes, três discos. A árvore é varrida nível por nível.

## O problema: carta P1, do estado inicial ao objetivo

Cada estado lista as hastes da base para o topo. As seis regras R1 a R6 movem o disco do topo de uma haste para outra.

In [ ]:
show_problem()

## A ideia

**Analise todos os nós de um nível antes de descer para o próximo.**

Backtracking: pilha, sai o último que entrou. Afunda num ramo.

Largura: fila, sai o primeiro que entrou. Varre por níveis.

In [ ]:
%%writefile src/core/algorithms/breadth_first/frontier.py
"""FRONTEIRA DA BUSCA EM LARGURA — fila.

O contrato (push, pop, len) está em core/search_tree/frontier.py.
"""

from __future__ import annotations

from collections import deque

from core.search_tree.frontier import EmptyFrontierError
from core.search_tree.node import Node


class QueueFrontier:
    """FILA (FIFO): sai o primeiro que entrou. Usada pela busca em largura.

    ```
    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S1
    ```

    Entra por um lado, sai pelo outro, então os filhos esperam atrás de todo
    o nível atual: a busca varre nível por nível e acha o caminho mais curto.

    Usa deque, não list, porque popleft é O(1) e list.pop(0) é O(n).
    """

    __slots__ = ("_nodes",)

    def __init__(self) -> None:
        self._nodes: deque[Node] = deque()

    def push(self, node: Node) -> None:
        """Enfileira o nó no fim da fila."""
        self._nodes.append(node)

    def pop(self) -> Node:
        """Tira e devolve o nó do início: o PRIMEIRO que entrou.

        É a única linha que difere da StackFrontier, e é ela que troca
        profundidade por largura.
        """
        if not self._nodes:
            raise EmptyFrontierError
        return self._nodes.popleft()

    def __len__(self) -> int:
        """Quantos nós estão na fila."""
        return len(self._nodes)

In [ ]:
frontier_demo("breadth_first")

## O laço e as listas

1. Tira o primeiro nó de ABERTOS.
2. É o objetivo? **SUCESSO**.
3. Não é: coloca o nó em FECHADOS.
4. Aplica todas as regras válidas, na ordem da estratégia.
5. Os filhos entram no fim de ABERTOS. Volta ao passo 1.

**ABERTOS**: fila dos nós gerados que ainda não foram analisados.

**FECHADOS**: nós já expandidos. Com poda, também servem para não gerar estados repetidos.

## As regras: seis regras, uma para cada par origem e destino

Vale se a origem tem disco e o destino tem espaço. Toda regra tem inversa: aplicar as duas em seguida volta ao mesmo estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

In [ ]:
rules_table()

In [ ]:
show_rule_example("R4")

## Estratégia de controle: a ordem em que as regras são aplicadas

Na largura a estratégia não muda quais nós existem em cada nível. Muda só a ordem deles dentro do nível, e portanto quantos nós são analisados antes do objetivo.

In [ ]:
strategies_table()

## As duas estratégias na mesma raiz

Os filhos são os mesmos, só a ordem na fila muda. Em P1 a solução começa por R4: `descending` já põe esse ramo na frente.

In [ ]:
root_children("breadth_first")

## Sem poda

Sem poda, a árvore cresce sem parar: a regra inversa sempre gera de volta o avô. As árvores sem poda abaixo vão até o objetivo, no nível 3.

In [ ]:
growth("breadth_first")

### Sem poda · crescente

In [ ]:
sem_poda_crescente = case("breadth_first", "ascending", prune=False)

#### ABERTOS e FECHADOS, iteração a iteração (crescente, sem poda)

In [ ]:
lists_table(sem_poda_crescente)

### Sem poda · decrescente

In [ ]:
sem_poda_decrescente = case("breadth_first", "descending", prune=False)

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, sem poda)

In [ ]:
lists_table(sem_poda_decrescente)

## Com poda

Uma regra é descartada se o estado que ela gera já está em ABERTOS ou em FECHADOS.

**Poda global**: vale para a árvore toda, não só para o caminho, como no backtracking. Cada estado entra uma vez, então a árvore cabe inteira, sem limitar a profundidade.

In [ ]:
levels_chart("breadth_first")

### Com poda · crescente

In [ ]:
com_poda_crescente = case("breadth_first", "ascending")

#### ABERTOS e FECHADOS, iteração a iteração (crescente, com poda)

In [ ]:
lists_table(com_poda_crescente)

### Com poda · decrescente

In [ ]:
com_poda_decrescente = case("breadth_first", "descending")

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, com poda)

In [ ]:
lists_table(com_poda_decrescente)

## Pseudocódigo

```text
busca_em_largura(problema, estratégia, poda)
    ABERTOS = fila com a raiz
    FECHADOS = vazio
    enquanto ABERTOS não estiver vazia:
        nó = tira o primeiro de ABERTOS
        se nó é o objetivo: SUCESSO
        coloca nó em FECHADOS
        para cada regra válida, na ordem da estratégia:
            filho = aplica a regra em nó
            se poda e filho está em ABERTOS ou FECHADOS:
                descarta a regra
            senão: põe filho no fim de ABERTOS
    FRACASSO
```

**poda** vem de `--no-prune`: desligada, a linha da poda não roda. A **estratégia** só muda a ordem do *para cada*.

O código do projeto:

In [ ]:
%%writefile src/core/algorithms/breadth_first/algorithm.py
"""BUSCA EM LARGURA

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Critério

    As regras são analisadas na ordem crescente R1 → R2 → R3 → R4 → R5 → R6.
    Em vez de escolher uma, a busca aplica todas as válidas e coloca os
    filhos no fim de ABERTOS. O estado visitado vai para FECHADOS. Uma
    regra cujo resultado já está em ABERTOS ou FECHADOS é descartada (poda).

3. Execução

    Nível 0
        ABERTOS = [S0]               FECHADOS = []

    Expansão de S0 = ([V, R], [A], [])
        R1 → S1 = ([V], [A, R], [])
        R2 → S2 = ([V], [A], [R])
        R3 → S3 = ([V, R, A], [], [])
        R4 → S4 = ([V, R], [], [A])
        R5, R6: inválidas, H3 vazia
        ABERTOS = [S1, S2, S3, S4]   FECHADOS = [S0]

    Nível 1

    Expansão de S1 = ([V], [A, R], [])
        R1: inválida, H2 cheia
        R2 → S5 = ([], [A, R], [V])
        R3 → S0: poda
        R4 → S2: poda
        ABERTOS = [S2, S3, S4, S5]   FECHADOS = [S0, S1]

    Expansão de S2 = ([V], [A], [R])
        R1 → S6 = ([], [A, V], [R])
        R3 → S7 = ([V, A], [], [R])
        R5 → S0, R6 → S1: poda
        ABERTOS = [S3, S4, S5, S6, S7]

    Expansão de S3 = ([V, R, A], [], [])
        R1 → S0, R2 → S4: poda.  Nada novo.

    Expansão de S4 = ([V, R], [], [A])
        R1 → S8 = ([V], [R], [A])
        R5 → S3, R6 → S0: poda
        ABERTOS = [S5, S6, S7, S8]   FECHADOS = [S0, S1, S2, S3, S4]

    Nível 2
        S5 gera S9.  S6 gera S10.  S7 gera S11 e S12.
        S8 = ([V], [R], [A]) gera:
            R1 → S13 = ([], [R, V], [A])   ← é Sf, entra na fila
            R6 → S14 = ([V], [R, A], [])
        ABERTOS = [S9, S10, S11, S12, S13, S14]

    Nível 3
        S9 gera S15 e S16.  S10, S11 e S12 só geram repetidos.
        Chega a vez de S13 = ([], [R, V], [A])
        ✓ SUCESSO

4. Árvore de busca

    S0 ([V,R], [A], [])
    ├── R1 → S1 ([V], [A,R], [])
    │   └── R2 → S5 ([], [A,R], [V])
    │       └── R3 → S9 ([R], [A], [V])
    │           ├── R3 → S15
    │           └── R5 → S16
    ├── R2 → S2 ([V], [A], [R])
    │   ├── R1 → S6 ([], [A,V], [R])
    │   │   └── R5 → S10
    │   └── R3 → S7 ([V,A], [], [R])
    │       ├── R5 → S11
    │       └── R6 → S12
    ├── R3 → S3 ([V,R,A], [], [])
    └── R4 → S4 ([V,R], [], [A])
        └── R1 → S8 ([V], [R], [A])
            ├── R1 → S13 ([], [R,V], [A])   (objetivo)
            └── R6 → S14 ([V], [R,A], [])

5. Caminho solução

    S0 ([V,R], [A], [])
     │ R4: azul H2 → H3
     ↓
    S4 ([V,R], [], [A])
     │ R1: vermelho H1 → H2
     ↓
    S8 ([V], [R], [A])
     │ R1: verde H1 → H2
     ↓
    Sf ([], [R,V], [A])

    Número de movimentos: 3      Iterações: 14      Gerados: 17

6. Conclusão

    Nada de profundidade d+1 é visitado antes de esgotar a profundidade d,
    então o primeiro caminho até um estado é o mais curto até ele. O objetivo
    encontrado é o ótimo em movimentos, e por isso este método serve de
    referência de comprimento para os outros. A estratégia só muda a ordem
    dentro de um nível.

    A poda aqui é global: um estado descoberto por qualquer ramo nunca é
    gerado de novo. A irrevogável e o backtracking só evitam repetir dentro
    do próprio caminho.
"""

from __future__ import annotations

from typing import ClassVar

from config import settings
from core.algorithms.breadth_first.frontier import QueueFrontier
from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.tree import SearchTree


class BreadthFirstSearch(SearchAlgorithm):
    name: ClassVar[str] = "breadth_first"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
        prune: bool = True,
    ) -> None:
        super().__init__(tree, strategy, max_iterations=max_iterations, prune=prune)
        self._closed: set[State] = set()
        self._open_states: set[State] = set()

    def _search(self, problem: Problem) -> Outcome:
        context = self._require_context()
        frontier = QueueFrontier()
        frontier.push(context.root)

        self._closed = set()
        self._open_states = {context.root.state}
        self._observe_frontier(len(frontier))

        while len(frontier):
            if not self._next_iteration():
                return Outcome.CUTOFF

            node = frontier.pop()
            self._open_states.discard(node.state)
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._closed.add(node.state)
            self._visit(node)
            for rule in self._applicable_rules(node):
                child = self._expand(node, rule)
                self._open_states.add(child.state)
                frontier.push(child)
            self._observe_frontier(len(frontier))

        return self._exhausted()

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        return successor in self._closed or successor in self._open_states

## Caminho solução

3 movimentos. As duas estratégias, com e sem poda, chegam a este mesmo caminho, e ele é o menor possível.

In [ ]:
show_path(solve("breadth_first", "ascending"))

## Comparativo: os caminhos encontrados

Cada ponto é um movimento; o verde é o que chega ao objetivo.

In [ ]:
paths_plot("breadth_first")

## Comparativo: crescente contra decrescente

In [ ]:
strategy_table("breadth_first")

## Comparativo: busca em largura contra os outros, mesmas estratégias

In [ ]:
algorithms_table("breadth_first")

## Comparativo: quem explorou menos nós

In [ ]:
expanded_chart("breadth_first")

## Complexidade

| Algoritmo | Tempo | Memória | Solução | Por quê |
|---|---|---|---|---|
| **Largura sem poda** | O(bᵈ) | O(bᵈ) | ótima | a fila guarda o nível inteiro |
| **Largura com poda** | O(V + E) | O(V) | ótima | cada estado entra uma vez |
| **Backtracking** | O(bᵐ) | O(m) | a primeira achada | só guarda o caminho atual |

In [ ]:
complexity_values("breadth_first", ('b', 'd', 'm', 'V', 'E'))

## Conclusão

- Em P1 as duas estratégias acham R4, R1, R1: 3 movimentos, o ótimo. A estratégia só muda a ordem dentro de um nível e, com ela, o número de iterações.
- Nos 36 objetivos acha sempre o mínimo de movimentos (média 4,14). No custo, passa do menor em 9 das 72 execuções, porque não olha o custo das regras.
- É a referência de comprimento para os outros algoritmos. Em custo, a referência é a busca ordenada.

## Extras de busca em largura

### Fluxograma

In [ ]:
show_flowchart("breadth_first")

### Passo a passo do trace, com poda e ordem crescente

In [ ]:
show_trace(solve("breadth_first", "ascending"))

### Comparação em P1, placar lexicográfico

In [ ]:
p1_comparison("breadth_first")

### Comparação nos 36 objetivos

Cada métrica: média / mediana; movimentos e custo contam só os sucessos.

In [ ]:
goals_comparison("breadth_first")

# 4. Busca ordenada

Cada regra tem um custo, e ABERTOS vira uma fila ordenada pelo custo acumulado desde a raiz.

## O problema: carta P1, do estado inicial ao objetivo

Cada estado lista as hastes da base para o topo. As seis regras R1 a R6 movem o disco do topo de uma haste para outra.

In [ ]:
show_problem()

## A ideia

**Expanda sempre o nó aberto de menor custo acumulado.**

Largura: fila, sai o primeiro que entrou.

Ordenada: fila por custo, sai o mais barato; no empate, o gerado primeiro. Com o mesmo custo em todas as regras, seria a busca em largura.

In [ ]:
%%writefile src/core/algorithms/ordered/frontier.py
"""FRONTEIRA DA BUSCA ORDENADA — fila de prioridade por custo.

O contrato (push, pop, len) está em core/search_tree/frontier.py.
"""

from __future__ import annotations

import heapq

from core.search_tree.frontier import EmptyFrontierError
from core.search_tree.node import Node


class PriorityFrontier:
    """FILA ORDENADA POR CUSTO: sai o de menor custo acumulado. Usada pela
    busca ordenada.

    ```
    push S1(5), S2(6), S3(6), S4(1)  ->  pop devolve S4, S1, S2, S3
    ```

    Empate de custo sai o gerado primeiro (menor `order`). Irmãos de mesmo
    custo nascem na ordem da estratégia, então ela é o desempate entre eles.

    Heap de (custo, ordem, nó): a ordem é única, então o nó nunca é comparado.
    """

    __slots__ = ("_heap", "_removed")

    def __init__(self) -> None:
        self._heap: list[tuple[int, int, Node]] = []
        self._removed: set[int] = set()

    def push(self, node: Node) -> None:
        """Insere o nó na posição do seu custo."""
        heapq.heappush(self._heap, (node.cost, node.order, node))

    def pop(self) -> Node:
        """Tira e devolve o nó de MENOR custo; no empate, o mais antigo."""
        while self._heap:
            _, order, node = heapq.heappop(self._heap)
            if order in self._removed:
                self._removed.discard(order)
                continue
            return node
        raise EmptyFrontierError

    def remove(self, node: Node) -> None:
        """Tira um nó que ainda está na fila, sem devolvê-lo.

        A remoção é preguiçosa: o nó só é descartado quando chegaria a vez
        dele no pop. O nó precisa estar na fila.
        """
        self._removed.add(node.order)

    def __len__(self) -> int:
        """Quantos nós ainda esperam, sem contar os removidos."""
        return len(self._heap) - len(self._removed)

In [ ]:
frontier_demo("ordered")

## O laço e as listas

1. Tira de ABERTOS o nó de menor custo; no empate, o gerado primeiro.
2. É o objetivo? **SUCESSO**.
3. Não é: coloca o nó em FECHADOS.
4. Aplica todas as regras válidas, na ordem da estratégia. O custo do filho é o do pai mais o da regra.
5. Os filhos entram em ABERTOS na posição do seu custo. Volta ao passo 1.

**ABERTOS**: fila dos nós gerados que ainda não foram analisados, ordenada por custo.

**FECHADOS**: nós já expandidos. Com poda, o vetor de menor custo cobre ABERTOS e FECHADOS de uma vez.

## As regras: seis regras, uma para cada par origem e destino

Vale se a origem tem disco e o destino tem espaço. Toda regra tem inversa: aplicar as duas em seguida volta ao mesmo estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

In [ ]:
rules_table()

In [ ]:
show_rule_example("R4")

## A regra de custo

**custo = 10 + peso do disco × distância**

| Parte | Valor | Por quê |
|---|---|---|
| custo por jogada | 10 | toda jogada conta, e conta mais que o esforço |
| distância | 1 entre hastes vizinhas; 2 de H1 para H3 | o disco anda mais quando pula a haste do meio |
| peso do disco | verde 1, vermelho 2, azul 3 | carregar um disco mais pesado custa mais |

O custo de um nó é a soma das regras do caminho desde a raiz. A tabela dá o custo de cada regra conforme o disco que está no topo da origem.

In [ ]:
cost_table()

In [ ]:
explain_cost("R4")

### 1. Dez por jogada: menos movimentos vem primeiro

O esforço de uma jogada (peso × distância) vai de 1 a 6; o 10 fixo pesa mais. Em todos os 1.260 pares início → objetivo, o caminho mais barato é sempre um dos de menos movimentos: a ordenada não troca movimentos por esforço. É o que a Torre de Londres mede, o número de movimentos.

**Por que é boa:** sem o 10, só com peso × distância, a ordenada passa do mínimo de movimentos em 3,7% das execuções.

In [ ]:
effort_range()

In [ ]:
cost_study().loc[["peso × distância, sem o 10", "10 + peso × distância (escolhido)"], ["Mínimo de movimentos"]].round(1)

### 2. Distância: a viagem longa fica com o disco leve

Hastes vizinhas (H1 ↔ H2, H2 ↔ H3) valem 1; H1 ↔ H3, que pula a do meio, vale 2. O esforço de uma jogada é **peso × distância**, como o trabalho na física (força × deslocamento): carregar o azul por duas hastes custa o dobro de carregá-lo por uma.

**O que ela faz:** o peso diz *qual* disco se move; a distância diz *quanto* ele anda. Sem ela, dois caminhos que movem os mesmos discos o mesmo número de vezes empatam, e quem decide é a ordem das regras. Com ela, vence o caminho em que a viagem longa fica com o disco mais leve.

Exemplo, objetivo RV / A / –: os dois caminhos têm 4 movimentos e fazem duas viagens longas. Só com o peso, empatam em 46; com a distância, o que leva o verde nas viagens longas custa 48 e o que leva o vermelho custa 50.

In [ ]:
show_hop(("R1",), "R2", "R1 R2 R3 R5: o verde faz a viagem longa")
show_hop((), "R2", "R2 R1 R5 R3: o vermelho faz a viagem longa")

In [ ]:
distance_example()

In [ ]:
distance_tiebreak()

**Só com o peso:** empate em 46. A crescente acha o primeiro, a decrescente acha o segundo.

**Com a distância:** as duas estratégias acham o primeiro (48), o do disco leve.

### 3. Peso do disco: o azul pesa mais

Pesos: verde 1 · vermelho 2 · azul 3. Objetivo VA / R / –, ordem decrescente: os dois fazem 3 movimentos.

In [ ]:
show_hop(("R4", "R1"), "R5", "Largura: R4 R1 R5, o azul anda duas hastes")
show_hop(("R2",), "R3", "Ordenada: R2 R3 R6, o azul anda uma haste")

In [ ]:
compare_with_breadth("G11", "descending")

Total: 13 + 12 + 16 = **41** na largura, 14 + 13 + 12 = **39** na ordenada. Em média, nos 1.260 pares, a ordenada sai **0,4%** mais barata que a largura.

In [ ]:
cost_study().loc[["10 + peso × distância (escolhido)"], ["Economia média de custo"]].round(1)

### Ganho real: menos de 0,5% de custo

Economia média: quanto o caminho da ordenada custa a menos que o da largura, em relação à largura, nos 1.260 pares início → objetivo e nas duas estratégias.

In [ ]:
cost_study().loc[["distância (1 ou 2)", "10 + peso do disco", "10 + peso × distância (escolhido)"], ["Economia média de custo", "Maior economia", "Nós gerados vs largura", "Trocas por execução", "Movimentos a mais"]].round(2)

## Estratégia de controle: a ordem em que as regras são aplicadas

Na ordenada a estratégia só desempata irmãos de mesmo custo: decide quem fica na frente em ABERTOS, nunca o custo da solução. Com o custo 10 + peso × distância os empates são raros, e em P1 as duas ordens fazem as mesmas 11 iterações.

In [ ]:
strategies_table()

## As duas estratégias na mesma raiz

Na raiz, R1 leva o vermelho por uma haste (12), R3 e R4 levam o azul por uma haste (13) e R2 leva o vermelho por duas (14). A fila fica ordenada pelo custo nas duas ordens; a estratégia só decide entre R3 e R4, que empatam em 13.

In [ ]:
root_children("ordered")

## Sem poda

Sem poda, estados repetidos entram de novo em ABERTOS, cada um com o custo do caminho que o gerou. A busca ainda acha o menor custo, porque só para quando o objetivo sai da fila, mas gera muito mais nós.

In [ ]:
growth("ordered")

### Sem poda · crescente

In [ ]:
sem_poda_crescente = case("ordered", "ascending", prune=False)

#### ABERTOS e FECHADOS, iteração a iteração (crescente, sem poda)

In [ ]:
lists_table(sem_poda_crescente)

### Sem poda · decrescente

In [ ]:
sem_poda_decrescente = case("ordered", "descending", prune=False)

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, sem poda)

In [ ]:
lists_table(sem_poda_decrescente)

## Com poda

Técnica de poda da busca ordenada, com o **vetor de menor custo**: toda vez que um nó é gerado,

- se é a primeira vez que o estado aparece, guarda o estado com o custo atual;
- se o estado aparece repetido com custo maior ou igual ao do vetor, poda o nó;
- se aparece com custo menor, atualiza o vetor, tira o nó antigo de ABERTOS e da árvore e inclui o novo.

Com o custo 10 + peso × distância, o terceiro caso aparece em P1: na iteração 9, VA/R/– reaparece por 39 e tira de ABERTOS o nó antigo, de custo 41.

In [ ]:
levels_chart("ordered")

### Com poda · crescente

In [ ]:
com_poda_crescente = case("ordered", "ascending")

#### ABERTOS e FECHADOS, iteração a iteração (crescente, com poda)

In [ ]:
lists_table(com_poda_crescente)

### Com poda · decrescente

In [ ]:
com_poda_decrescente = case("ordered", "descending")

#### ABERTOS e FECHADOS, iteração a iteração (decrescente, com poda)

In [ ]:
lists_table(com_poda_decrescente)

## Pseudocódigo

```text
busca_ordenada(problema, estratégia, poda)
    ABERTOS = fila por custo com a raiz (custo 0)
    MENOR_CUSTO = {raiz: 0}
    enquanto ABERTOS não estiver vazia:
        nó = tira o de menor custo de ABERTOS (no empate, o mais antigo)
        se nó é o objetivo: SUCESSO
        coloca nó em FECHADOS
        para cada regra válida, na ordem da estratégia:
            custo = custo(nó) + custo(regra)
            se poda e custo >= MENOR_CUSTO[estado do filho]:
                descarta a regra
            senão:
                se o estado do filho está em ABERTOS: tira o nó antigo
                MENOR_CUSTO[estado do filho] = custo
                põe o filho em ABERTOS, na posição do custo
    FRACASSO
```

**poda** vem de `--no-prune`: desligada, a linha da poda não roda. A **estratégia** só muda a ordem do *para cada*.

O código do projeto:

In [ ]:
%%writefile src/core/algorithms/ordered/algorithm.py
"""BUSCA ORDENADA

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Custos

    custo da regra = 10 + peso do disco movido x distância entre as hastes

        10          por jogada; pesa mais que o esforço, e em todo par do
                    espaço o caminho mais barato é um dos mais curtos
        distância   hastes vizinhas: 1; H1 <-> H3, que pula a do meio: 2
        peso        verde 1, vermelho 2, azul 3

    Na raiz: R1 (vermelho, 1) 12, R2 (vermelho, 2) 14, R3 (azul, 1) 13,
    R4 (azul, 1) 13. O custo de um nó é a soma das regras desde a raiz.

3. Critério

    ABERTOS é ordenada pelo MENOR custo. No empate, sai o nó gerado primeiro;
    entre irmãos de mesmo custo, decide a estratégia (aqui, a ordem crescente
    R1 → R2 → R3 → R4 → R5 → R6). A busca só encerra quando o objetivo vira
    o estado atual, não quando ele é gerado.

    Poda pelo vetor de menor custo: o filho cujo estado já foi gerado com
    custo menor ou igual é descartado. Se o novo custo for menor, o nó
    antigo sai de ABERTOS e da árvore e o novo entra no lugar.

4. Execução

    Expansão de S0 = ([V, R], [A], []), custo 0
        R1 → S1 = ([V], [A, R], [])      12
        R2 → S2 = ([V], [A], [R])        14
        R3 → S3 = ([V, R, A], [], [])    13
        R4 → S4 = ([V, R], [], [A])      13
        ABERTOS  = [S1(12), S3(13), S4(13), S2(14)]
        FECHADOS = [S0]

    Expansão de S1 = ([V], [A, R], []), custo 12
        R2 → S5 = ([], [A, R], [V])      24
        R3 → S0, R4 → estado de S2 (24 ≥ 14): poda
        ABERTOS  = [S3(13), S4(13), S2(14), S5(24)]

    Expansão de S3 = ([V, R, A], [], []), custo 13
        R1 → S0, R2 → S4: poda.  ✗ impasse
        ABERTOS  = [S4(13), S2(14), S5(24)]

    Expansão de S4 = ([V, R], [], [A]), custo 13
        R1 → S6 = ([V], [R], [A])        25
        R5, R6: poda
        ABERTOS  = [S2(14), S5(24), S6(25)]

    Expansão de S2 = ([V], [A], [R]), custo 14
        R1 → S7 = ([], [A, V], [R])      25
        R3 → S8 = ([V, A], [], [R])      27
        ABERTOS  = [S5(24), S6(25), S7(25), S8(27)]

    Expansão de S5 = ([], [A, R], [V]), custo 24
        R3 → S9 = ([R], [A], [V])        36

    Expansão de S6 = ([V], [R], [A]), custo 25
        R1 → S10 = ([], [R, V], [A])     36   ← é Sf, mas só entra na fila
        R5 → S11 = ([V, A], [R], [])     41   (azul, 2 hastes: 10 + 6)
        R6 → S12 = ([V], [R, A], [])     38
        ABERTOS  = [S7(25), S8(27), S9(36), S10(36), S12(38), S11(41)]

    Expansão de S7 = ([], [A, V], [R]), custo 25
        R5 → S13 = ([R], [A, V], [])     39

    Expansão de S8 = ([V, A], [], [R]), custo 27
        R5 → S14 = ([V, A, R], [], [])   41
        R6 → S15 = ([V, A], [R], [])     39   (vermelho, 1 haste: 10 + 2)
        ✓ TROCA: S15 chega ao estado de S11 por 39, menos que 41. S11 sai
          de ABERTOS e da árvore; S15 entra no lugar.
        ABERTOS  = [S9(36), S10(36), S12(38), S13(39), S15(39), S14(41)]

    Expansão de S9 = ([R], [A], [V]), custo 36
        S9 e S10 custam 36; S9 foi gerado antes e sai primeiro.
        R3 → S16 (49), R5 → S17 (48)

    Chega a vez de S10 = ([], [R, V], [A])
        ✓ SUCESSO

5. Árvore de busca (custo acumulado entre parênteses)

    S0 ([V,R], [A], [])  (0)
    ├── R1 → S1 ([V], [A,R], [])  (12)
    │   └── R2 → S5 ([], [A,R], [V])  (24)
    │       └── R3 → S9 ([R], [A], [V])  (36)
    │           ├── R3 → S16  (49)
    │           └── R5 → S17  (48)
    ├── R2 → S2 ([V], [A], [R])  (14)
    │   ├── R1 → S7 ([], [A,V], [R])  (25)
    │   │   └── R5 → S13  (39)
    │   └── R3 → S8 ([V,A], [], [R])  (27)
    │       ├── R5 → S14  (41)
    │       └── R6 → S15 ([V,A], [R], [])  (39)   (substituiu S11)
    ├── R3 → S3 ([V,R,A], [], [])  (13)   (impasse)
    └── R4 → S4 ([V,R], [], [A])  (13)
        └── R1 → S6 ([V], [R], [A])  (25)
            ├── R1 → S10 ([], [R,V], [A])  (36)   (objetivo)
            ├── R5 → S11  (41)   (removido pela troca)
            └── R6 → S12 ([V], [R,A], [])  (38)

6. Caminho solução

    S0 ([V,R], [A], [])
     │ R4: azul H2 → H3        10 + 3 x 1 = 13
     ↓
    S4 ([V,R], [], [A])
     │ R1: vermelho H1 → H2    10 + 2 x 1 = 12
     ↓
    S6 ([V], [R], [A])
     │ R1: verde H1 → H2       10 + 1 x 1 = 11
     ↓
    Sf ([], [R,V], [A])

    Custo da solução: 13 + 12 + 11 = 36
    Número de movimentos: 3      Iterações: 11      Gerados: 18

    Com a ordem decrescente: o mesmo caminho, o mesmo custo e também 11
    iterações.

7. Conclusão

    Um nó só é expandido depois de todos os mais baratos, então quando o
    objetivo vira o estado atual nenhum nó aberto chega nele mais barato:
    a solução é a de MENOR CUSTO. Como cada jogada custa 10 e o esforço de
    uma jogada varia só de 1 a 6, a de menor custo é, em todo par do
    espaço, uma das de menos movimentos; entre as de mesmo tamanho, vence a
    que carrega os discos mais pesados por menos distância.
"""

from __future__ import annotations

from typing import ClassVar

from config import settings
from core.algorithms.domain.base import SearchAlgorithm
from core.algorithms.ordered.frontier import PriorityFrontier
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.tree import SearchTree


class OrderedSearch(SearchAlgorithm):
    name: ClassVar[str] = "ordered"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
        prune: bool = True,
    ) -> None:
        super().__init__(tree, strategy, max_iterations=max_iterations, prune=prune)
        self._best_cost: dict[State, int] = {}
        self._open: dict[State, Node] = {}

    def _search(self, problem: Problem) -> Outcome:
        """Sempre expande o nó aberto de menor custo acumulado.

        O objetivo só encerra a busca quando vira o estado atual, não quando
        é gerado: só aí nenhum nó aberto pode chegar nele mais barato.
        """
        context = self._require_context()
        frontier = PriorityFrontier()
        frontier.push(context.root)

        # Vetor de menor custo: o custo do caminho mais barato já gerado até
        # cada estado. Cobre ABERTOS e FECHADOS de uma vez.
        self._best_cost = {context.root.state: 0}
        # O nó de cada estado que ainda espera em ABERTOS, para poder tirá-lo
        # de lá quando um caminho mais barato até o mesmo estado aparecer.
        self._open = {context.root.state: context.root}
        self._observe_frontier(len(frontier))

        while len(frontier):
            if not self._next_iteration():
                return Outcome.CUTOFF

            node = frontier.pop()
            if self._open.get(node.state) is node:
                del self._open[node.state]
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._visit(node)
            candidates = self._applicable_rules(node)
            if not candidates:
                self._deadlock(node)

            for rule in candidates:
                child = self._expand(node, rule)
                older = self._open.get(child.state) if self.prune else None
                if older is not None:
                    # Chegou mais barato a um estado que ainda está aberto:
                    # o nó antigo sai de ABERTOS e da árvore.
                    frontier.remove(older)
                    self._discard(older)
                self._best_cost[child.state] = child.cost
                self._open[child.state] = child
                frontier.push(child)
            self._observe_frontier(len(frontier))

        return self._exhausted()

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        """Poda o filho que não melhora o menor custo conhecido do estado.

        Com custos positivos, um estado já fechado nunca é alcançado mais
        barato depois, então ele sempre cai aqui.
        """
        best = self._best_cost.get(successor)
        return best is not None and node.cost + rule.cost(node.state) >= best

    def _discard(self, node: Node) -> None:
        """Só contabilidade: registra o nó substituído como poda do pai."""
        context = self._require_context()
        parent, rule = node.parent, node.rule
        assert parent is not None and rule is not None
        context.trace.record_prune(
            iteration=context.metrics.iterations,
            node=parent,
            rule_id=rule.id,
            state=node.state,
        )

## Caminho solução

R4, R1, R1: 13 + 12 + 11 = 36. Levar o azul uma haste custa 13, o vermelho 12 e o verde 11. É o menor custo possível; as duas estratégias, com e sem poda, chegam a este mesmo caminho.

In [ ]:
show_path(solve("ordered", "ascending"))

## Comparativo: os caminhos encontrados

Cada ponto é um movimento; o verde é o que chega ao objetivo.

In [ ]:
paths_plot("ordered")

## Comparativo: crescente contra decrescente

In [ ]:
strategy_table("ordered")

## Comparativo: busca ordenada contra os outros, mesmas estratégias

In [ ]:
algorithms_table("ordered")

## Comparativo: quem explorou menos nós

In [ ]:
expanded_chart("ordered")

## Comparativo: os 36 objetivos

Cada estado do espaço como objetivo, sempre a partir da mesma posição inicial, nas duas estratégias, da menor para a maior média.

### Nós expandidos nos 36 objetivos

In [ ]:
goals_boxplot("ordered", "expanded")

Na ordenada e na largura, expandidos = iterações − 1: o objetivo sai de ABERTOS, mas não é expandido. Por isso os valores são sempre 0, 1, …, 35.

In [ ]:
goals_stats("ordered", "expanded")

### Iterações nos 36 objetivos

In [ ]:
goals_boxplot("ordered", "iterations")

Com poda, cada estado sai de ABERTOS uma vez, numa ordem que não depende do objetivo: o objetivo i é achado na posição i dessa ordem. Nos 36 objetivos, as iterações são sempre 1, 2, …, 36, só embaralhadas, para qualquer custo.

In [ ]:
goals_stats("ordered", "iterations")

### Nós gerados nos 36 objetivos

In [ ]:
goals_boxplot("ordered", "generated")

In [ ]:
goals_stats("ordered", "generated")

### Custo da solução: só P1

In [ ]:
p1_cost_chart("ordered")

### Custo da solução: todos os 36 objetivos

In [ ]:
goals_boxplot("ordered", "cost")

In [ ]:
goals_stats("ordered", "cost")

## Complexidade

| Algoritmo | Tempo | Memória | Solução | Por quê |
|---|---|---|---|---|
| **Ordenada sem poda** | O(b^(1 + C*/ε)) | O(b^(1 + C*/ε)) | menor custo | a fila guarda todo nó mais barato que o objetivo, repetido ou não |
| **Ordenada com poda** | O((V + E) log V) | O(V) | menor custo | cada estado entra uma vez; o heap custa log V por operação |

In [ ]:
complexity_values("ordered", ('b', 'C*', 'ε', 'V', 'E'))

## Conclusão

- Em P1: R4, R1, R1, custo 36, em 11 iterações e 18 nós gerados, com as duas ordens. Houve uma troca de nó: S15 entrou no lugar de S11.
- Nos 36 objetivos acha sempre o menor custo (média 51,33, a menor de todas) e também o mínimo de movimentos. A média de iterações é 18,5, a mesma da largura.
- A estratégia só desempata irmãos de mesmo custo: nunca muda o custo da solução.

## Extras de busca ordenada

### Fluxograma

In [ ]:
show_flowchart("ordered")

### Passo a passo do trace, com poda e ordem crescente

In [ ]:
show_trace(solve("ordered", "ascending"))

### Comparação em P1, placar lexicográfico

In [ ]:
p1_comparison("ordered")

### Comparação nos 36 objetivos

Cada métrica: média / mediana; movimentos e custo contam só os sucessos.

In [ ]:
goals_comparison("ordered")

# 5. Comparação dos algoritmos

Uma carta só não diz qual método é melhor. Esta seção roda a matriz completa (4 algoritmos × 2 estratégias) primeiro na carta P1 e depois em todos os 36 estados do espaço como objetivo, sempre a partir da mesma posição inicial (cartas G01 a G36).

As tabelas trazem os quatro algoritmos. Os gráficos deixam a irrevogável de fora: ela não garante solução, e uma execução sem caminho não tem movimentos para comparar.

## A carta P1

As 8 execuções de P1, ordenadas pelo placar lexicográfico: desfecho, movimentos, iterações e nós gerados, nessa ordem de prioridade.

In [ ]:
p1_comparison().set_index("#")

### P1 em gráfico

Em cima, a carta P1. Embaixo, movimentos, iterações e nós gerados de cada algoritmo, agrupados por estratégia. A linha tracejada é o ótimo de P1 (3 movimentos).

In [ ]:
p1_chart()

Em P1, largura e ordenada acham o ótimo com as duas estratégias, e a ordenada com `ascending` é a mais rápida (10 iterações). O backtracking sempre chega, mas com 14 a 22 movimentos. A irrevogável, fora do gráfico, trava em `descending`. Uma carta, porém, é uma amostra de um: a próxima parte repete tudo para os 36 objetivos possíveis.

## Todos os 36 objetivos

Cada estado do espaço vira o objetivo de uma carta sintética (G01 a G36), todas a partir da posição inicial: 36 cartas × 4 algoritmos × 2 estratégias = 288 execuções. O ótimo de cada carta vem dos oráculos de `state_space`, escritos fora do motor: `shortest_distance` (menor número de movimentos) e `cheapest_cost` (menor custo).

### Resumo consolidado

O mesmo resumo de `python main.py --all-goals`: para cada combinação, média / mediana. Movimentos e custo contam só os sucessos. A irrevogável resolve 18 objetivos com `ascending` e 3 com `descending`; os outros três algoritmos resolvem os 36.

In [ ]:
goals_comparison()

### Os 36 objetivos em gráfico

O mesmo gráfico de P1, agora com as 36 execuções de cada combinação. A caixa vai do 1º ao 3º quartil, a barra preta é a mediana e o losango branco é a média. Em movimentos, a caixa cinza é o ótimo de cada objetivo. Iterações e nós gerados estão em escala logarítmica: o backtracking com `descending` passa de 2 mil iterações num objetivo.

In [ ]:
goals_chart()

### Melhor por critério

Menor valor vence, pela média e pela mediana. Só concorre quem tem o maior número de sucessos, para a irrevogável não vencer resolvendo apenas os objetivos fáceis.

In [ ]:
best_by_criterion()

### Qualidade da solução

Quanto cada solução passa do ótimo, em movimentos e em custo. Zero é a solução ótima.

In [ ]:
quality_chart()

In [ ]:
above_optimum()

### Por que largura e ordenada têm a mesma média de iterações

As duas visitam cada estado no máximo uma vez e só encerram quando o objetivo vira o estado atual. Com a mesma estratégia, a ordem em que visitam os 36 estados não depende do objetivo: o objetivo i é encontrado na posição dele nessa ordem, que é uma permutação de 1 a 36. A média é (1 + 36) / 2 = 18,5 para qualquer ordem de visita. O critério que as separa é o número de nós gerados.

In [ ]:
permutation_check()

## Complexidade dos quatro algoritmos

| Algoritmo | Tempo | Memória | Solução | Por quê |
|---|---|---|---|---|
| **Irrevogável sem poda** | não termina | O(1) | pode não achar | pode girar num ciclo para sempre |
| **Irrevogável com poda** | O(b·m²) | O(m) | pode não achar | um nó por iteração, no máximo m; cada regra é comparada com o caminho |
| **Backtracking sem poda** | não termina | O(L) | pode não achar | o ciclo cresce a pilha até o limite L de iterações |
| **Backtracking com poda** | O(bᵐ) | O(m) | a primeira achada | só guarda o caminho atual |
| **Largura sem poda** | O(bᵈ) | O(bᵈ) | ótima | a fila guarda o nível inteiro |
| **Largura com poda** | O(V + E) | O(V) | ótima | cada estado entra uma vez |
| **Ordenada sem poda** | O(b^(1 + C*/ε)) | O(b^(1 + C*/ε)) | menor custo | a fila guarda todo nó mais barato que o objetivo, repetido ou não |
| **Ordenada com poda** | O((V + E) log V) | O(V) | menor custo | cada estado entra uma vez; o heap custa log V por operação |

Os símbolos, com os valores medidos em P1:

In [ ]:
complexity_values(None, ('b', 'd', 'm', 'V', 'E', 'C*', 'ε'))

## Análise

**Garantias.** Largura e ordenada resolvem os 36 objetivos com o número mínimo de movimentos, com qualquer estratégia. A ordenada também acha sempre o menor custo; a largura passa do menor custo em 9 das 72 execuções, porque conta movimentos e não olha o custo das regras. Isso confere com a teoria: largura é ótima em comprimento, ordenada é ótima em custo, e no empate de comprimento as duas coincidem.

**A irrevogável depende da estratégia.** Ela é barata (um nó gerado por iteração), mas não garante solução: resolve 18 ou 3 dos 36 objetivos conforme a estratégia. Quando resolve, o caminho pode ser bem mais longo que o ótimo.

**O backtracking sempre resolve, mas não bem.** O espaço é conexo, então o retrocesso sempre acha um caminho. Só que é o primeiro, não o mais curto: a média fica entre 12,8 e 16,5 movimentos, contra 4,14 do ótimo. O esforço é irregular: com `descending` a média de iterações é 82,9, enquanto a mediana fica em 21,5. Poucos objetivos exigem milhares de passos de ida e volta.

**Esforço.** Largura e ordenada empatam em iterações (média 18,5, pelo argumento da permutação acima) e ficam próximas em nós gerados. O backtracking com `ascending` gera menos nós, mas paga com as soluções mais longas de todas.

**Recomendação.** Para este problema, a busca ordenada é a escolha: resolve tudo, com o menor custo e o menor número de movimentos, com as mesmas iterações médias da largura e cerca de 7% mais nós gerados. Quando as regras têm o mesmo custo, a largura dá o mesmo resultado. A irrevogável e o backtracking servem para mostrar o que acontece sem memória da fronteira e sem garantia de otimalidade.

# 6. Integrantes

| Integrante | Matrícula |
|---|---|
| Charles Lelis Braga | 202035015 |
| Darlan Henrique da Costa Silva | 202176038 |
| Isaac Dizolele Kapela João | 202376017 |
| Fábio do Vale Affonso | 202076021 |
| Felipe Iglesias Cordeiro Leite | 202465031AB |
| Rafaela Oliveira de Souza | 202365133C |
| Rian Cesar Quintanilha | 202465080AC |
| Gustavo Dias de Almeida | 202165571C |
| Larissa Rezende Fazza | 202335021 |
| Ian Félix Fernandes | 202376007 |
| Michel Gomes de Andrade | 201876037 |

DCC014 · Inteligência Artificial · Universidade Federal de Juiz de Fora · Profa. Luciana